# Résiliation client SaaS (churn) — Notebook de certification

| | |
|---|---|
| **Certification** | « Concevoir et implémenter une solution d'intelligence artificielle » — référentiel C1 → C9 |
| **Cas d'usage** | 01 — Résiliation client SaaS (churn) : classification binaire + régression de la valeur vie client |
| **Candidat** | Guillaume Saidani |
| **Date** | 23 septembre 2026 (date d'exécution affichée par la cellule ci-dessous) |
| **Version** | v2.1 — Gold dataset v2 révisé (retards de paiement impossibles neutralisés, §6.5), modèles v2.1 (la v1 reste consultable dans `notebooks/00` à `06`) |
| **Code source** | https://github.com/guillaumesaidani-coder/churn_saas |
| **Jeu de données et modèles (téléchargement libre)** | https://github.com/guillaumesaidani-coder/churn_saas/releases/tag/v2.0 |
| **Versioning DVC** (compte DagsHub gratuit requis pour naviguer) | https://dagshub.com/guillaume.saidani/churn_saas |
| **Suivi des entraînements (MLflow)** | https://dagshub.com/guillaume.saidani/churn_saas.mlflow |

## Comment lire ce notebook

Ce notebook suit **le plan imposé par le règlement de l'épreuve** (§0 à §15). Il se lit seul,
sans explication orale : chaque résultat affiché est produit par le code qui le précède.

Quatre types de blocs reviennent tout au long du document :

| Bloc | Rôle |
|---|---|
| 📘 **Explication** | Commentaire pédagogique placé **avant chaque cellule de code** : ce qu'elle fait, pourquoi, et ce qu'il faut en retenir. Ces cellules portent le tag `pedagogie` : elles pourront être retirées automatiquement pour une version plus courte. |
| 📓 **Journal de bord** | Synthèse des constats et des décisions à la fin de chaque grande étape, comme l'exige le règlement. |
| ⚖️ **Décisions D1 à D15** | Choix de cadrage argumentés (atelier de cadrage n°1 : D1–D7 ; point de contact métier n°3 : D8–D15). Exercice mené seul : ce sont des choix justifiés par le candidat, pas des validations d'un commanditaire réel. |
| ⚠️ **Hypothèses H01 à H05** | Paramètres non vérifiables dans les données, toujours étiquetés comme tels et testés en sensibilité. |

**Règle méthodologique centrale.** Le jeu de test (20 % des comptes) est mis de côté dès la
préparation et **n'est utilisé qu'une seule fois**, en §9, pour mesurer la performance du
modèle final. Une seconde lecture, pour la révision v2.1, est expliquée et assumée en §9.3. Tous les choix (variables, modèle, hyperparamètres, seuil de décision) sont faits
en validation croisée sur le seul jeu d'entraînement.

📘 **Explication — cellule d'environnement**

Cette première cellule prépare tout ce dont le notebook a besoin :

- **Imports** : `pandas`/`numpy` pour les tableaux, `matplotlib` pour les graphiques, `sklearn`
  pour les modèles, `joblib` pour sauvegarder les modèles sur disque.
- **Chemins** : le notebook peut être lancé depuis `notebooks/` (cas normal, et cas du pipeline
  DVC) ou depuis la racine du projet ; `ROOT` pointe toujours vers la racine. On ajoute `ROOT` au
  `sys.path` pour pouvoir importer le code réutilisable du dossier `src/`.
- **Constantes métier** : elles traduisent les décisions de cadrage en paramètres (rappel cible
  de 80 % = D9, capacité de 150 comptes par mois = D10, etc.). Les regrouper ici évite des
  « nombres magiques » dispersés dans le code.
- **`RANDOM_STATE = 42`** : graine aléatoire fixe. Toute opération aléatoire (découpage des
  données, forêts aléatoires) donnera exactement le même résultat à chaque exécution : c'est la
  condition de la **reproductibilité**.
- **`sauver_figure`** : petite fonction utilitaire qui enregistre chaque graphique dans
  `data/model_v2/figures/`. Ces fichiers sont des **artefacts** livrés avec le notebook.

À retenir : un notebook reproductible commence par un environnement explicite (versions,
chemins, graines). La cellule affiche les versions des bibliothèques pour que n'importe qui
puisse recréer le même environnement (voir aussi `requirements-dev.txt`, §15).

In [ ]:
import json
import os
import platform
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn

# Racine du projet, que le notebook soit lancé depuis notebooks/ ou depuis la racine
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))                       # rend le dossier src/ importable
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore", category=FutureWarning)

# Emplacements des données et des artefacts
RAW_DIR = ROOT / "Examen_cas d'usage candidat"      # fichiers fournis par l'énoncé (versionnés par DVC)
GOLD_DIR = ROOT / "data" / "gold"                   # Gold v1 (notebook 03) et Gold v2 (ce notebook)
MODEL_V1_DIR = ROOT / "data" / "model"              # modèles v1 (notebooks 04-06), pour comparaison
MODEL_DIR = ROOT / "data" / "model_v2"              # artefacts produits par ce notebook
FIG_DIR = MODEL_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Constantes issues du cadrage (décisions D8-D10, faits F14, hypothèses H01)
RANDOM_STATE = 42          # graine unique : reproductibilité
RAPPEL_CIBLE_D9 = 0.80     # D9 : le seuil doit garantir un rappel >= 80 %
CAPACITE_CSM_D10 = 150     # D10 : comptes traitables par les CSM à chaque cycle mensuel
GAIN_MIN_D8 = 0.15         # D8 : le modèle doit battre la baseline métier de +0,15 de PR-AUC
COUT_FN_EUR = 6879         # F14 : valeur vie client médiane des comptes qui résilient
COUTS_FP_EUR = [50, 150]   # H01 : coût d'une relance CS (fourchette, hypothèse)
PSI_SEUIL_ALERTE = 0.25    # seuil d'alerte de dérive, identique à drift_alert_rules.yml

LIENS = {
    "GitHub": "https://github.com/guillaumesaidani-coder/churn_saas",
    "DagsHub (données DVC)": "https://dagshub.com/guillaume.saidani/churn_saas",
    "MLflow": "https://dagshub.com/guillaume.saidani/churn_saas.mlflow",
}

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
COULEURS = {"non_churn": "#4C78A8", "churn": "#E45756", "neutre": "#9D9D9D"}


def sauver_figure(fig, nom: str) -> Path:
    """Enregistre une figure dans data/model_v2/figures/ (artefact livré avec le notebook)."""
    chemin = FIG_DIR / f"{nom}.png"
    fig.savefig(chemin, dpi=120, bbox_inches="tight")
    return chemin


def commit_git() -> str:
    """Identifiant du commit Git courant : relie les résultats à une version exacte du code."""
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "inconnu"


print(f"Exécuté le      : {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC")
print(f"Python          : {platform.python_version()} ({platform.system()} {platform.release()})")
print(f"Commit Git      : {commit_git()}")
print("Bibliothèques   :", ", ".join(f"{m.__name__} {m.__version__}"
                                     for m in [pd, np, sklearn, scipy, matplotlib, joblib]))
print(f"Racine projet   : {ROOT}")

## 1. Résumé exécutif

| | |
|---|---|
| **Problème** | Un éditeur SaaS B2B voit 28 % de ses comptes résilier à l'échéance contractuelle (5 000 comptes étudiés). Chaque résiliation coûte le revenu récurrent et la valeur vie client restante (médiane 6 879 € pour les comptes qui partent). |
| **Objectif** | Donner aux équipes Customer Success une **liste mensuelle de 150 comptes à contacter en priorité**, ordonnée par enjeu économique, sans jamais automatiser la décision (RGPD, art. 22). |
| **Approche** | Une solution en **trois briques** : **(1)** un modèle de classification qui estime la probabilité de churn de chaque compte ; **(2)** un modèle de régression qui estime sa valeur vie client ; **(3)** un **système de décision** à base de règles métier (seuil de rappel D9, perte attendue = probabilité × valeur, capacité de 150 comptes D10, catalogue d'actions D11) qui transforme ces deux prédictions en liste de travail. En amont : données nettoyées et versionnées (Gold v2, DVC), fuite et leurres exclus, trois familles de modèles comparées en validation croisée. En aval : API de scoring, suivi MLflow, surveillance de dérive. |

**Résultats clés** (jeu de test de 1 000 comptes ; seconde lecture assumée pour la révision v2.1, §9.3) :

- **Régression logistique** retenue : la plus performante *et* la plus explicable. PR-AUC de
  **0,748** (ROC-AUC 0,882), contre 0,593 pour une règle métier simple : le critère
  d'acceptation D8 (+0,15) est **respecté (+0,154)**, avec une marge faible.
- Au seuil retenu (0,283, calculé sans toucher au test), le modèle détecte **82,5 %** des
  comptes qui vont partir, avec une précision de 63 %. Les probabilités sont bien calibrées.
- La liste des **150 priorités Hautes** contient **102 comptes qui allaient réellement partir**
  (42 attendus au hasard) et capte **82 %** de la perte réelle du cycle.
- Valeur vie client : erreur relative médiane de 42 %, suffisante pour **ordonner** les comptes
  par enjeu (corrélation de rang de 0,94 avec la valeur réelle).
- **Données contrôlées avant d'apprendre (v2.1)** : 13 règles de cohérence entre colonnes. Les
  valeurs extrêmes sont cohérentes et conservées ; **449 comptes (9 %) déclarent plus de retards
  de paiement que de mois facturés**, une valeur impossible et liée à la cible. Elle est
  neutralisée : le modèle v2.0 en faisait le premier facteur de risque de 23 % des priorités
  Hautes. Coût accepté : 0,013 de PR-AUC sur le test (0,761 en v2.0).
- Le Gold v2 fait aussi bien que la v1 avec **20 variables au lieu de 30**, et corrige trois
  défauts de la v1 : choix du modèle et du seuil faits sur le test, imputation calculée avant
  le découpage, API qui échouait sur une valeur manquante.
- **Explicabilité contrôlée** : chaque priorité est livrée avec ses facteurs de risque et la
  règle appliquée, et le modèle est confronté à une **base de connaissance** écrite a priori.
  Ce contrôle bloque un modèle avec fuite que sa PR-AUC (0,997) aurait fait retenir, et il a
  révélé deux variables construites sans apport, à retirer à la prochaine version.

**Limites** : un seul instantané des comptes (pas de validation dans le temps) ; la recette D15
n'est que partiellement satisfaite (les comptes de très grande valeur mais de risque modéré
échappent à la liste, §12.4 ; leurs explications montrent qu'ils ne présentent aucun signal de
désengagement dans les données, §12.4.1) ; l'impact métier est **estimé**, pas mesuré : il le sera par le
groupe témoin prévu en production (D12) ; la date de calcul de `derniere_connexion_jours` est à
confirmer, comme trois autres données à la définition douteuse (§6.5).

## 2. Cadrage métier et cas d'usage — journal de bord [C1]

### 2.1 Contexte

L'éditeur SaaS B2B du cas d'usage tire ses revenus d'abonnements récurrents (plan × licences ×
usage). La résiliation d'un compte (**churn**) le prive à la fois du revenu mensuel récurrent
(MRR) et d'une partie de la **valeur vie client** (CLV) restante. Le churn est donc un
indicateur critique pour la rentabilité.

### 2.2 Besoin et objectif

Les équipes **Customer Success (CS)** veulent anticiper les comptes susceptibles de résilier,
pour concentrer leurs actions de rétention **avant l'échéance contractuelle**.

Je retiens un **outil d'aide à la décision** : hiérarchiser les comptes à risque, jamais
remplacer le jugement d'un CSM par une exécution automatique. Ce n'est pas une préférence de
conception : c'est une contrainte de l'énoncé (« l'objectif n'est pas de remplacer la décision
humaine par un algorithme ») et une condition de conformité (article 22 du RGPD, §4).

### 2.3 Traduction en problème de données

| Question métier | Traduction technique |
|---|---|
| Ce compte va-t-il résilier à l'échéance ? | **Classification binaire** sur `churn` (cible principale) |
| Combien vaut ce compte ? | **Régression** sur `valeur_vie_client_eur` (cible secondaire, jamais utilisée comme variable explicative du churn) |
| Quels comptes appeler en premier, avec 150 appels par mois ? | **Système de décision** (règles, sans apprentissage) : perte attendue = p(churn) × CLV estimée, sous contrainte de capacité |

### 2.3 bis La solution : deux modèles et un système de décision

```
 Données du compte (20 variables, Gold v2)
          │                          │
          ▼                          ▼
 ┌────────────────────┐    ┌────────────────────┐
 │ 1. Modèle CHURN     │    │ 2. Modèle CLV       │
 │ classification      │    │ régression          │
 │ → p(churn)          │    │ → valeur estimée €  │
 └─────────┬──────────┘    └──────────┬─────────┘
           └──────────────┬───────────┘
                          ▼
 ┌───────────────────────────────────────────────┐
 │ 3. SYSTÈME DE DÉCISION (règles métier)         │
 │ signalé si p(churn) ≥ seuil D9                 │
 │ perte attendue = p(churn) × CLV estimée (D14)  │
 │ Haute = 150 plus fortes pertes signalées (D10) │
 │ action recommandée (D11), jamais exécutée (D3) │
 └──────────────────────┬────────────────────────┘
                        ▼
        Liste mensuelle pour les CSM (export CRM)
```

| Brique | Nature | Où dans ce notebook |
|---|---|---|
| 1. Modèle churn | Apprentissage supervisé (classification) | §8, §9.1 à §9.6 |
| 2. Modèle CLV | Apprentissage supervisé (régression) | §9.7, §12.2 |
| 3. Système de décision | Règles métier, **sans apprentissage** ; paramètres figés dans `scoring_manifest.json` | §9.2 (seuil), §10.2 (application), §12.3 et §12.4 (évaluation) |

Les deux modèles sont entraînés séparément et aucun ne sert de variable à l'autre : leurs sorties
ne se rencontrent que dans le système de décision. Les modèles disent **ce qui va probablement se
passer** ; le système de décision dit **quoi faire, pour qui, dans quel ordre**.

### 2.4 Décisions de cadrage (atelier n°1, D1–D7)

| # | Décision | Justification |
|---|---|---|
| D1 | Unité d'analyse = le **compte client B2B** (`client_id`), jamais une personne physique | Grain du jeu de données ; cadre la finalité RGPD sur une entité contractante |
| D2 | `churn` = **non-reconduction à l'échéance**, pas un simple arrêt d'usage | Seule définition compatible avec la colonne fournie et actionnable avant l'échéance |
| D3 | Le modèle **priorise** ; aucune action n'est déclenchée automatiquement | Article 22 du RGPD ; condition d'adoption par les CSM |
| D4 | Un **faux négatif** coûte structurellement plus qu'un **faux positif** | Un compte perdu coûte sa CLV (médiane 6 879 €, F14) ; une relance inutile coûte du temps CSM (50–150 €, H01) : ratio de 45:1 à plus de 1 000:1 |
| D5 | Le seuil de décision n'est pas fixé en atelier n°1 | Renvoyé au point de contact n°3 (D9) |
| D6 | Finalité unique : **priorisation des actions de rétention** ; base légale : intérêt légitime (art. 6.1.f) | Principe de finalité (art. 5 RGPD) |
| D7 | `commentaire_csm` (texte libre) **n'est pas utilisé** comme variable explicative | Risque résiduel de contenu nominatif, par précaution |

### 2.5 Décisions opérationnelles (point de contact n°3, D8–D15)

| # | Décision | Où elle est appliquée |
|---|---|---|
| D8 | Critère d'acceptation : le modèle doit battre une **baseline métier simple** d'au moins **+0,15 de PR-AUC** | §8, §9, §12 |
| D9 | Seuil fixé pour garantir un **rappel ≥ 80 %** sur la classe churn | §9 |
| D10 | Capacité CS : **150 comptes** priorisés par cycle mensuel | §10, §12 |
| D11 | Catalogue d'actions à 3 niveaux (appel sous 5 jours / email ciblé / surveillance passive) | §10 |
| D12 | Mesure d'impact : **groupe témoin** de 20 % des comptes priorisés, traités avec un mois de décalage | §13 |
| D13 | Consolidation trimestrielle de l'impact | §13 |
| D14 | Priorité : Haute = signalés (score ≥ seuil D9) triés par perte attendue dans la limite de 150 ; Moyenne = signalés restants ; Basse = non signalés | §10 |
| D15 | Recette : corrélation de rang (Spearman) ≥ 0,7 entre perte attendue et perte réelle, et aucun faux négatif à forte perte réelle ignoré | §12 |

### 📓 Journal de bord — §2 Cadrage [C1]

- **Besoin** : prioriser les actions de rétention des CSM avant l'échéance, sans automatiser la
  décision (D3).
- **Deux cibles, deux modèles** : churn (classification) et CLV (régression), combinés
  seulement à la fin pour calculer une perte attendue.
- **Asymétrie des erreurs** (D4) : rater un compte qui part coûte de 45 à plus de 1 000 fois plus
  qu'une relance inutile. Conséquence : on privilégiera le **rappel**, traduit en règle
  opérationnelle par D9 (rappel ≥ 80 %).
- **Contrainte de capacité** (D10) : un score seul ne suffit pas ; il faut une liste de 150
  comptes par mois, ordonnée par enjeu économique (D14).
- **Critère d'acceptation explicite** (D8) : le modèle n'a de valeur que s'il fait nettement
  mieux qu'une règle simple que le métier pourrait appliquer sans IA.
- **Limite assumée** : exercice mené seul, sans commanditaire réel. Les décisions sont
  argumentées ici plutôt que « validées » par un tiers.

## 3. Données : disponibilité, gouvernance et alternatives — journal de bord [C1]

### 3.1 Données disponibles

| Fichier | Contenu | Usage |
|---|---|---|
| `churn_saas_complet.csv` | ~5 000 comptes × 29 colonnes : usage, facturation, support, contrat, deux cibles | Entraînement et évaluation |
| `churn_saas_echantillon.csv` | 50 comptes, même format | Simulation d'un « export brut » à scorer (§10) |
| `catalogue_plans.csv` | 4 plans tarifaires (prix par siège, SLA, quota…) | Jointure sur `plan` |

Les données sont **réalistes et volontairement imparfaites** : valeurs manquantes, nombres
stockés en texte, dates en formats multiples, doublons, variables leurres et un piège de fuite.

### 3.2 Gouvernance mise en place

- **Portique RGPD avant toute ingestion** (`notebooks/00`) : recherche systématique de données
  identifiantes et de motifs nominatifs ; décision GO/NO-GO tracée dans
  `data/rgpd/rgpd_gate_manifest.json` (lu par la cellule ci-dessous).
- **Architecture en couches** Bronze (copie brute) → Silver (nettoyée) → Gold (prête pour le
  modèle), chaque couche accompagnée d'un **manifeste JSON** qui contient le hash SHA-256 de ses
  entrées et de ses sorties : on peut prouver quelle version des données a produit quel modèle.
- **Versioning des données avec DVC** : les fichiers de données et les modèles sont stockés sur
  DagsHub ; Git ne contient que leurs empreintes (`*.dvc`, `dvc.lock`). `dvc pull` récupère la
  version exacte correspondant à un commit.
- **Pseudonymisation disponible** mais non appliquée par défaut (`client_id` n'est pas nominatif) ;
  la clé et la table de correspondance ne sont **jamais publiées** (exclues de Git et de DVC).

### 3.3 Alternatives envisagées

| Alternative | Décision | Raison |
|---|---|---|
| Base PostgreSQL pour la couche Silver | Écartée | Non exigée par l'énoncé ; le notebook doit s'exécuter sans serveur ; Parquet typé + manifestes couvre le besoin (schéma, traçabilité) |
| Données externes (données sectorielles, historique de tickets détaillé, logs d'usage) | Non disponibles | Piste d'amélioration (§13) : les logs d'usage datés permettraient des tendances (usage en baisse sur 3 mois) |
| Historique multi-périodes (plusieurs instantanés mensuels) | Non disponible | Limite majeure : pas de validation temporelle possible (§14) |

📘 **Explication — contrôle de la gouvernance**

Cette cellule ne transforme aucune donnée : elle **vérifie** que les garde-fous de gouvernance
sont en place avant d'aller plus loin.

1. Elle calcule le **hash SHA-256** de chaque fichier source. Un hash est une empreinte
   numérique : si un seul caractère du fichier change, le hash change complètement. En le
   comparant au hash enregistré par DVC, on sait que l'on travaille sur les données prévues.
2. Elle lit la **décision du portique RGPD** enregistrée par `notebooks/00` : colonnes
   identifiantes détectées, motifs nominatifs trouvés, décision GO/NO-GO.
3. Elle vérifie que les fichiers sensibles (clé de pseudonymisation) sont bien **ignorés par
   Git**, donc jamais publiés.

À retenir pour l'oral : la gouvernance n'est pas un paragraphe de principe, c'est un ensemble de
contrôles exécutables et rejoués à chaque exécution.

In [ ]:
from src.versioning import sha256_of

FICHIERS_SOURCES = {
    "principal": RAW_DIR / "churn_saas_complet.csv",
    "echantillon": RAW_DIR / "churn_saas_echantillon.csv",
    "catalogue": RAW_DIR / "catalogue_plans.csv",
}
inventaire = pd.DataFrame([
    {"fichier": p.name, "taille_ko": round(p.stat().st_size / 1024, 1), "sha256 (début)": sha256_of(p)[:16]}
    for p in FICHIERS_SOURCES.values()
])
display(inventaire)

portique = json.loads((ROOT / "data" / "rgpd" / "rgpd_gate_manifest.json").read_text(encoding="utf-8"))
print("Portique RGPD")
print("  colonnes identifiantes détectées :", portique["colonnes_identifiantes_detectees"] or "aucune")
print("  motifs nominatifs détectés       :", portique["motifs_nominatifs_detectes"])
print("  AIPD obligatoire                 :", portique["aipd_obligatoire"])
print("  décision                         :", portique["decision"])

# Les fichiers de pseudonymisation doivent être ignorés par Git (jamais publiés)
for sensible in ["data/rgpd/keymap_secret.json", "data/rgpd/keymap_client_id.parquet"]:
    ignore = subprocess.run(["git", "check-ignore", "-q", sensible], cwd=ROOT).returncode == 0
    print(f"  {sensible:38s} ignoré par Git : {ignore}")

### 📓 Journal de bord — §3 Données et gouvernance [C1]

- **Sources** : trois fichiers fournis, identifiés par leur hash SHA-256 et versionnés par DVC
  (lien DagsHub en page de garde).
- **RGPD** : portique exécuté avant toute ingestion : aucune colonne identifiante, aucun motif
  nominatif, AIPD non obligatoire, décision GO. La clé de pseudonymisation n'est jamais publiée.
- **Traçabilité** : chaîne de hash RGPD → Bronze → Silver → Gold → modèle → run MLflow. Ce
  notebook produit un **Gold v2** avec son propre manifeste, sans écraser la v1.
- **Alternatives** : PostgreSQL écarté (non requis, dépendance serveur) ; données externes et
  historique multi-périodes non disponibles, identifiés comme pistes d'amélioration.
- **Limite** : un seul instantané des comptes ; impossible de vérifier comment le modèle vieillit
  dans le temps sans nouvel export.

## 4. Enjeux éthiques, sociétaux et conformité — journal de bord [C2]

### 4.1 Conformité RGPD

| Principe | Application dans le projet |
|---|---|
| **Base légale** (art. 6) | Intérêt légitime de l'éditeur à assurer la continuité de la relation contractuelle B2B (D6) |
| **Finalité** (art. 5.1.b) | Priorisation des actions de rétention uniquement ; tout autre usage (évaluation d'un CSM, scoring commercial) exigerait une nouvelle analyse |
| **Minimisation** (art. 5.1.c) | Données agrégées au niveau du compte ; `commentaire_csm` exclu (D7) ; export CRM limité à 5 colonnes (§10) |
| **Décision automatisée** (art. 22) | Le modèle produit un score et une recommandation, **jamais une action exécutée** (D3), vérifié par un test automatique (cellule ci-dessous) |
| **Registre de traitement** (art. 30) | Texte proposé : finalité, base légale, données, conservation (table de scores écrasée à chaque cycle), destinataires (équipes CS) |

### 4.2 Conséquences des erreurs du modèle

| Erreur | Conséquence pour le client | Conséquence pour l'éditeur |
|---|---|---|
| **Faux négatif** (compte qui part sans être signalé) | Aucune aide proposée alors qu'il rencontrait des difficultés | Perte de la CLV restante (médiane 6 879 €) |
| **Faux positif** (compte fidèle signalé) | Sollicitation non nécessaire, risque de lassitude | Temps CSM (50–150 €, H01) ; usure de l'équipe si trop fréquent |

Le choix d'un rappel élevé (D9) accepte davantage de faux positifs : c'est acceptable parce que
l'action associée (un appel ou un email) est **bienveillante et réversible**, pas une sanction.

### 4.3 Variables écartées pour des raisons éthiques ou de conformité

- `commentaire_csm` : texte libre, risque de données personnelles (D7).
- `pays` : sans pouvoir prédictif (§6) **et** susceptible d'introduire une différence de
  traitement géographique ; son exclusion supprime ce risque sans coût de performance.
- `client_id` : identifiant, sans valeur prédictive légitime.

### 4.4 Biais et équité

Un modèle peut être globalement bon et moins performant pour certains segments. En §12, un
**audit d'équité** mesure le rappel et la précision par taille d'entreprise, secteur et plan,
avec des intervalles de confiance, pour vérifier qu'aucun segment n'est systématiquement
moins bien servi.

📘 **Explication — test de conformité à l'article 22**

Les recommandations envoyées aux CSM sont des phrases (catalogue D11). La fonction
`action_est_conforme_d3` (module `src/scoring.py`) vérifie qu'aucune d'elles ne décrit une
action exécutée sans intervention humaine (par exemple « résiliation appliquée » ou
« downgrade appliqué »).

On l'applique ici aux trois actions réelles du catalogue, puis à une formulation volontairement
interdite, pour montrer que le contrôle détecte bien un cas non conforme.

À retenir : l'exigence de non-automatisation est **testée**, pas seulement affirmée. Le même
contrôle fait partie des tests unitaires du projet (`tests/test_scoring.py`).

In [ ]:
from src.scoring import ACTIONS_PAR_PRIORITE, action_est_conforme_d3

for priorite, action in ACTIONS_PAR_PRIORITE.items():
    print(f"{priorite:8s} | conforme D3 : {action_est_conforme_d3(action)} | {action}")

exemple_interdit = "Downgrade appliqué automatiquement au plan Starter"
print(f"\nContrôle d'un contre-exemple -> conforme : {action_est_conforme_d3(exemple_interdit)}")
assert all(action_est_conforme_d3(a) for a in ACTIONS_PAR_PRIORITE.values())
assert not action_est_conforme_d3(exemple_interdit)

### 4.5 Cadre réglementaire et chartes éthiques

Trois références encadrent ce projet. Pour chacune : ce qu'elle demande, et ce que le projet en
applique.

**Règlement européen sur l'IA (règlement (UE) 2024/1689, dit « AI Act »).** Il classe les
systèmes d'IA par niveau de risque. Un score de risque de résiliation de comptes
**professionnels**, qui sert à prioriser des relances faites par des humains, ne relève ni des
pratiques interdites (article 5) ni des usages à haut risque de l'annexe III (emploi, accès aux
services essentiels, solvabilité des personnes physiques…) : c'est un système à **risque
minimal**. L'obligation qui s'applique est la **maîtrise de l'IA** (article 4) : les CSM qui
reçoivent les priorités doivent savoir lire un score et ses limites ; c'est le rôle des
explications livrées avec chaque priorité (§10.2.1). Cette classification est à revoir si
l'usage change (scoring de personnes physiques, action déclenchée sans intervention humaine).

**Lignes directrices en matière d'éthique pour une IA digne de confiance** (groupe d'experts de
haut niveau sur l'IA, Commission européenne, 2019). Elles posent sept exigences :

| Exigence | Application dans le projet |
|---|---|
| Action humaine et contrôle humain | Le modèle n'exécute aucune action (D3, testé en §4.1) ; le CSM peut contester une priorité grâce à son explication (§10.2.1) |
| Robustesse technique et sécurité | Validation croisée (§8), tests unitaires, garde-fou `/ready` de l'API (§10.4), surveillance de la dérive (§13) |
| Respect de la vie privée et gouvernance des données | Portique RGPD avant ingestion (§3.2), minimisation (§4.1), cycle de vie documenté (§7.11) |
| Transparence | Chaque score est livré avec ses raisons (§9.6, §10.2.1) ; model cards ; journal de bord |
| Diversité, non-discrimination et équité | Variables écartées (§4.3) ; audit par segment (§12.5) |
| Bien-être sociétal et environnemental | Action bienveillante et réversible (§4.2) ; coût énergétique mesuré et pris en compte (§8.6, §11.4) |
| Responsabilité | Décisions D1–D15 tracées ; données et modèles versionnés (empreintes, DVC, MLflow) ; registre de traitement à valider par un DPO |

**En France : la CNIL.** Le RGPD est appliqué en §4.1. Les recommandations de la CNIL sur le
développement des systèmes d'IA (fiches pratiques publiées depuis 2024) insistent sur une
finalité déterminée, une base légale, la minimisation et l'information des personnes. Le
portique RGPD (`data/rgpd/rgpd_gate_manifest.json`) conclut qu'une analyse d'impact (AIPD) n'est
pas obligatoire (un seul critère de la liste de la CNIL réuni) et liste trois points à traiter
côté client : base légale et finalité à documenter formellement, registre de traitement, contrat
de sous-traitance en cas d'hébergement par un tiers.

**Porter ces risques à la connaissance des acteurs.** Ce cadre, le registre de traitement (H04)
et les dilemmes ci-dessous sont à présenter au DPO et au commanditaire (direction Customer
Success) avant toute mise en service. Dans cet exercice, sans commanditaire réel, cette
validation n'a pas eu lieu : elle est une condition de mise en service.

### 4.6 Dilemmes éthiques

Un dilemme oppose deux exigences légitimes : on ne peut pas satisfaire l'une sans céder sur
l'autre. Quatre dilemmes concrets, l'arbitrage proposé et qui doit le valider :

| Dilemme | Arbitrage proposé | À valider par |
|---|---|---|
| **Aider les clients en difficulté, ou ne pas les solliciter inutilement** : viser un rappel de 80 % (D9) signale aussi des comptes fidèles (environ un signalé sur trois, §9.3) | Le rappel prime, parce que l'action est bienveillante et réversible (§4.2) ; le volume reste borné par la capacité CS (D10) | Direction CS |
| **Valeur ou égalité de traitement** : parmi les comptes signalés, l'appel du CSM va aux 150 plus fortes pertes attendues (D14) ; à risque égal, un petit compte reçoit un email plutôt qu'un appel | Priorité à la valeur, assumée et explicite ; aucun compte signalé n'est ignoré (email ciblé, D11) | Direction CS (arbitrage D14 à soumettre) |
| **Anticiper ou surveiller** : les données d'usage (connexions, heures) permettent d'aider, mais leur exploitation peut être perçue comme une surveillance | Données agrégées par compte, finalité unique (§4.1), aucune donnée par utilisateur ; information des clients à prévoir | DPO, juristes |
| **Un seuil unique ou un seuil par segment** : un segment moins bien servi (§12.5) pourrait justifier un seuil propre | Seuil unique, sans traitement différencié ; surveillance renforcée des segments signalés en §12.5 | Direction CS, DPO |

### 📓 Journal de bord — §4 Éthique et conformité [C2]

- **Cadre RGPD** : intérêt légitime, finalité unique, minimisation, aucune décision automatisée
  (vérifiée par un test exécutable).
- **Asymétrie éthique des erreurs** : un faux positif entraîne une sollicitation bienveillante ;
  un faux négatif prive le client d'une aide. Le choix d'un rappel élevé est donc aussi
  défendable du point de vue du client.
- **Variables écartées** : texte libre (D7), identifiant, et `pays` (aucun pouvoir prédictif,
  risque de traitement différencié).
- **Équité** : audit par segment en §12, lu avec prudence (petits effectifs).
- **Explicabilité au service du contrôle humain** : chaque priorité est livrée avec ses raisons
  et ses avertissements (§10.2.1). Le CSM peut ainsi contester une priorité, ce qui rend réelle
  l'intervention humaine exigée par l'article 22.
- **Cadre réglementaire** : AI Act (risque minimal, obligation de maîtrise de l'IA), lignes
  directrices européennes pour une IA digne de confiance (sept exigences, chacune reliée à un
  choix du projet), recommandations de la CNIL ; AIPD non obligatoire d'après le portique RGPD.
- **Dilemmes éthiques** : quatre arbitrages explicités (rappel ou sollicitation, valeur ou
  égalité, anticipation ou surveillance, seuil unique ou par segment), chacun avec son valideur.
- **Limite** : le registre de traitement est rédigé à titre d'exercice ; il devrait être validé
  par un DPO dans un contexte réel.

## 5. Chargement et compréhension des données [C3]

📘 **Explication — chargement des fichiers bruts**

On charge le fichier principal **exactement tel qu'il a été reçu**, grâce à la fonction
`load_raw` de `src/bronze.py` :

- `dtype=str` : toutes les colonnes sont lues comme du texte. Si on laissait pandas deviner les
  types, une valeur comme `"12,5 %"` deviendrait silencieusement une valeur manquante et on
  perdrait l'information nécessaire au diagnostic.
- `keep_default_na=False` : une cellule vide reste une chaîne vide `""` au lieu d'être convertie
  en `NaN`. On garde ainsi la trace exacte de ce qui était vide dans le fichier.
- Deux colonnes de traçabilité sont ajoutées (`_source_file`, `_ingested_at_utc`) : d'où vient
  chaque ligne, et quand elle a été lue.

Le catalogue des plans est chargé de la même manière, puis on compare les colonnes reçues au
dictionnaire de données de l'énoncé (29 colonnes attendues).

In [ ]:
from src.bronze import load_raw

brut = load_raw(FICHIERS_SOURCES["principal"], ingested_at=datetime.now(timezone.utc).isoformat())
catalogue_brut = pd.read_csv(FICHIERS_SOURCES["catalogue"], dtype=str, keep_default_na=False)

COLONNES_ENONCE = [
    "client_id", "date_souscription", "jour_souscription", "secteur", "pays", "taille_entreprise",
    "plan", "anciennete_mois", "sieges_souscrits", "utilisateurs_actifs", "taux_adoption_pct",
    "connexions_30j", "heures_usage_30j", "fonctionnalites_total", "fonctionnalites_utilisees",
    "nb_integrations", "derniere_connexion_jours", "tickets_support_90j", "delai_reponse_support_h",
    "csat", "retards_paiement_12m", "revenu_mensuel_recurrent_eur", "couleur_theme_interface",
    "code_datacenter", "groupe_experimentation", "commentaire_csm", "sante_compte_fin_periode",
    "valeur_vie_client_eur", "churn",
]
colonnes_metier = [c for c in brut.columns if not c.startswith("_")]
print(f"Fichier principal : {brut.shape[0]} lignes x {len(colonnes_metier)} colonnes métier")
print("Colonnes conformes au dictionnaire de l'énoncé :", set(colonnes_metier) == set(COLONNES_ENONCE))
print(f"Catalogue : {catalogue_brut.shape[0]} plans")
display(catalogue_brut)
display(brut[colonnes_metier].head(3))

📘 **Explication — repérer les défauts avant de les corriger**

Avant de nettoyer, on **montre** les défauts annoncés par l'énoncé, colonne par colonne. Pour
chaque colonne délicate, la cellule affiche quelques valeurs distinctes du fichier brut :

- nombres écrits avec une virgule, un symbole `%` ou `€`, ou une unité (`h`) ;
- dates dans trois formats différents ;
- casse et espaces hétérogènes (`" pro "`, `"PRO"`) ;
- secteurs dont l'accent a été corrompu à l'encodage.

Elle compte aussi les **doublons stricts** (lignes identiques sur toutes les colonnes métier)
et les cellules vides par colonne.

À retenir : documenter un défaut avant de le corriger permet de vérifier ensuite que la
correction a bien fonctionné (§7).

In [ ]:
from src.silver import business_columns

for col in ["taux_adoption_pct", "revenu_mensuel_recurrent_eur", "delai_reponse_support_h",
            "date_souscription", "plan", "secteur"]:
    exemples = brut[col].drop_duplicates().head(6).tolist()
    print(f"{col:30s} -> {exemples}")

n_doublons = int(brut.duplicated(subset=business_columns(brut)).sum())
print(f"\nDoublons stricts sur les colonnes métier : {n_doublons}")

vides = (brut[colonnes_metier] == "").mean().mul(100).round(1)
print("\nPart de cellules vides par colonne (%), colonnes concernées :")
print(vides[vides > 0].sort_values(ascending=False).to_string())

## 6. Analyse exploratoire (EDA) : visualisations et interprétation — journal de bord [C3]

Pour explorer, il faut d'abord pouvoir lire les nombres : on applique donc dès maintenant la
chaîne de nettoyage de `src/silver.py`, **sans imputation** des valeurs manquantes. Chaque étape
de ce nettoyage est détaillée et justifiée en §7 ; ici, on s'en sert seulement pour observer.

La section procède en deux temps, du général au particulier :

1. **§6.1 Portrait de la donnée** : ce que contient le jeu, variable par variable (complétude,
   plage, forme des distributions, modalités), pour juger s'il est **exploitable** ;
2. **§6.2 à §6.4 Analyse détaillée** : relations de chaque variable avec la cible (§6.2), relations
   des variables entre elles par la matrice de corrélation (§6.3), examen critique de la variable
   la plus prédictive (§6.4) ;
3. **§6.5 Tri des valeurs atypiques et incohérentes** : ce que l'on garde, ce que l'on écarte, et
   pourquoi.

Outils : `pandas` pour les calculs (`describe`, `quantile`, `skew`, `value_counts`, `corr`),
`matplotlib` pour les graphiques.

### 6.1 Portrait de la donnée : que contient le jeu, et est-il exploitable ?

Avant de chercher ce qui explique le churn, on dresse le **portrait** du jeu de données : la
cible, les valeurs manquantes, puis la **distribution** de chaque variable. Une distribution
répond à la question : les valeurs sont-elles assez variées, assez complètes et assez
plausibles pour qu'un modèle puisse en tirer quelque chose ?

📘 **Explication — vue typée pour l'exploration**

`clean_silver` enchaîne six opérations (détaillées en §7) : suppression des doublons, lecture des
dates, conversion des nombres écrits en texte, normalisation des catégories, jointure avec le
catalogue, traitement des valeurs manquantes.

Le paramètre **`impute_medians=False`** est important : les valeurs manquantes restent
manquantes (`NaN`). Les remplacer maintenant par une médiane calculée sur **toutes** les lignes
reviendrait à utiliser des informations du futur jeu de test ; on le fera plus tard, dans le
pipeline du modèle, avec une médiane calculée sur le seul jeu d'entraînement.

La cellule affiche ensuite le taux de churn : c'est la première information à connaître sur une
cible binaire, car elle conditionne le choix des métriques.

In [ ]:
from src.silver import clean_silver

silver, rapport_silver = clean_silver(brut, catalogue_brut, impute_medians=False)
silver["churn"] = silver["churn"].astype(int)          # cible binaire 0/1 (lue comme nombre décimal)
print(f"Vue typée : {silver.shape[0]} comptes (après suppression de {rapport_silver['n_doublons']} doublons)")

taux_churn = silver["churn"].mean()
fig, ax = plt.subplots(figsize=(5, 3))
comptes = silver["churn"].value_counts().sort_index()
ax.bar(["Reste client (0)", "Résilie (1)"], comptes.values,
       color=[COULEURS["non_churn"], COULEURS["churn"]])
for i, v in enumerate(comptes.values):
    ax.text(i, v, f"{v} ({v / len(silver):.0%})", ha="center", va="bottom")
ax.set_title("Répartition de la cible churn")
ax.set_ylabel("Nombre de comptes")
sauver_figure(fig, "01_repartition_cible")
plt.show()
print(f"Taux de churn : {taux_churn:.1%} -> classes déséquilibrées (environ 1 compte sur 4 résilie)")

**Lecture.** 28 % des comptes résilient. Les classes sont déséquilibrées : un modèle qui
prédirait « aucun churn » aurait 72 % d'exactitude sans rien apprendre. C'est pourquoi on
n'utilisera **pas l'exactitude** (accuracy) comme métrique, mais la **PR-AUC** (aire sous la
courbe précision-rappel), plus informative quand la classe d'intérêt est minoritaire, en
complément de la ROC-AUC exigée par l'énoncé.

📘 **Explication — valeurs manquantes**

On mesure, pour chaque colonne, la part de valeurs manquantes **après** conversion des types.
Ce chiffre est différent du nombre de cellules vides du §5 : une valeur illisible convertie
proprement n'est plus manquante, tandis qu'une cellule vide le reste.

`commentaire_csm` est traité à part : un texte vide n'est pas une valeur « manquante » au sens
statistique, c'est simplement l'absence de commentaire. Cette colonne n'entrera de toute façon
pas dans le modèle (D7).

In [ ]:
manquants = silver.drop(columns=["commentaire_csm"]).isna().mean().mul(100)
manquants = manquants[manquants > 0].sort_values()

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(manquants.index, manquants.values, color=COULEURS["neutre"])
for i, v in enumerate(manquants.values):
    ax.text(v, i, f" {v:.1f} %", va="center")
ax.set_xlabel("% de valeurs manquantes")
ax.set_title("Valeurs manquantes après conversion des types")
sauver_figure(fig, "02_valeurs_manquantes")
plt.show()
print(f"commentaire_csm vide : {(silver['commentaire_csm'].fillna('').str.strip() == '').mean():.0%} des comptes")

**Lecture.** Six variables ont entre 4 et 10 % de valeurs manquantes. C'est assez pour ne pas
supprimer les lignes concernées (on perdrait jusqu'à un tiers des comptes en cumulé), et assez
peu pour qu'une imputation simple soit raisonnable. La stratégie retenue, colonne par colonne,
est détaillée en §7.6.

📘 **Explication — le portrait des variables numériques**

`pandas` résume chaque variable en une ligne. On lit trois familles d'indicateurs :

- **Complétude et variété** : part de valeurs manquantes, nombre de valeurs distinctes, part de la
  valeur la plus fréquente. Une variable dont une seule valeur couvre 95 % des lignes est
  **quasi constante** : elle ne peut rien distinguer.
- **Position et dispersion** : minimum, médiane, moyenne, maximum, écart-type. Une moyenne très
  au-dessus de la médiane signale quelques valeurs très grandes qui tirent la moyenne.
- **Forme** : le coefficient d'**asymétrie** (*skewness*). Proche de 0, la distribution est à peu
  près symétrique ; au-delà de 1, elle est étirée vers la droite (beaucoup de petites valeurs,
  quelques très grandes) ; au-delà de 2, très fortement.

Deux contrôles complètent le tableau :

- **valeurs atypiques** selon la règle de Tukey (au-delà de 1,5 écart interquartile sous le
  premier quartile ou au-dessus du troisième). C'est un **repérage**, pas une condamnation : dans
  une distribution étirée, la règle signale la queue naturelle de la distribution ;
- **respect des plages** du dictionnaire de l'énoncé, lues dans la base de connaissance
  (`knowledge/base_connaissance.yaml`) : une valeur hors plage serait une erreur de donnée.

La colonne **diagnostic** traduit ces chiffres en une lecture : quasi constante, discrète (au
plus 10 valeurs), peu asymétrique, asymétrique, très asymétrique.

In [ ]:
from src.explain import charger_base_connaissance
from src.gold import CATEGORICAL_CANDIDATES

VARIABLES_NUMERIQUES = [
    "anciennete_mois", "sieges_souscrits", "utilisateurs_actifs", "taux_adoption_pct",
    "connexions_30j", "heures_usage_30j", "fonctionnalites_total", "fonctionnalites_utilisees",
    "nb_integrations", "derniere_connexion_jours", "tickets_support_90j", "delai_reponse_support_h",
    "csat", "retards_paiement_12m", "revenu_mensuel_recurrent_eur", "sante_compte_fin_periode",
]
VARIABLES_CATEGORIELLES = list(CATEGORICAL_CANDIDATES)
PLAGES = {nom: v["plage"] for nom, v in charger_base_connaissance()["variables"].items() if "plage" in v}


def diagnostic_distribution(ligne) -> str:
    """Lecture d'une distribution à partir de son portrait."""
    if ligne["valeur dominante %"] >= 95:
        return "quasi constante"
    if ligne["valeurs distinctes"] <= 10:
        return f"discrète ({ligne['valeurs distinctes']} valeurs)"
    if abs(ligne["asymétrie"]) >= 2:
        return "très asymétrique"
    return "asymétrique" if abs(ligne["asymétrie"]) >= 1 else "peu asymétrique"


def portrait_numerique(df: pd.DataFrame, colonnes: list[str]) -> pd.DataFrame:
    """Une ligne par variable : complétude, position, dispersion, forme, atypiques, plage."""
    x = df[colonnes]
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    hors_tukey = (x < q1 - 1.5 * (q3 - q1)) | (x > q3 + 1.5 * (q3 - q1))
    portrait = pd.DataFrame({
        "manquants %": x.isna().mean() * 100,
        "valeurs distinctes": x.nunique(),
        "valeur dominante %": [x[c].value_counts(normalize=True).iloc[0] * 100 for c in colonnes],
        "zéros %": (x == 0).mean() * 100,
        "min": x.min(), "médiane": x.median(), "moyenne": x.mean(), "max": x.max(),
        "écart-type": x.std(),
        "asymétrie": x.skew(),
        "atypiques (Tukey) %": hors_tukey.mean() * 100,
    })
    portrait["dans la plage"] = [
        ("oui" if PLAGES[c][0] <= x[c].min() and x[c].max() <= PLAGES[c][1] else "NON")
        if c in PLAGES else "—" for c in colonnes]
    portrait["diagnostic"] = portrait.apply(diagnostic_distribution, axis=1)
    return portrait


portrait = portrait_numerique(silver, VARIABLES_NUMERIQUES + ["valeur_vie_client_eur"])
display(portrait.round(2))
print("Répartition des diagnostics :", portrait["diagnostic"].str.split(" \\(").str[0].value_counts().to_dict())
print("Variables hors plage du dictionnaire :", portrait.index[portrait["dans la plage"] == "NON"].tolist() or "aucune")

**Lecture — le portrait numérique.**

- **Aucune variable n'est quasi constante** : la valeur la plus fréquente ne dépasse jamais 59 %
  des comptes renseignés (`retards_paiement_12m` : 59 % n'ont aucun retard). Toutes les
  variables varient assez pour porter une information.
- **Toutes les valeurs sont dans les plages du dictionnaire** de l'énoncé : les valeurs les plus
  grandes (898 sièges, 829 utilisateurs actifs, 89 403 € de revenu mensuel, 200 jours sans
  connexion) sont extrêmes mais **légitimes**. Les 12 à 16 % de valeurs « atypiques » selon Tukey
  sur ces variables sont la queue naturelle de distributions étirées, pas des erreurs : on ne
  les supprime pas.
- **Trois formes coexistent**. Deux variables sont proches de la symétrie (ancienneté, taux
  d'adoption ; la santé de fin de période aussi, mais elle est écartée au §6.2). Trois sont
  **discrètes** (4 à 8 valeurs : CSAT, nombre de fonctionnalités du plan, retards de paiement).
  Toutes les autres sont **étirées vers la droite**, et six le sont très fortement (asymétrie de
  2,3 à 5,6) : sièges, utilisateurs actifs, heures d'usage, jours depuis la dernière connexion,
  revenu mensuel et valeur vie client. Pour ces six variables, la moyenne est 2 à 7 fois la
  médiane (74 sièges en moyenne pour une médiane de 28) : **la médiane décrit mieux le compte
  type**, ce qui justifie l'imputation par la médiane retenue au §7.6.

📘 **Explication — voir les distributions**

Le tableau résume ; l'**histogramme** montre. Pour chaque variable numérique, on découpe la plage
des valeurs en intervalles et on compte les comptes dans chacun. Le trait pointillé marque la
médiane.

Le second graphique reprend les six variables très asymétriques en **échelle logarithmique**
(`log(1 + x)`, qui accepte les zéros) : une distribution écrasée contre zéro devient lisible, et
l'on voit la forme qu'elle prend une fois « dépliée » : une cloche, plusieurs paliers, deux
populations. C'est aussi la transformation appliquée à la cible de la régression CLV au §9.7.

La cellule vérifie enfin par le calcul les deux formes remarquables que montre ce second
graphique : les paliers des sièges, et le groupe de comptes à usage quasi nul.

In [ ]:
colonnes_hist = portrait.index.tolist()
fig, axes = plt.subplots(4, 5, figsize=(16, 11))
for ax, col in zip(axes.ravel(), colonnes_hist):
    valeurs = silver[col].dropna()
    n_valeurs = valeurs.nunique()
    bornes = np.arange(valeurs.min() - 0.5, valeurs.max() + 1.5) if n_valeurs <= 10 else 30
    ax.hist(valeurs, bins=bornes, color=COULEURS["non_churn"], alpha=0.8)
    ax.axvline(valeurs.median(), color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{col}\n{portrait.loc[col, 'diagnostic']} (asym. {portrait.loc[col, 'asymétrie']:.1f})", fontsize=9)
    ax.tick_params(labelsize=8)
for ax in axes.ravel()[len(colonnes_hist):]:
    ax.set_visible(False)
fig.suptitle("Distribution de chaque variable numérique (trait pointillé : médiane)")
fig.tight_layout()
sauver_figure(fig, "01b_distributions_numeriques")
plt.show()

tres_asymetriques = portrait.index[portrait["diagnostic"] == "très asymétrique"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), tres_asymetriques):
    ax.hist(np.log1p(silver[col].dropna()), bins=30, color=COULEURS["non_churn"], alpha=0.8)
    ax.set_title(f"log(1 + {col})", fontsize=9)
fig.suptitle("Les six variables très asymétriques, en échelle logarithmique")
fig.tight_layout()
sauver_figure(fig, "01c_distributions_log")
plt.show()
print("Rapport moyenne / médiane :", (portrait.loc[tres_asymetriques, "moyenne"] / portrait.loc[tres_asymetriques, "médiane"]).round(1).to_dict())
print("\nSièges souscrits par taille d'entreprise (paliers visibles en échelle log) :")
display(silver.groupby("taille_entreprise")["sieges_souscrits"].agg(["min", "median", "max"]))
usage_quasi_nul = silver["heures_usage_30j"] < 1
print(f"Usage < 1 h sur 30 jours : {usage_quasi_nul.sum()} comptes "
      f"({usage_quasi_nul.sum() / silver['heures_usage_30j'].notna().sum():.0%} des comptes renseignés), "
      f"taux de churn {silver.loc[usage_quasi_nul, 'churn'].mean():.0%} contre {taux_churn:.0%} en moyenne")

**Lecture — les distributions.**

- Les variables d'**usage et de taille** ont la forme typique des données SaaS : beaucoup de
  petits comptes, quelques très gros. L'échelle logarithmique révèle ce que l'échelle brute
  écrase :
  - **revenu mensuel et valeur vie client** prennent une forme en cloche : l'information est là,
    elle est seulement concentrée sur une petite plage ;
  - **les sièges** présentent plusieurs paliers : ce sont les segments de taille d'entreprise
    (de 1 à 11 sièges pour une TPE, de 6 à 59 pour une PME, de 30 à 219 pour une ETI, de 125 à
    898 pour une GE, tableau ci-dessus) ;
  - **les heures d'usage** séparent deux populations : un groupe à usage quasi nul (moins d'une
    heure sur 30 jours pour 29 % des comptes renseignés), puis une cloche. Ce groupe résilie à
    47 %, contre 28 % en moyenne : la forme même de la distribution porte déjà du signal ;
  - les trous de `utilisateurs_actifs` viennent des petites valeurs entières (0, 1, 2…), pas
    d'un défaut.
- `derniere_connexion_jours` reste concentrée même en échelle log : la moitié des comptes s'est
  connectée il y a 2 jours ou moins, et une queue de comptes décroche. C'est cette queue qui
  porte le signal de désengagement (§6.4).
- Les variables discrètes (CSAT de 1 à 5, 4 niveaux de `fonctionnalites_total`, retards de 0 à
  7) sont bien réparties entre leurs valeurs, sauf les retards, concentrés sur 0.

**Conséquence pour la modélisation.** Les distributions sont exploitables, mais l'asymétrie a un
coût pour un modèle linéaire : la standardisation (§8) change l'échelle, pas la forme, et les
très gros comptes pèsent plus que les autres dans l'ajustement. Le modèle v2 utilise les
variables brutes ; une transformation logarithmique des variables de taille est une piste à
tester en validation croisée à la prochaine version (§14, recommandation 7), et non à
appliquer après coup.

📘 **Explication — le portrait des variables catégorielles**

Pour une variable catégorielle, la distribution est la **fréquence de chaque modalité**
(`value_counts`). On y cherche deux défauts :

- une **modalité écrasante**, qui rend la variable presque constante ;
- des **modalités rares** (moins de 1 % des comptes) : le modèle les verrait trop peu pour en
  apprendre quoi que ce soit, et l'encodage *one-hot* (§8) créerait des colonnes presque vides.

La modalité `Inconnu` a été ajoutée par le nettoyage pour les valeurs manquantes de `secteur`
et `pays` (§7.6).

In [ ]:
portrait_cat = pd.DataFrame([{
    "variable": col,
    "modalités": silver[col].nunique(),
    "modalité dominante": silver[col].value_counts().index[0],
    "dominante %": silver[col].value_counts(normalize=True).iloc[0] * 100,
    "modalité la plus rare": silver[col].value_counts().index[-1],
    "plus rare %": silver[col].value_counts(normalize=True).iloc[-1] * 100,
    "Inconnu %": (silver[col] == "Inconnu").mean() * 100,
} for col in VARIABLES_CATEGORIELLES]).set_index("variable")
display(portrait_cat.round(1))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), VARIABLES_CATEGORIELLES):
    frequences = silver[col].value_counts(normalize=True).mul(100)
    ax.bar(frequences.index.astype(str), frequences.values, color=COULEURS["non_churn"], alpha=0.8)
    ax.axhline(1, color=COULEURS["churn"], linestyle="--", linewidth=1)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("% des comptes")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
fig.suptitle("Fréquence des modalités des variables catégorielles (trait rouge : seuil de rareté de 1 %)")
fig.tight_layout()
sauver_figure(fig, "01d_distributions_categorielles")
plt.show()

**Lecture — les catégorielles.** Aucune modalité n'est rare : la plus petite, `Inconnu` dans
`pays`, couvre encore 4 % des comptes (200 comptes). Aucune variable n'est écrasée par une
modalité, sauf `pays`, où la France représente 54 % des comptes. Le portefeuille est surtout
composé de TPE et de PME (75 %), et un compte sur neuf est en plan Enterprise.

Les quatre variables sans lien plausible avec le churn (jour de souscription, couleur du thème,
datacenter, groupe d'A/B test) ont des modalités **presque uniformes** (de 13,8 à 15,1 % par jour de
la semaine, par exemple) : c'est la signature d'une variable attribuée au hasard ou sans lien
avec le client. Le §6.2 vérifie par un test qu'elles n'ont aucun effet sur le churn.

**Verdict du portrait.** Le jeu est **exploitable** : 5 000 comptes, une cible déséquilibrée mais
pas rare (28 %), au plus 10 % de manquants par variable, aucune variable constante, aucune valeur
hors plage, aucune modalité rare. Ses deux particularités, les distributions très étirées des
variables de taille et d'usage et les variables uniformes suspectes, orientent l'analyse
détaillée qui suit.

### 6.2 Dans le détail : quelles variables séparent les comptes qui partent ?

Le portrait décrit chaque variable seule. On regarde maintenant chacune **par rapport à la
cible** : pouvoir séparateur (AUC univariée), distributions comparées, tests statistiques des
leurres.

📘 **Explication — pouvoir séparateur de chaque variable (AUC univariée)**

Pour chaque variable numérique prise **seule**, on calcule l'AUC ROC obtenue en l'utilisant
directement comme score de churn :

- **0,5** = la variable ne sépare pas du tout les comptes qui partent de ceux qui restent ;
- **1,0** = séparation parfaite.

On prend `max(AUC, 1 − AUC)` pour ne pas dépendre du sens de la relation (une ancienneté
*faible* est associée au churn, un nombre de tickets *élevé* aussi).

Pourquoi c'est utile :
1. **Détecter une fuite de données.** Une variable qui sépare presque parfaitement les deux
   classes est suspecte : elle contient souvent une information connue seulement *après* la
   résiliation. L'énoncé prévient qu'une AUC quasi parfaite « cache presque toujours une fuite ».
2. **Repérer les variables sans pouvoir prédictif**, candidates au statut de leurre.

In [ ]:
from sklearn.metrics import roc_auc_score

lignes = []
for col in VARIABLES_NUMERIQUES:
    valides = silver[[col, "churn"]].dropna()
    auc = roc_auc_score(valides["churn"], valides[col])
    lignes.append({"variable": col, "auc_univariee": round(max(auc, 1 - auc), 3),
                   "sens": "plus élevé = plus de churn" if auc >= 0.5 else "plus faible = plus de churn"})
auc_univ = pd.DataFrame(lignes).sort_values("auc_univariee", ascending=False).reset_index(drop=True)
display(auc_univ)

**Lecture — le piège de fuite.** `sante_compte_fin_periode` atteint une AUC de **0,999** à elle
seule, contre 0,53 à 0,76 pour toutes les autres variables. Son nom le confirme : un score de
santé calculé **en fin de période** est mesuré après la décision que le modèle doit anticiper.
Au moment réel du scoring, cette information n'existe pas encore. **Elle est exclue des deux
modèles** (churn et CLV), non parce qu'elle est trop corrélée, mais parce qu'elle **n'est pas
disponible au moment de la décision**. Elle reste dans la table Gold pour la traçabilité.

Les autres variables ont des AUC plausibles pour des signaux d'engagement observables avant
l'échéance (intégrations, ancienneté, usage, satisfaction).

📘 **Explication — distributions comparées**

Des boîtes à moustaches (*boxplots*) comparent la distribution de six variables entre les comptes
qui restent (bleu) et ceux qui partent (rouge). La boîte contient la moitié centrale des
valeurs, le trait central est la médiane.

On cherche des différences **visibles et interprétables par le métier** : si elles existent, le
modèle a quelque chose à apprendre, et on pourra expliquer ses résultats aux CSM.

In [ ]:
variables_cles = ["derniere_connexion_jours", "nb_integrations", "anciennete_mois",
                  "csat", "tickets_support_90j", "taux_adoption_pct"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), variables_cles):
    donnees = [silver.loc[silver["churn"] == k, col].dropna() for k in (0, 1)]
    boites = ax.boxplot(donnees, patch_artist=True, showfliers=False)
    for boite, couleur in zip(boites["boxes"], [COULEURS["non_churn"], COULEURS["churn"]]):
        boite.set_facecolor(couleur)
        boite.set_alpha(0.7)
    ax.set_xticks([1, 2], ["Reste", "Résilie"])
    ax.set_title(col)
fig.suptitle("Distribution de six variables selon le statut churn")
fig.tight_layout()
sauver_figure(fig, "03_distributions_par_churn")
plt.show()
display(silver.groupby("churn")[variables_cles].median().T.rename(columns={0: "médiane reste", 1: "médiane résilie"}))

**Lecture.** Les comptes qui résilient se connectent moins récemment, ont moins
d'intégrations, sont plus récents, moins satisfaits (CSAT), ouvrent plus de tickets et adoptent
moins la plateforme. Ces signaux ont un **sens métier clair** : ce sont exactement ceux qu'un
CSM expérimenté surveille. Le modèle devra les combiner.

📘 **Explication — identifier les leurres par un test statistique**

L'énoncé annonce des **variables leurres** : sans pouvoir prédictif, à identifier et à exclure.
On ne les choisit pas « à l'œil » : on applique un test adapté au type de chaque variable.

- **Variables numériques** : corrélation *point-bisériale* avec `churn` (c'est une corrélation
  de Pearson quand l'une des deux variables est binaire). Seuil : |r| < 0,03.
- **Variables catégorielles** : **V de Cramér**, calculé à partir du test du χ² d'indépendance.
  Il mesure la *force* de l'association entre 0 (aucune) et 1 (totale). On l'utilise plutôt
  que la seule p-value, parce qu'avec 5 000 lignes une association minuscule peut être
  « significative » sans avoir aucun intérêt pratique. Seuil : V < 0,05.

La fonction `identify_leurres` de `src/gold.py` applique ces deux tests. Pour ce test
descriptif, les valeurs manquantes sont remplacées temporairement par la médiane (le calcul de
corrélation ne les accepte pas).

In [ ]:
from scipy import stats
from src.gold import CATEGORICAL_CANDIDATES, NUMERIC_CANDIDATES, cramers_v, identify_leurres

silver_test = silver.fillna(silver[NUMERIC_CANDIDATES].median())
leurres_detectes = identify_leurres(silver_test)

tests = []
for col in NUMERIC_CANDIDATES:
    r = stats.pointbiserialr(silver_test["churn"], silver_test[col].astype(float))[0]
    tests.append({"variable": col, "type": "numérique", "test": "corrélation point-bisériale",
                  "effet": round(abs(r), 3), "leurre": col in leurres_detectes})
for col in CATEGORICAL_CANDIDATES:
    v = cramers_v(pd.crosstab(silver_test[col], silver_test["churn"]).values)
    tests.append({"variable": col, "type": "catégorielle", "test": "V de Cramér",
                  "effet": round(v, 3), "leurre": col in leurres_detectes})
tests = pd.DataFrame(tests).sort_values(["leurre", "effet"], ascending=[False, True]).reset_index(drop=True)
display(tests)
print("Variables signalées par le test :", leurres_detectes)

📘 **Explication — visualiser un leurre**

Un leurre catégoriel se reconnaît facilement sur un graphique : le taux de churn est le même
dans toutes ses modalités. On compare quatre leurres (couleur du thème, datacenter, groupe
d'A/B test, jour de souscription) à deux variables réellement liées au churn (taille
d'entreprise, plan). La ligne pointillée représente le taux de churn global (28 %).

In [ ]:
a_comparer = ["couleur_theme_interface", "code_datacenter", "groupe_experimentation",
              "jour_souscription", "taille_entreprise", "plan"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col in zip(axes.ravel(), a_comparer):
    taux = silver.groupby(col)["churn"].mean().sort_values()
    couleur = COULEURS["neutre"] if col in leurres_detectes else COULEURS["churn"]
    ax.bar(taux.index.astype(str), taux.values, color=couleur)
    ax.axhline(taux_churn, linestyle="--", color="black", linewidth=1)
    ax.set_ylim(0, 0.6)
    ax.set_title(f"{col}\n({'leurre' if col in leurres_detectes else 'signal réel'})")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Taux de churn par modalité : leurres (gris) contre variables informatives (rouge)")
fig.tight_layout()
sauver_figure(fig, "04_leurres_vs_signal")
plt.show()

**Lecture — les leurres.** Le test signale **six** variables : les cinq variables sans lien
plausible avec la résiliation (`jour_souscription`, `pays`, `code_datacenter`,
`couleur_theme_interface`, `groupe_experimentation`) et `sieges_souscrits`.

- Les **cinq premières sont exclues** du modèle (§7.8), comme le demande l'énoncé. Le §8
  vérifie en validation croisée que les retirer ne coûte rien.
- **`sieges_souscrits` est conservée** : sa corrélation *directe* avec le churn est nulle, mais
  c'est une variable métier centrale (taille du contrat), qui porte l'information de la
  régression CLV et entre dans le calcul du taux d'adoption. Un test univarié ne voit pas ces
  interactions : on vérifie en §8 que la garder ne dégrade pas le modèle.

### 6.3 Matrice de corrélation : relations et redondances entre variables

📘 **Explication — la matrice de corrélation**

Une **matrice de corrélation** croise toutes les variables numériques deux à deux. Chaque case
contient un coefficient entre −1 et +1 :

- **+1** : les deux variables montent ensemble ; **−1** : quand l'une monte, l'autre baisse ;
- **0** : aucune relation monotone.

Elle sert à quatre choses :

1. **repérer les variables qui évoluent ensemble**, et les regrouper en familles d'information ;
2. **détecter les variables redondantes** : deux variables corrélées au-delà de 0,8 disent
   presque la même chose. Elles n'apportent rien de plus au modèle, et elles gênent un modèle
   linéaire : l'effet peut se répartir entre elles de façon instable (*colinéarité*), ce qui
   brouille l'explication ;
3. **aider à choisir les variables** : on ajoute la cible `churn` comme dernière ligne ; sa
   corrélation avec chaque variable mesure, en première approche, l'information qu'elle apporte ;
4. **comprendre rapidement le jeu de données**, par familles plutôt que variable par variable.

**Quel coefficient ?** Le coefficient de **Pearson** mesure une relation *linéaire* et il est très
sensible aux valeurs extrêmes, or le §6.1 a montré six variables très asymétriques. On affiche donc
le coefficient de **Spearman**, calculé sur les **rangs** des valeurs : il capte toute relation
monotone et n'est pas dominé par les très gros comptes. Les deux sont comparés pour les paires les
plus liées. `sante_compte_fin_periode` est laissée de côté (fuite, §6.2).

Avant la matrice, on vérifie une redondance d'un autre type : les colonnes issues du catalogue des
plans (prix par siège, SLA, quota…) ne prennent-elles qu'une valeur par plan ?

In [ ]:
colonnes_catalogue = ["prix_mensuel_par_siege_eur", "fonctionnalites_incluses", "sla_reponse_h",
                      "quota_stockage_go", "support_dedie", "fonctionnalites_total"]
valeurs_par_plan = silver.groupby("plan")[colonnes_catalogue].nunique().max()
print("Nombre maximal de valeurs distinctes par plan :")
print(valeurs_par_plan.to_string())
print("fonctionnalites_total == fonctionnalites_incluses sur",
      f"{(silver['fonctionnalites_total'] == silver['fonctionnalites_incluses']).mean():.0%} des comptes")

numeriques = [c for c in VARIABLES_NUMERIQUES if c != "sante_compte_fin_periode"]
colonnes_corr = numeriques + ["churn"]
spearman = silver[colonnes_corr].corr(method="spearman")
pearson = silver[colonnes_corr].corr(method="pearson")

triangle = np.tril(np.ones(spearman.shape, dtype=bool))          # moitié basse + diagonale
fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(spearman.where(triangle), cmap="RdBu_r", vmin=-1, vmax=1)
for i in range(len(colonnes_corr)):
    for j in range(i + 1):
        rho = spearman.iloc[i, j]
        ax.text(j, i, f"{rho:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(rho) >= 0.6 else "black")
ax.set_xticks(range(len(colonnes_corr)), colonnes_corr, rotation=90, fontsize=8)
ax.set_yticks(range(len(colonnes_corr)), colonnes_corr, fontsize=8)
ax.axhline(len(colonnes_corr) - 1.5, color="black", linewidth=1)   # isole la ligne de la cible
fig.colorbar(image, ax=ax, shrink=0.8, label="corrélation de Spearman")
ax.set_title("Matrice de corrélation des variables numériques et de la cible (Spearman)")
ax.grid(False)
sauver_figure(fig, "05_correlations")
plt.show()

haut = np.triu(np.ones(spearman.shape, dtype=bool), k=1)
paires = pd.DataFrame({
    "spearman": spearman.drop(index="churn", columns="churn").where(haut[:-1, :-1]).stack(),
    "pearson": pearson.drop(index="churn", columns="churn").where(haut[:-1, :-1]).stack(),
})
paires = paires.reindex(paires["spearman"].abs().sort_values(ascending=False).index)
print("Paires de variables les plus liées (|Spearman| >= 0,6) :")
display(paires[paires["spearman"].abs() >= 0.6].round(2))
print(f"Paires redondantes (|Spearman| >= 0,8) : {int((paires['spearman'].abs() >= 0.8).sum())}")

lien_cible = pd.DataFrame({"spearman": spearman["churn"], "pearson": pearson["churn"]}).drop(index="churn")
print("\nCorrélation de chaque variable avec la cible churn :")
display(lien_cible.reindex(lien_cible["spearman"].abs().sort_values(ascending=False).index).round(3))

print("Délai de réponse du support et SLA du plan : Spearman =",
      round(silver[["delai_reponse_support_h", "sla_reponse_h"]].corr(method="spearman").iloc[0, 1], 2))
print("Délai de réponse médian par plan (h) :",
      silver.groupby("plan")["delai_reponse_support_h"].median().round(1).to_dict())

**Lecture — redondance du catalogue.** Les cinq colonnes issues du catalogue ne prennent **qu'une
valeur par plan** : elles répètent l'information de `plan` sous une autre forme. Elles sont exclues
(§7.8). `fonctionnalites_total` est gardée (identique à `fonctionnalites_incluses`, on n'en conserve
qu'une) car elle sert au ratio d'utilisation des fonctionnalités.

**Lecture — la matrice, selon ses quatre usages.**

1. **Les variables évoluent en trois familles.**
   - **Taille du compte** : sièges, utilisateurs actifs et revenu mensuel (Spearman de 0,79 à
     0,95). Un gros contrat a plus d'utilisateurs et paie plus.
   - **Intensité d'usage** : connexions, heures d'usage et taux d'adoption (0,84 à 0,95), liées
     négativement aux jours depuis la dernière connexion (−0,58 à −0,66).
   - **Relation client** : tickets et satisfaction (−0,59), ancienneté et intégrations (0,70),
     ancienneté et retards de paiement (−0,64).
   - Une relation surprend : `fonctionnalites_total` et `delai_reponse_support_h` (−0,82). Elle
     passe par le plan : le délai de réponse suit le SLA du plan (Spearman 0,82 ; délai médian de
     23,5 h en Starter contre 3,1 h en Enterprise), comme le nombre de fonctionnalités. Ce n'est pas
     un lien de cause entre les deux variables, c'est un **facteur commun**.
2. **Six paires sont redondantes** (|Spearman| ≥ 0,8). Les deux plus fortes sont sièges et revenu
   mensuel (0,95) et connexions et heures d'usage (0,95). **Le modèle v2 garde les deux variables
   de chaque paire.** Pour la prédiction, la colinéarité gêne peu un modèle régularisé (§9.1) ;
   l'effet du retrait de ces paires n'a pas été mesuré. Pour l'explication, c'est un point de
   vigilance : c'est exactement le mécanisme qui a rendu instables les deux ratios du §9.6.3.
   Retirer une variable de chaque paire est à tester à la prochaine version, par la procédure de
   sélection du §8 (§14, recommandation 7).
3. **Pour le choix des variables**, la ligne `churn` confirme l'AUC univariée du §6.2 : les liens
   les plus forts sont les intégrations (−0,41), l'ancienneté (−0,33) et l'usage (−0,29 à −0,31) ;
   les plus faibles, la taille du contrat (sièges −0,05, revenu −0,09). **Aucune variable ne
   dépasse 0,5 en valeur absolue** : il n'y a pas d'autre fuite du type de
   `sante_compte_fin_periode`, et le churn dépend d'une **combinaison** de signaux modérés, ce qui
   justifie un modèle multivarié plutôt qu'une règle sur une variable.
4. **En résumé**, le jeu se comprend en trois questions métier : quelle est la taille du compte,
   l'utilise-t-il, la relation se passe-t-elle bien ? La taille n'est presque pas liée au churn ;
   l'usage et la relation le sont.

Pearson donne des valeurs proches pour la taille du compte, mais plus faibles pour l'usage
(connexions et heures : 0,90 contre 0,95 ; adoption et heures : 0,68 contre 0,84) : ces relations
sont monotones mais non linéaires, et Pearson les sous-estime.

### 6.4 Examen critique de la variable la plus prédictive

📘 **Explication — examen critique de la variable la plus prédictive**

`derniere_connexion_jours` (nombre de jours depuis la dernière connexion) est l'une des
variables les plus liées au churn. Avant de lui faire confiance, on vérifie qu'elle n'est pas
elle aussi une fuite déguisée : si elle était mesurée *à l'échéance*, un client ayant déjà
décidé de partir aurait arrêté de se connecter, et la variable « connaîtrait » la décision.

L'histogramme compare les deux groupes et la cellule cherche une zone de séparation parfaite.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bornes = np.arange(0, 205, 5)
for k, nom in [(0, "non_churn"), (1, "churn")]:
    ax.hist(silver.loc[silver["churn"] == k, "derniere_connexion_jours"], bins=bornes, alpha=0.6,
            color=COULEURS[nom], label="Reste" if k == 0 else "Résilie", density=True)
max_fideles = silver.loc[silver["churn"] == 0, "derniere_connexion_jours"].max()
ax.axvline(max_fideles, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Jours depuis la dernière connexion")
ax.set_ylabel("Densité")
ax.legend()
ax.set_title("derniere_connexion_jours selon le statut churn")
sauver_figure(fig, "06_derniere_connexion")
plt.show()
au_dela = silver["derniere_connexion_jours"] > max_fideles
print(f"Maximum observé chez les comptes fidèles : {max_fideles:.0f} jours")
print(f"Comptes au-delà de ce maximum : {au_dela.sum()} ({au_dela.mean():.1%}), dont churn : {silver.loc[au_dela, 'churn'].mean():.0%}")
print(f"AUC univariée : {auc_univ.set_index('variable').loc['derniere_connexion_jours', 'auc_univariee']}")

**Lecture.** Au-delà de 75 jours sans connexion, tous les comptes observés résilient. Mais cela
ne concerne qu'une petite part des comptes, et l'AUC de la variable (0,67) reste très loin de la
séparation quasi parfaite du piège de fuite (0,999). Le dictionnaire la décrit comme « jours
depuis la dernière connexion », un signal de désengagement observable au moment du scoring
mensuel. **Je la conserve**, avec une réserve explicite : dans un déploiement réel, il faudrait
confirmer auprès du propriétaire de la donnée qu'elle est bien calculée à la date du scoring,
et non à l'échéance (point listé en §14).

### 6.5 Valeurs atypiques et incohérences : que retenir, qu'écarter ?

Le portrait (§6.1) a repéré des valeurs **extrêmes** ; l'analyse détaillée (§6.2 à §6.4), des
comportements tranchés. Avant de modéliser, il faut trier. Un modèle apprend une classe à partir
d'exemples : s'il apprend sur des événements **impossibles** (une saisie erronée) ou
**incompréhensibles** (une variable dont on ne sait pas ce qu'elle mesure), il en tire une règle
qui ne s'appliquera à aucune observation réelle. On distingue trois cas :

| Cas | Exemple | Traitement |
|---|---|---|
| **Valeur extrême mais possible** | un compte de 898 sièges | on la **garde** : c'est un vrai client, le modèle doit savoir le scorer |
| **Valeur impossible** | plus de retards de paiement que de mois facturés | on l'**écarte** : ni le modèle ni l'explication ne doivent s'appuyer dessus |
| **Valeur incompréhensible** | une mesure qui contredit sa définition | on la **garde avec réserve**, on explique pourquoi, et on remonte la question au propriétaire de la donnée |

Une valeur extrême se repère statistiquement (règle de Tukey, §6.1) ; une valeur impossible, non :
elle peut être tout à fait ordinaire prise seule (3 retards de paiement), et c'est sa combinaison
avec une autre colonne (1 mois d'ancienneté) qui la rend impossible. D'où des **règles de
cohérence** entre colonnes.

📘 **Explication — les règles de cohérence**

Chaque règle traduit une contrainte qui **doit** être vraie pour tout compte réel. Deux viennent
du dictionnaire de l'énoncé (« utilisateurs actifs ≤ sièges », « taux d'adoption = actifs /
sièges ») ; les autres, de la logique métier (on ne peut pas s'être connecté avant d'avoir
souscrit ; un mois facturé donne au plus un retard de paiement ; etc.).

Pour chaque règle, la cellule compte les comptes qui la **violent**, et leur taux de churn : si
les violations sont liées à la cible, les garder ferait apprendre au modèle un défaut de saisie
plutôt qu'un comportement client. Les règles portent sur la table Silver (avant toute
imputation), pour juger les données telles qu'elles ont été reçues.

In [ ]:
from src.features import masque_retards_impossibles

jours_semaine = dict(enumerate(["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]))
date_reference = silver["date_souscription"] + pd.to_timedelta(silver["anciennete_mois"] * 30.44, unit="D")
revenu_catalogue = silver["sieges_souscrits"] * silver["prix_mensuel_par_siege_eur"]
heures_par_connexion = silver["heures_usage_30j"] / silver["connexions_30j"].replace(0, np.nan)
sans_connexion = silver["connexions_30j"] == 0

REGLES_COHERENCE = {
    "utilisateurs actifs ≤ sièges (dictionnaire)":
        silver["utilisateurs_actifs"] > silver["sieges_souscrits"],
    "adoption = 100 × actifs / sièges, à 1 point près (dictionnaire)":
        (silver["taux_adoption_pct"] - 100 * silver["utilisateurs_actifs"] / silver["sieges_souscrits"]).abs() > 1,
    "fonctionnalités utilisées ≤ fonctionnalités offertes":
        silver["fonctionnalites_utilisees"] > silver["fonctionnalites_total"],
    "jour de souscription = jour de la semaine de la date":
        silver["date_souscription"].dt.dayofweek.map(jours_semaine) != silver["jour_souscription"],
    "date + ancienneté : même date d'observation (± 31 jours)":
        (date_reference - date_reference.median()).abs() > pd.Timedelta(days=31),
    "dernière connexion ≤ ancienneté":
        silver["derniere_connexion_jours"] > silver["anciennete_mois"] * 30.44,
    "aucune connexion ⇔ aucune heure d'usage":
        silver["heures_usage_30j"].notna() & (sans_connexion != (silver["heures_usage_30j"] == 0)),
    "aucun utilisateur actif ⇔ aucune connexion":
        (silver["utilisateurs_actifs"] == 0) != sans_connexion,
    "plus de 30 jours sans connexion ⇒ aucune connexion sur 30 jours":
        (silver["derniere_connexion_jours"] > 30) & ~sans_connexion,
    "revenu mensuel entre 0,5 et 2 × sièges × prix du plan":
        ~(silver["revenu_mensuel_recurrent_eur"] / revenu_catalogue).between(0.5, 2),
    "au plus 8 h d'usage par connexion":
        heures_par_connexion > 8,
    "valeur vie client ≥ revenu mensuel":
        silver["valeur_vie_client_eur"] < silver["revenu_mensuel_recurrent_eur"],
    "retards de paiement ≤ mois facturés, min(ancienneté, 12)":
        masque_retards_impossibles(silver),
}
controle_coherence = pd.DataFrame([{
    "règle": regle, "comptes en violation": int(masque.fillna(False).sum()),
    "% des comptes": masque.fillna(False).mean() * 100,
    "taux de churn des violations": silver.loc[masque.fillna(False), "churn"].mean() if masque.any() else np.nan,
} for regle, masque in REGLES_COHERENCE.items()])
display(controle_coherence.round(3))
print(f"Règles respectées par tous les comptes : {int((controle_coherence['comptes en violation'] == 0).sum())} "
      f"sur {len(controle_coherence)}")
print("Revenu mensuel / (sièges × prix du plan) : de",
      round(float((silver["revenu_mensuel_recurrent_eur"] / revenu_catalogue).min()), 2), "à",
      round(float((silver["revenu_mensuel_recurrent_eur"] / revenu_catalogue).max()), 2),
      "| heures par connexion au maximum :", round(float(heures_par_connexion.max()), 2))

**Lecture.** Douze règles sur treize sont respectées par **tous** les comptes. C'est la preuve
que les valeurs extrêmes du §6.1 ne sont pas des erreurs : un compte de 898 sièges paie bien
environ 898 × le prix de son plan (le revenu réel est compris entre 0,6 et 1,5 fois le prix
catalogue : remises et options), et ses utilisateurs actifs restent sous le nombre de sièges.
**On garde donc toutes les valeurs extrêmes.**

Une seule règle est violée, et massivement : **449 comptes (9 %) déclarent plus de retards de
paiement que de mois facturés**. Leur taux de churn (61 %) est plus du double de la moyenne. La
cellule suivante examine ce défaut.

📘 **Explication — examen des retards de paiement impossibles**

L'abonnement est facturé chaque mois : sur la fenêtre de 12 mois, un compte ne peut pas avoir
plus de retards que de mois facturés, c'est-à-dire min(ancienneté, 12). La cellule croise
l'ancienneté (1 à 6 mois) et le nombre de retards, puis compare le taux de churn des comptes
récents selon qu'ils violent la règle ou non.

In [ ]:
retards_impossibles = masque_retards_impossibles(silver)
recents = silver["anciennete_mois"] <= 6
display(pd.crosstab(silver.loc[recents, "anciennete_mois"], silver.loc[recents, "retards_paiement_12m"],
                    rownames=["ancienneté (mois)"], colnames=["retards de paiement sur 12 mois"]))

un_mois = (silver["anciennete_mois"] == 1) & silver["retards_paiement_12m"].notna()
print(f"Retards impossibles : {int(retards_impossibles.sum())} comptes, dont "
      f"{int((retards_impossibles & (silver['anciennete_mois'] == 1)).sum())} à 1 mois d'ancienneté "
      f"({retards_impossibles[un_mois].mean():.0%} des comptes d'un mois dont les retards sont connus)")
jeunes = (silver["anciennete_mois"] <= 3) & silver["retards_paiement_12m"].notna()
print(f"Churn des comptes de 3 mois ou moins : {silver.loc[jeunes & retards_impossibles, 'churn'].mean():.0%} "
      f"avec des retards impossibles, {silver.loc[jeunes & ~retards_impossibles, 'churn'].mean():.0%} sans")
ratio_brut = silver["retards_paiement_12m"] / silver["anciennete_mois"].clip(lower=1)
print(f"Ratio retards / mois d'ancienneté : jusqu'à {ratio_brut.max():.0f} retards par mois sans correction")

**Lecture — une donnée incompréhensible, et pourquoi.** Presque toutes les violations viennent
des comptes d'**un mois** : 419 d'entre eux déclarent de 2 à 7 retards de paiement, alors qu'ils
n'ont été facturés qu'une fois. Pour 63 % des comptes d'un mois, la variable est impossible : elle
ne mesure pas ce que dit le dictionnaire (« retards de paiement sur 12 mois ») pour ces comptes.
Les explications possibles (retards d'un contrat précédent du même client, relances comptées
comme des retards, erreur d'extraction) ne peuvent pas être départagées sans le propriétaire de
la donnée. Le ratio `taux_retard_paiement_par_mois` hérite du défaut : il atteint 7 retards par
mois.

**Le piège : le défaut est lié à la cible.** Les comptes récents aux retards impossibles
résilient à 61 %, contre 44 % pour les autres comptes récents. Un modèle entraîné sur ces
valeurs apprend donc qu'« un nombre de retards impossible » annonce le churn. C'est une règle
tirée d'un défaut de saisie : rien ne garantit qu'elle existera dans les données de production,
et elle ne peut pas être expliquée à un CSM (« ce client a 5 retards de paiement en 1 mois »).

**Décision (révision v2.1).** La vraie valeur est inconnue : on ne peut pas la **corriger**.
Plafonner à l'ancienneté inventerait une valeur ; supprimer les 449 comptes retirerait 9 % des
données alors que leurs autres colonnes sont valides, et ces comptes devront quand même être
scorés en production. On **neutralise la valeur** : elle devient manquante, et le pipeline
l'impute par la médiane du train comme n'importe quelle valeur manquante
(`neutraliser_incoherences`, `src/features.py`). La même règle s'applique à l'entraînement, au
scoring d'un export brut (§10.3) et dans l'API. Le coût de ce choix est mesuré en validation
croisée au §8.4.

📘 **Explication — trois autres données à la définition douteuse**

Certaines valeurs respectent toutes les règles, mais leur sens pose question. La cellule en
examine trois :

1. les variables d'usage (connexions, heures) rapportées au nombre d'utilisateurs actifs, par
   taille d'entreprise : des heures « cumulées sur 30 jours » devraient croître avec le nombre
   d'utilisateurs ;
2. le délai moyen de réponse du support pour les comptes **sans aucun ticket** : une moyenne sur
   zéro ticket n'existe pas ;
3. les valeurs de la valeur vie client exactement égales aux bornes du dictionnaire (300 € et
   2 000 000 €).

Elle vérifie aussi un cas qui ressemble à une incohérence : une dernière connexion il y a 30 jours
et aucune connexion sur 30 jours.

In [ ]:
actifs = silver["utilisateurs_actifs"].replace(0, np.nan)
usage_par_taille = pd.DataFrame({
    "utilisateurs actifs (médiane)": silver.groupby("taille_entreprise")["utilisateurs_actifs"].median(),
    "connexions sur 30 j (médiane)": silver.groupby("taille_entreprise")["connexions_30j"].median(),
    "heures sur 30 j (médiane)": silver.groupby("taille_entreprise")["heures_usage_30j"].median(),
    "heures par utilisateur actif (médiane)": (silver["heures_usage_30j"] / actifs).groupby(silver["taille_entreprise"]).median(),
}).loc[["TPE", "PME", "ETI", "GE"]]
display(usage_par_taille.round(2))

delai_sans_ticket = (silver["tickets_support_90j"] == 0) & silver["delai_reponse_support_h"].notna()
print(f"Délai de réponse renseigné sans aucun ticket : {int(delai_sans_ticket.sum())} comptes "
      f"({delai_sans_ticket.mean():.1%})")
for borne in (300, 2_000_000):
    print(f"Valeur vie client égale à {borne:,} € : {int((silver['valeur_vie_client_eur'] == borne).sum())} comptes".replace(",", " "))
fenetre = (silver["derniere_connexion_jours"] == 30) & sans_connexion
print(f"Dernière connexion il y a 30 jours et 0 connexion sur 30 jours : {int(fenetre.sum())} comptes, "
      f"dont {int((fenetre & (silver['utilisateurs_actifs'] == 0)).sum())} sans utilisateur actif ; "
      f"dernière connexion minimale des comptes sans connexion : {int(silver.loc[sans_connexion, 'derniere_connexion_jours'].min())} jours")

**Lecture.**

- **Les connexions et les heures ne croissent pas avec la taille du compte** : environ 15
  connexions médianes sur 30 jours, qu'une entreprise ait 2 ou 187 utilisateurs actifs, soit
  moins de 2 minutes d'usage par utilisateur et par mois dans une grande entreprise. C'est
  incompatible avec des « heures d'usage cumulées » sur tout le compte. Ces variables suivent en
  revanche le taux d'adoption (Spearman 0,84 à 0,88, §6.3) : elles se comportent comme un
  **indice d'intensité d'usage**, pas comme un volume. On les garde, car elles sont cohérentes
  entre elles et avec l'adoption, mais l'explication donnée aux CSM ne doit pas les présenter
  comme un temps total passé sur l'outil.
- **Le délai de réponse du support est renseigné pour 1 177 comptes sans aucun ticket**
  (23,5 %). Il suit le SLA du plan (Spearman 0,82, §6.3) : c'est plus probablement une valeur de
  service par plan qu'une moyenne propre au compte. On le garde, avec cette réserve.
- **La valeur vie client est bornée** : 154 comptes à exactement 300 € et 14 à exactement
  2 000 000 €, les bornes du dictionnaire. Ce sont des valeurs **plafonnées**, dont la vraie
  valeur est inférieure ou supérieure. Cela ne concerne que la régression CLV (§9.7), qui
  apprend à ces bornes une valeur tronquée ; le modèle de churn n'utilise pas la CLV.
- **Fausse alerte** : les 44 comptes « dernière connexion il y a 30 jours, aucune connexion sur
  30 jours » sont à la limite de la fenêtre (aucun compte sans connexion n'a une dernière
  connexion à moins de 30 jours) et n'ont aucun utilisateur actif : des comptes jamais activés.
  C'est cohérent, on les garde.

**Synthèse du tri.**

| Constat | Comptes | Décision | Raison |
|---|---|---|---|
| Valeurs extrêmes de taille et d'usage (queue des distributions, §6.1) | 6 à 16 % selon la variable | **retenues** | cohérentes avec la taille d'entreprise et le prix du plan |
| Plus de 75 jours sans connexion, tous résiliés (§6.4) | 82 | **retenues** | comportement plausible (compte abandonné), cohérent avec l'ancienneté |
| Dernière connexion à 30 jours sans connexion sur 30 jours | 44 | **retenues** | limite de la fenêtre, comptes jamais activés |
| Plus de retards de paiement que de mois facturés | 449 | **valeur écartée** (neutralisée) | impossible, et liée à la cible : le modèle apprendrait un défaut de saisie |
| Usage qui ne croît pas avec le nombre d'utilisateurs | tous | retenues **avec réserve** | indice d'intensité, pas un volume : à confirmer |
| Délai de réponse sans ticket | 1 177 | retenues **avec réserve** | probablement une valeur par plan |
| Valeur vie client aux bornes du dictionnaire | 168 | retenues **avec réserve** | valeurs plafonnées, cible secondaire seulement |

Les trois réserves sont des **questions au propriétaire de la donnée**, reportées en §14.

### 📓 Journal de bord — §6 Analyse exploratoire [C3]

- **Portrait de la donnée** : 5 000 comptes, aucune variable quasi constante, aucune valeur hors
  plage du dictionnaire, aucune modalité rare. Six variables de taille et d'usage sont très
  asymétriques (queue de gros comptes, légitime) : la médiane décrit mieux le compte type, et une
  transformation logarithmique est à tester à la prochaine version.
- **Cible déséquilibrée** (28 % de churn) : métriques retenues = PR-AUC (principale) et ROC-AUC,
  jamais l'exactitude seule.
- **Valeurs manquantes** : 6 variables entre 4 et 10 % ; imputation dans le pipeline, pas de
  suppression de lignes.
- **Piège de fuite** : `sante_compte_fin_periode`, AUC univariée de 0,999, calculée en fin de
  période donc indisponible au scoring → exclue des deux modèles.
- **Leurres** : 5 variables sans lien plausible et sans effet mesurable → exclues. Une sixième
  (`sieges_souscrits`) est signalée par le test mais conservée pour sa valeur métier, sous
  réserve de vérification en §8.
- **Redondances** : 5 colonnes du catalogue répètent `plan` → exclues. La matrice de corrélation
  (Spearman) montre trois familles (taille, usage, relation) et six paires redondantes
  (|ρ| ≥ 0,8), gardées dans le modèle v2 : point de vigilance
  pour l'explication, retrait à tester en v3.
- **Signaux métier** : récence de connexion, intégrations, ancienneté, satisfaction, tickets,
  adoption. Tous interprétables par un CSM.
- **Valeurs atypiques et incohérences** (§6.5) : 12 règles de cohérence sur 13 respectées par
  tous les comptes, donc valeurs extrêmes retenues. **449 comptes (9 %) ont plus de retards de
  paiement que de mois facturés** : valeur impossible, liée à la cible (61 % de churn) →
  neutralisée en valeur manquante (révision v2.1). Trois données retenues avec réserve (usage non
  proportionnel à la taille, délai de support sans ticket, CLV plafonnée).
- **Point de vigilance** : `derniere_connexion_jours` sépare parfaitement au-delà de 75 jours ;
  conservée, avec une vérification de sa date de calcul à prévoir en situation réelle.

## 7. Préparation des données — journal de bord [C3]

Chaque transformation est une fonction de `src/` couverte par des tests unitaires : la même
logique sert à l'entraînement et au scoring en production, sans copier-coller.

📘 **Explication — 7.1 à 7.3 : doublons, dates, nombres écrits en texte**

La cellule démontre chaque correction sur des exemples concrets :

- **Doublons** : `drop_strict_duplicates` supprime les lignes identiques sur toutes les colonnes
  métier. Les colonnes de traçabilité sont exclues de la comparaison, sinon deux copies
  ingérées à des instants différents ne seraient jamais reconnues comme doublons.
- **Dates** : `parse_date_multi` essaie successivement les trois formats rencontrés
  (`AAAA-MM-JJ`, `JJ/MM/AAAA`, `JJ Mon AAAA`). On vérifie qu'aucune date ne reste illisible. La
  date n'entre pas dans le modèle (l'ancienneté en mois porte déjà cette information), mais
  elle est conservée pour la traçabilité.
- **Nombres en texte** : `parse_numeric_fr` retire les symboles (`%`, `€`, `h`), remplace la
  virgule décimale par un point, puis convertit. La démonstration compare avec la conversion
  naïve de pandas, qui transforme silencieusement ces valeurs en manquantes : un bug classique
  qui gonflerait artificiellement le taux de valeurs manquantes.

In [ ]:
from src.silver import drop_strict_duplicates, parse_date_multi, parse_numeric_fr

_, n_supprimes = drop_strict_duplicates(brut)
print(f"7.1 Doublons stricts supprimés : {n_supprimes}")

dates = parse_date_multi(brut["date_souscription"])
exemples_dates = brut["date_souscription"].drop_duplicates().head(3)
print(f"7.2 Dates illisibles après lecture multi-formats : {int(dates.isna().sum())}")
print("    exemples :", dict(zip(exemples_dates, parse_date_multi(exemples_dates).dt.date.astype(str))))

exemples = pd.Series(["12,5 %", "990,0 €", "2 h", "37.5", ""])
demo = pd.DataFrame({
    "valeur brute": exemples,
    "pd.to_numeric naïf": pd.to_numeric(exemples, errors="coerce"),
    "parse_numeric_fr": parse_numeric_fr(exemples),
})
print("7.3 Conversion des nombres écrits en texte :")
display(demo)

📘 **Explication — 7.4 et 7.5 : catégories et jointure avec le catalogue**

- **Normalisation des catégories** : `plan` et `taille_entreprise` sont mis dans une casse unique
  et débarrassés des espaces. Pour `secteur`, dont les accents ont été corrompus à l'encodage
  (`SantÃ©`), réparer l'encodage n'est pas fiable : on reconnaît chaque secteur par un morceau
  de mot qui ne contient pas d'accent (`SANT` → Santé, `DUCATION` → Éducation).
- **Jointure** : une fois `plan` normalisé des deux côtés, chaque compte doit retrouver son plan
  dans le catalogue. On vérifie qu'il n'y a **aucun orphelin** (compte sans plan correspondant).

La cellule affiche les valeurs de `secteur` avant et après normalisation.

In [ ]:
avant = brut["secteur"].value_counts().head(10)
apres = silver["secteur"].value_counts()
print("7.4 secteur, 10 valeurs brutes les plus fréquentes :", avant.index.tolist())
print("    après normalisation :", apres.to_dict())
print(f"7.5 Comptes sans plan correspondant dans le catalogue : {rapport_silver['n_orphelins_catalogue']}")

### 7.6 Stratégie de traitement des valeurs manquantes

Une décision par colonne, et non une imputation uniforme :

| Colonne(s) | Manquants | Traitement | Justification |
|---|---|---|---|
| `revenu_mensuel_recurrent_eur` | recalculé | sièges × prix catalogue du plan | Valeur reconstituable exactement par une règle métier |
| `taux_adoption_pct` | 5 % | **recalculé** : 100 × actifs / sièges | Définition même de la variable (écart moyen de 0,02 point sur les valeurs connues) |
| `secteur`, `pays` | quelques % | modalité explicite « Inconnu » | Ne pas inventer une catégorie ; l'absence reste visible |
| `heures_usage_30j`, `nb_integrations`, `delai_reponse_support_h`, `csat`, `retards_paiement_12m` | 4 à 10 % | **médiane, calculée dans le pipeline sur le seul jeu d'entraînement** | Robuste aux valeurs extrêmes ; calculée après le découpage pour ne pas utiliser d'information du test |
| `commentaire_csm` | 55 % vides | non traité | Exclu du modèle (D7) |
| `retards_paiement_12m` **impossible** (plus de retards que de mois facturés) | 9 % | **neutralisé en valeur manquante**, puis médiane comme ci-dessus (v2.1) | Valeur impossible et non corrigeable (§6.5) |

**Changement par rapport à la v1** : la v1 calculait les médianes sur les 5 000 comptes avant le
découpage train/test. La fuite d'information était minime, mais réelle ; elle est corrigée ici.
Des indicateurs de valeur manquante (« csat absent ») ont aussi été testés en §8 : ils
n'apportent rien et ne sont pas retenus.

📘 **Explication — 7.7 et 7.8 : construction des features du Gold v2**

`construire_features_v2` (module `src/features.py`) produit les 20 variables du modèle :

1. elle **recalcule** `taux_adoption_pct` quand il manque ;
2. elle **neutralise** les retards de paiement impossibles (§6.5) : ils deviennent des valeurs
   manquantes, avant le calcul du ratio de retards ;
3. elle ajoute deux **ratios métier** déjà présents en v1 :
   - `taux_utilisation_fonctionnalites` = fonctionnalités utilisées / fonctionnalités offertes ;
   - `taux_retard_paiement_par_mois` = retards de paiement / ancienneté, pour comparer des
     comptes d'âges différents ;
4. elle ne garde que les colonnes autorisées, dans un ordre fixe.

La cellule affiche le tableau récapitulatif des colonnes **exclues**, chacune avec sa raison :
l'énoncé demande de justifier chaque exclusion.

D'autres ratios ont été essayés (connexions par utilisateur actif, sièges inutilisés, remise
par rapport au prix catalogue…) : le §8 montre qu'ils n'améliorent pas le modèle ; ils ne sont
donc pas retenus. Essayer une idée n'oblige pas à la garder.

In [ ]:
from src.features import (CATEGORIELLES_V2, FEATURES_V2, LEURRES, NUMERIQUES_V2,
                          REDONDANTES_CATALOGUE, construire_features_v2)

features = construire_features_v2(silver)
print(f"{len(FEATURES_V2)} features retenues : {len(CATEGORIELLES_V2)} catégorielles, {len(NUMERIQUES_V2)} numériques")
print("taux_adoption_pct manquants : avant", int(silver["taux_adoption_pct"].isna().sum()),
      "-> après recalcul", int(features["taux_adoption_pct"].isna().sum()))
print("retards_paiement_12m manquants : avant", int(silver["retards_paiement_12m"].isna().sum()),
      "-> après neutralisation des valeurs impossibles", int(features["retards_paiement_12m"].isna().sum()),
      f"| ratio de retards par mois au maximum : {features['taux_retard_paiement_par_mois'].max():.2f}")

exclusions = pd.DataFrame(
    [("client_id", "identifiant", "aucune valeur prédictive légitime")]
    + [("date_souscription", "redondante", "information portée par anciennete_mois")]
    + [("commentaire_csm", "conformité (D7)", "texte libre, risque de données personnelles")]
    + [("sante_compte_fin_periode", "fuite de données", "calculée en fin de période : indisponible au scoring (AUC 0,999)")]
    + [("valeur_vie_client_eur", "cible secondaire", "interdite comme variable explicative du churn (énoncé §3.2)")]
    + [(c, "leurre", "aucun effet mesurable sur le churn (§6)") for c in LEURRES]
    + [(c, "redondante (catalogue)", "une seule valeur par plan : répète l'information de plan") for c in REDONDANTES_CATALOGUE],
    columns=["colonne exclue", "catégorie", "raison"],
)
display(exclusions)

📘 **Explication — 7.9 : découpage entraînement / test**

`assign_split` (module `src/gold.py`) découpe les comptes en **80 % d'entraînement** et **20 % de
test**, de façon :

- **stratifiée** : le taux de churn est le même (28 %) dans les deux parties ; sans cela, le
  hasard pourrait mettre plus de comptes qui partent d'un côté et fausser l'évaluation ;
- **reproductible** : graine fixe (42), donc le même découpage à chaque exécution.

Le découpage est **matérialisé par une colonne `split`** dans la table Gold, au lieu d'être
recalculé à chaque fois : n'importe qui peut vérifier sur quels comptes le modèle a été testé.
On vérifie enfin que ce découpage est **identique à celui de la v1** : les résultats v1 et v2
sont donc comparables compte par compte.

In [ ]:
from src.gold import assign_split

silver["split"] = assign_split(silver, test_size=0.2, random_state=RANDOM_STATE)
recap = silver.groupby("split")["churn"].agg(comptes="size", taux_churn="mean")
display(recap.style.format({"taux_churn": "{:.1%}"}))

gold_v1 = pd.read_parquet(GOLD_DIR / "clients_churn_gold.parquet")[["client_id", "split"]]
identique = silver[["client_id", "split"]].merge(gold_v1, on="client_id", suffixes=("_v2", "_v1"))
print("Découpage identique à celui de la v1 :", bool((identique["split_v2"] == identique["split_v1"]).all()))

📘 **Explication — 7.10 : persistance du Gold v2 et de son manifeste**

On enregistre la table Gold v2 au format **Parquet** (format en colonnes, typé et compressé),
avec :

- les 20 features, la colonne `split` et les deux cibles ;
- `sante_compte_fin_periode`, conservée **pour la traçabilité uniquement** (elle n'est jamais
  donnée aux modèles).

Un **manifeste JSON** accompagne le fichier : version, date, hash SHA-256 des sources et du
fichier produit, commit Git du code, liste des features et des exclusions. Une **fiche
d'identité** en Markdown explique les changements par rapport à la v1. La v1 n'est pas écrasée :
les deux versions coexistent, et MLflow permet de comparer les modèles entraînés sur chacune.

In [ ]:
from src.versioning import write_manifest

gold_v2 = pd.concat(
    [silver[["client_id", "split"]], features,
     silver[["sante_compte_fin_periode", "valeur_vie_client_eur", "churn"]]], axis=1)
chemin_gold_v2 = GOLD_DIR / "clients_churn_gold_v2.parquet"
gold_v2.to_parquet(chemin_gold_v2, index=False)
sha_gold_v2 = sha256_of(chemin_gold_v2)

manifeste_gold_v2 = {
    "couche": "gold", "version": "v2.1",
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "produit_par": "notebooks/notebook_certifiant_churn_saas.ipynb", "commit_git": commit_git(),
    "sources": {p.name: sha256_of(p) for p in FICHIERS_SOURCES.values() if p.name != "churn_saas_echantillon.csv"},
    "sha256_gold": sha_gold_v2, "lignes": int(len(gold_v2)),
    "features_modele": FEATURES_V2,
    "colonnes_exclues": exclusions.set_index("colonne exclue")["catégorie"].to_dict(),
    "imputation": "médiane dans le pipeline sklearn, ajustée sur le train ; taux_adoption_pct recalculé ; "
                  "retards de paiement impossibles neutralisés en NaN (v2.1)",
    "split": {"random_state": RANDOM_STATE, "test_size": 0.2, "stratifie_sur": "churn",
              "taux_churn_train": round(float(recap.loc["train", "taux_churn"]), 4),
              "taux_churn_test": round(float(recap.loc["test", "taux_churn"]), 4),
              "identique_v1": True},
}
write_manifest(GOLD_DIR / "gold_v2_manifest.json", manifeste_gold_v2)

fiche = f"""# Fiche d'identité — Gold dataset v2

Produit par `notebooks/notebook_certifiant_churn_saas.ipynb` (commit `{commit_git()}`).
Fichier : `data/gold/clients_churn_gold_v2.parquet` — SHA-256 `{sha_gold_v2}`.

## Changements par rapport à la v1
- 5 leurres exclus ({", ".join(LEURRES)}) : effet nul sur le churn, exclusion demandée par l'énoncé.
- 5 colonnes du catalogue exclues ({", ".join(REDONDANTES_CATALOGUE)}) : une valeur par plan.
- Imputation par la médiane déplacée dans le pipeline (ajustée sur le train) : plus de fuite train/test.
- `taux_adoption_pct` manquant recalculé (100 × actifs / sièges) au lieu d'être imputé.
- Révision v2.1 : {int(masque_retards_impossibles(silver).sum())} retards de paiement impossibles (plus de retards que de mois facturés) neutralisés en valeur manquante (§6.5).

## Inchangé
- Même découpage train/test que la v1 (graine 42, stratifié) : comparaison compte par compte possible.
- `sante_compte_fin_periode` conservée pour traçabilité, jamais utilisée par les modèles.

## Composition
{len(gold_v2)} comptes, {len(FEATURES_V2)} features, taux de churn {taux_churn:.1%}.
"""
(GOLD_DIR / "gold_fiche_identite_v2.md").write_text(fiche, encoding="utf-8")
print(f"Gold v2 : {gold_v2.shape[0]} lignes x {gold_v2.shape[1]} colonnes -> {chemin_gold_v2.relative_to(ROOT)}")
print(f"SHA-256 : {sha_gold_v2}")

### 7.11 Cycle de vie du jeu de données

De l'export brut aux scores, chaque état de la donnée a un format, un lieu de stockage, une règle
de conservation et des destinataires.

| Étape | Format et stockage | Conservation aujourd'hui | Accès |
|---|---|---|---|
| Export brut (CSV fournis) | Fichiers versionnés par DVC, stockés sur DagsHub (stockage objet compatible S3) | Toutes les versions | Équipe data |
| Portique RGPD | Manifeste JSON ; table de pseudonymisation de `client_id` gardée hors Git et hors DVC | Clé conservée en local uniquement | Équipe data, DPO |
| Bronze, Silver, Gold v2 | Parquet (format en colonnes, typé, compressé), versionné par DVC ; manifestes avec empreintes SHA-256 dans Git | Toutes les versions (chaîne d'empreintes vérifiable) | Équipe data |
| Modèles et règle de décision | joblib et manifestes (`data/model_v2/`), runs MLflow | Version en service et versions précédentes (retour arrière possible) | Équipe data |
| Scores du cycle | Export de 5 colonnes vers le CRM (§10) | Table écrasée à chaque cycle (registre, §4.1) | Équipes CS |

**Choix du stockage.** Le Parquet en stockage objet suffit à cette volumétrie (5 000 comptes par
mois, traitement par lots) : lecture rapide par colonnes, types conservés, versions gérées par
DVC. Une base relationnelle n'apporterait rien ici ; elle deviendrait utile pour un scoring en
continu (§11.4, scénario B).

**Accessibilité vérifiée.** Données et modèles se récupèrent par `dvc pull` depuis DagsHub ; la
CI le fait à chaque push (job `docker`).

**Usages futurs prévus.** Réentraînement sur les données à jour (§13.2), mesure de la dérive avec
le jeu d'entraînement comme référence (§13.1), audit d'une décision passée (empreintes et commit
dans chaque manifeste).

**À soumettre aux parties prenantes.** Les durées de conservation (aujourd'hui, toutes les
versions sont gardées, ce que le principe de limitation de la conservation du RGPD, article
5.1.e, ne permet pas sans justification) et les droits d'accès à la clé de pseudonymisation sont
à fixer avec le DPO ; le format et la fréquence de l'export, avec l'équipe qui le produit. Ce
cycle de vie n'a pas encore été présenté : c'est une condition de mise en service.

### 📓 Journal de bord — §7 Préparation [C3]

- **Nettoyage** : 35 doublons supprimés, 0 date illisible, nombres en texte convertis (le bug de
  la conversion naïve est démontré), secteurs normalisés, 0 orphelin après jointure.
- **Valeurs manquantes** : une stratégie par colonne ; MRR et taux d'adoption **recalculés** par
  règle métier ; médianes calculées **dans le pipeline, sur le seul train** (correction d'une
  fuite minime de la v1).
- **Incohérence neutralisée (v2.1)** : 449 retards de paiement impossibles deviennent des valeurs
  manquantes, avant le calcul du ratio de retards ; même règle à l'entraînement, au scoring
  et dans l'API.
- **Features** : 20 variables (3 catégorielles, 17 numériques, dont 2 ratios métier).
- **Exclusions justifiées** : identifiant, date, texte libre (D7), fuite, cible secondaire,
  5 leurres, 5 redondances.
- **Découpage** : 80/20 stratifié, graine 42, identique à la v1, matérialisé dans la table.
- **Versioning** : Gold v2 enregistré avec son manifeste (hashes, commit) et sa fiche
  d'identité, sans écraser la v1.
- **Cycle de vie** : format, stockage, conservation et accès documentés de l'export brut aux
  scores (§7.11) ; durées de conservation à fixer avec le DPO (aujourd'hui, toutes les versions
  sont gardées).

## 8. Choix du modèle et démarche scientifique — journal de bord [C4]

### 8.1 Protocole

- **Données** : uniquement le jeu d'entraînement (4 000 comptes). Le jeu de test reste fermé
  jusqu'au §9.
- **Validation croisée stratifiée à 5 plis** : l'entraînement est découpé en 5 parts ; chaque
  modèle est entraîné 5 fois sur 4 parts et évalué sur la cinquième. On obtient 5 scores, donc
  une moyenne **et une dispersion** : on sait si un écart entre deux modèles est réel ou dû au
  hasard du découpage.
- **Métrique principale** : PR-AUC (classes déséquilibrées) ; ROC-AUC en complément (exigée par
  l'énoncé).
- **Règle de sélection fixée à l'avance** : le meilleur score moyen, sauf si un modèle plus
  simple et plus explicable fait jeu égal (écart inférieur à un écart-type).
- **Critère d'acceptation D8** : battre la baseline métier d'au moins +0,15 de PR-AUC.

📘 **Explication — préparer les données et le pipeline de modélisation**

1. On sépare la table Gold v2 en **entraînement** et **test** grâce à la colonne `split`.
2. On définit un **pipeline** scikit-learn : une chaîne d'étapes appliquées toujours dans le
   même ordre, à l'entraînement comme à la prédiction :
   - variables catégorielles → **encodage one-hot** (une colonne 0/1 par modalité ; une
     modalité inconnue à la prédiction est ignorée au lieu de provoquer une erreur) ;
   - variables numériques → **imputation par la médiane** puis **standardisation** (centrer et
     réduire : chaque variable a une moyenne de 0 et un écart-type de 1, ce qui rend les
     coefficients d'un modèle linéaire comparables entre eux) ;
   - puis le **modèle**.
3. Point essentiel : dans la validation croisée, le pipeline complet est réentraîné à chaque
   pli. La médiane d'imputation et les paramètres de standardisation sont donc calculés **sur
   les seules données d'entraînement du pli** : aucune information du pli de validation ne
   fuit dans la préparation.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = gold_v2[gold_v2["split"] == "train"].reset_index(drop=True)
test = gold_v2[gold_v2["split"] == "test"].reset_index(drop=True)
X_train, y_train = train[FEATURES_V2], train["churn"]
X_test, y_test = test[FEATURES_V2], test["churn"]
print(f"Entraînement : {X_train.shape}, churn {y_train.mean():.1%} | Test : {X_test.shape}, churn {y_test.mean():.1%}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc"}


def preprocesseur(categorielles, numeriques, standardiser=True, indicateurs=False):
    """One-hot des catégorielles ; imputation médiane (+ standardisation) des numériques."""
    etapes_num = [("imputation", SimpleImputer(strategy="median", add_indicator=indicateurs))]
    if standardiser:
        etapes_num.append(("standardisation", StandardScaler()))
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorielles),
        ("num", Pipeline(etapes_num), numeriques),
    ])


def pipeline_modele(estimateur, categorielles=CATEGORIELLES_V2, numeriques=NUMERIQUES_V2,
                    standardiser=True, indicateurs=False):
    return Pipeline([
        ("preparation", preprocesseur(categorielles, numeriques, standardiser, indicateurs)),
        ("modele", estimateur),
    ])


def evaluer_cv(nom, pipeline, X=X_train):
    """Validation croisée 5 plis : moyenne et écart-type de la PR-AUC et de la ROC-AUC."""
    res = cross_validate(pipeline, X, y_train, cv=cv, scoring=SCORING)
    return {"modèle": nom, "pr_auc": res["test_pr_auc"].mean(), "pr_auc_std": res["test_pr_auc"].std(),
            "roc_auc": res["test_roc_auc"].mean(), "plis": res["test_pr_auc"]}

### 8.2 Les baselines : ce que l'on sait faire sans modèle

Deux références, à battre nettement :

1. **Baseline naïve** (`DummyClassifier`) : prédit pour tous les comptes la même probabilité,
   le taux de churn moyen. Sa PR-AUC vaut le taux de base (≈ 0,28) : c'est le plancher absolu.
2. **Baseline métier (D8)** : une règle qu'un CSM pourrait appliquer à la main. D8 la décrit
   comme une « règle sur ancienneté et santé du compte ». **Attention** : le score de santé
   disponible (`sante_compte_fin_periode`) est la variable de fuite ; l'utiliser donnerait une
   baseline impossible à battre et, surtout, inapplicable au moment du scoring. J'interprète
   donc la « santé » par des signaux d'engagement **observables avant l'échéance** : nombre
   d'intégrations et récence de connexion. La règle, sans aucun apprentissage :

   `score = − ancienneté / 36 − intégrations / 16 + jours_depuis_connexion / 200`

   (chaque terme est ramené à une échelle de 0 à 1 par le maximum du dictionnaire de données).

📘 **Explication — évaluer les baselines**

- Le `DummyClassifier(strategy="prior")` est évalué avec la même validation croisée que les
  modèles.
- La règle métier n'apprend rien, mais on la mesure **sur les mêmes plis de validation** pour
  que la comparaison soit équitable. Les intégrations manquantes sont remplacées par la médiane
  du pli d'entraînement (même principe que dans le pipeline).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score


def score_regle_metier(df, mediane_integrations):
    """Baseline D8 : ancienneté faible, peu d'intégrations, connexion ancienne = risque."""
    return (-df["anciennete_mois"] / 36
            - df["nb_integrations"].fillna(mediane_integrations) / 16
            + df["derniere_connexion_jours"] / 200)


resultats_cv = [evaluer_cv("Baseline naïve (Dummy)", DummyClassifier(strategy="prior"))]
plis_regle, plis_regle_roc = [], []
for idx_app, idx_val in cv.split(X_train, y_train):
    mediane = X_train.iloc[idx_app]["nb_integrations"].median()
    score = score_regle_metier(X_train.iloc[idx_val], mediane)
    plis_regle.append(average_precision_score(y_train.iloc[idx_val], score))
    plis_regle_roc.append(roc_auc_score(y_train.iloc[idx_val], score))
resultats_cv.append({"modèle": "Baseline métier D8 (règle)", "pr_auc": np.mean(plis_regle),
                     "pr_auc_std": np.std(plis_regle), "roc_auc": np.mean(plis_regle_roc),
                     "plis": np.array(plis_regle)})
display(pd.DataFrame(resultats_cv).drop(columns="plis").round(3))

### 8.3 Trois familles de modèles

| Famille | Modèle | Pourquoi l'essayer |
|---|---|---|
| Linéaire | **Régression logistique** | Simple, rapide, probabilités bien calibrées, coefficients directement explicables aux CSM |
| Ensemble d'arbres (bagging) | **Forêt aléatoire** | Capte les interactions et les effets non linéaires sans hypothèse de forme |
| Ensemble d'arbres (boosting) | **Gradient boosting** (`HistGradientBoosting`) | Souvent le plus performant sur des données tabulaires ; gère nativement les manquants |

📘 **Explication — comparer les trois familles en validation croisée**

Chaque modèle est placé dans le même pipeline (seule la standardisation est retirée pour les
modèles à base d'arbres, qui n'en ont pas besoin : un arbre compare des seuils, l'échelle des
variables ne change rien). Les hyperparamètres sont des valeurs raisonnables par défaut ; le
réglage fin est fait en §9 sur le modèle retenu.

Le graphique en boîtes montre les 5 scores de chaque modèle : une boîte étroite signifie un
modèle stable d'un découpage à l'autre. La ligne pointillée marque le critère D8 (baseline
métier + 0,15).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

candidats = {
    "Régression logistique": pipeline_modele(LogisticRegression(max_iter=3000)),
    "Forêt aléatoire": pipeline_modele(
        RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
        standardiser=False),
    "Gradient boosting": pipeline_modele(
        HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=RANDOM_STATE),
        standardiser=False),
}
for nom, pipe in candidats.items():
    resultats_cv.append(evaluer_cv(nom, pipe))

comparaison = pd.DataFrame(resultats_cv)
seuil_d8_cv = comparaison.set_index("modèle").loc["Baseline métier D8 (règle)", "pr_auc"] + GAIN_MIN_D8
display(comparaison.drop(columns="plis").round(3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(comparaison["plis"], tick_labels=comparaison["modèle"], showmeans=True)
ax.axhline(seuil_d8_cv, linestyle="--", color=COULEURS["churn"], label=f"critère D8 : baseline métier + {GAIN_MIN_D8}")
ax.set_ylabel("PR-AUC (5 plis)")
ax.set_title("Comparaison des modèles en validation croisée (jeu d'entraînement)")
ax.tick_params(axis="x", rotation=15)
ax.legend(loc="lower right")
sauver_figure(fig, "07_comparaison_modeles_cv")
plt.show()

📘 **Explication — pourquoi aucun `class_weight` n'est appliqué**

Le churn est déséquilibré (28 % / 72 %, ratio ≈2,6:1) mais ce n'est pas un déséquilibre sévère (à comparer à un cas de fraude à <5 %). Vérifié empiriquement sur la régression logistique, en validation croisée sur le seul train (cellule suivante) : `class_weight="balanced"` ne change ni la ROC-AUC (0,886) ni la PR-AUC (0,783) par rapport à `class_weight=None` — seul le point de fonctionnement à un seuil fixe se déplace (rappel 0,605 → 0,794 à 0,5, au prix de la précision 0,755 → 0,604).

Le seuil D9 (rappel ≥ 80 %), obtenu par recalibrage du seuil avec `class_weight=None`, atteint déjà un point quasi identique (précision 0,600 pour un rappel de 0,800) — rééquilibrer les classes et ajuster le seuil sont ici deux leviers redondants, pas cumulatifs. On garde `class_weight=None` et on pilote l'opérationnalisation uniquement par le seuil (§9, D9) : plus simple à expliquer et à faire varier sans réentraîner.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_predict
from src.scoring import calibrer_seuil_d9

lignes_poids = []
for poids in (None, "balanced"):
    pipe_poids = pipeline_modele(LogisticRegression(max_iter=3000, class_weight=poids))
    res = evaluer_cv(f"class_weight={poids}", pipe_poids)
    hors_pli = cross_val_predict(pipe_poids, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    seuil = calibrer_seuil_d9(y_train, hors_pli, RAPPEL_CIBLE_D9)
    lignes_poids.append({
        "modèle": res["modèle"], "pr_auc": res["pr_auc"], "roc_auc": res["roc_auc"],
        "rappel à 0,5": recall_score(y_train, hors_pli >= 0.5),
        "précision à 0,5": precision_score(y_train, hors_pli >= 0.5),
        "seuil D9 (hors pli)": seuil,
        "rappel au seuil D9": recall_score(y_train, hors_pli >= seuil),
        "précision au seuil D9": precision_score(y_train, hors_pli >= seuil),
    })
display(pd.DataFrame(lignes_poids).set_index("modèle").round(3))

### 8.4 Le Gold v2 est-il justifié ? Étude d'ablation

Une **étude d'ablation** retire ou ajoute un groupe de variables et mesure l'effet sur la
performance. Elle répond à trois questions : retirer les leurres et les redondances coûte-t-il
quelque chose ? Les idées de variables supplémentaires (indicateurs de manquants, nouveaux
ratios) apportent-elles quelque chose ? Garder `sieges_souscrits` est-il neutre ?

📘 **Explication — l'ablation**

On compare huit jeux de variables avec **le même modèle** (régression logistique) et **la même
validation croisée** : seule la liste de variables change, donc toute différence vient d'elle.

- *v1 (30 variables)* reconstruit le jeu de la version précédente : leurres et colonnes du
  catalogue compris (`support_dedie` est alors une catégorielle Oui/Non).
- Les cinq ratios candidats sont calculés à la volée sur le jeu d'entraînement.
- *v2.0* reprend les 20 variables **sans** neutraliser les retards de paiement impossibles
  (§6.5) : l'écart avec la v2.1 est le prix payé pour ne plus apprendre sur ces valeurs.

Lecture attendue : si deux jeux ont des scores dont l'écart est bien inférieur à l'écart-type,
ils sont équivalents ; on préfère alors **le plus simple**.

In [ ]:
v1_features = json.loads((GOLD_DIR / "gold_manifest.json").read_text(encoding="utf-8"))["features_modele_principal"]
v1_categorielles = [c for c in v1_features if c in
                    ["secteur", "pays", "taille_entreprise", "plan", "jour_souscription",
                     "couleur_theme_interface", "code_datacenter", "groupe_experimentation", "support_dedie"]]
silver_train = silver[silver["split"] == "train"].reset_index(drop=True)
assert (silver_train["client_id"] == train["client_id"]).all()   # mêmes comptes, même ordre

# Ratios candidats (non retenus si sans gain)
actifs = silver_train["utilisateurs_actifs"].clip(lower=1)
ratios = pd.DataFrame({
    "sieges_inutilises_pct": 1 - silver_train["utilisateurs_actifs"] / silver_train["sieges_souscrits"].clip(lower=1),
    "connexions_par_actif": silver_train["connexions_30j"] / actifs,
    "heures_par_actif": silver_train["heures_usage_30j"] / actifs,
    "tickets_par_actif": silver_train["tickets_support_90j"] / actifs,
    "remise_vs_catalogue": silver_train["revenu_mensuel_recurrent_eur"]
                           / (silver_train["sieges_souscrits"] * silver_train["prix_mensuel_par_siege_eur"]),
})
X_train_etendu = pd.concat([silver_train[[c for c in v1_features if c not in FEATURES_V2]],  # colonnes v1 seulement
                            X_train, ratios], axis=1)
X_train_etendu["retards_manquants"] = X_train["retards_paiement_12m"].isna().astype(int)   # vide = inconnu ou impossible
sans_leurres_v1 = [c for c in v1_features if c not in LEURRES]
sans_sieges = [c for c in NUMERIQUES_V2 if c != "sieges_souscrits"]
lr = LogisticRegression(max_iter=3000)

variantes = [
    ("v1 : 30 variables (leurres + catalogue)",
     pipeline_modele(lr, v1_categorielles, [c for c in v1_features if c not in v1_categorielles]), v1_features),
    ("v1 sans les 5 leurres (25)",
     pipeline_modele(lr, [c for c in v1_categorielles if c not in LEURRES],
                     [c for c in sans_leurres_v1 if c not in v1_categorielles]), sans_leurres_v1),
    ("v2.1 : 20 variables (retenue)", pipeline_modele(lr), FEATURES_V2),
    ("v2 + indicateurs de manquants", pipeline_modele(lr, indicateurs=True), FEATURES_V2),
    ("v2 + indicateur « retards manquants » seul",
     pipeline_modele(lr, CATEGORIELLES_V2, NUMERIQUES_V2 + ["retards_manquants"]), FEATURES_V2 + ["retards_manquants"]),
    ("v2 + 5 ratios candidats", pipeline_modele(lr, CATEGORIELLES_V2, NUMERIQUES_V2 + list(ratios.columns)),
     FEATURES_V2 + list(ratios.columns)),
    ("v2 sans sieges_souscrits", pipeline_modele(lr, CATEGORIELLES_V2, sans_sieges), CATEGORIELLES_V2 + sans_sieges),
]
retards_bruts = silver_train["retards_paiement_12m"]
X_train_v20 = X_train.assign(retards_paiement_12m=retards_bruts.to_numpy(),
                             taux_retard_paiement_par_mois=(retards_bruts / silver_train["anciennete_mois"].clip(lower=1)).to_numpy())
ablation = pd.DataFrame([{**evaluer_cv(nom, pipe, X_train_etendu[cols]), "nb_variables": len(cols)}
                         for nom, pipe, cols in variantes]
                        + [{**evaluer_cv("v2.0 : retards impossibles conservés", pipeline_modele(lr), X_train_v20),
                            "nb_variables": len(FEATURES_V2)}]).drop(columns="plis")
display(ablation[["modèle", "nb_variables", "pr_auc", "pr_auc_std", "roc_auc"]].round(4))

📘 **Explication — appliquer la règle de sélection**

La règle, fixée **avant** de regarder les résultats (§8.1) : on prend le modèle au meilleur
score moyen ; tout modèle dont l'écart avec le meilleur est inférieur à un écart-type est
considéré à égalité, et on retient alors le plus simple et le plus explicable (ordre de
préférence : régression logistique, puis gradient boosting, puis forêt aléatoire).

La cellule applique cette règle automatiquement, puis vérifie le critère D8.

In [ ]:
modeles_ml = comparaison[comparaison["modèle"].isin(candidats)].set_index("modèle")
meilleur = modeles_ml["pr_auc"].idxmax()
a_egalite = modeles_ml.index[modeles_ml["pr_auc"] >= modeles_ml.loc[meilleur, "pr_auc"] - modeles_ml.loc[meilleur, "pr_auc_std"]]
ordre_simplicite = ["Régression logistique", "Gradient boosting", "Forêt aléatoire"]
MODELE_RETENU = next(m for m in ordre_simplicite if m in a_egalite)

print(f"Meilleur score moyen : {meilleur} ({modeles_ml.loc[meilleur, 'pr_auc']:.3f} ± {modeles_ml.loc[meilleur, 'pr_auc_std']:.3f})")
print(f"Modèles à égalité (écart < 1 écart-type) : {list(a_egalite)}")
print(f"Modèle retenu : {MODELE_RETENU}")
print(f"Critère D8 en CV : {modeles_ml.loc[MODELE_RETENU, 'pr_auc']:.3f} >= {seuil_d8_cv:.3f} ? "
      f"{modeles_ml.loc[MODELE_RETENU, 'pr_auc'] >= seuil_d8_cv}")

**Lecture.**

- Les trois familles dépassent largement la baseline naïve et la baseline métier.
- La **régression logistique** obtient le meilleur score moyen, devant le gradient boosting et
  la forêt aléatoire. Les modèles plus complexes n'apportent rien : la relation entre les
  signaux d'engagement et le churn est essentiellement **linéaire** (au sens du logit). C'est
  une bonne nouvelle : le modèle le plus simple est aussi le plus performant et le plus
  explicable.
- **Ablation** : retirer les 5 leurres et les 5 colonnes redondantes ne coûte rien (30 → 20
  variables pour un score identique). Les ratios candidats n'apportent rien : ils ne sont pas
  retenus.
- **Prix de la neutralisation (v2.1)** : conserver les retards impossibles donnerait 0,791 de
  PR-AUC contre 0,783. L'écart (0,008) est inférieur à un écart-type : même au regard de la
  règle de sélection, la v2.1 ne perd rien de significatif, et elle n'apprend plus sur des
  valeurs impossibles (§6.5).
- **Indicateurs de manquants** : l'indicateur « retards manquants » seul fait autant (0,786) que
  tous les indicateurs réunis (0,785) : le petit gain vient entièrement de lui. Or, depuis la
  v2.1, une case vide dans les retards correspond surtout à une valeur impossible (449 des 699
  valeurs manquantes, §7.8) : cet indicateur **réintroduirait le défaut neutralisé** par une autre
  porte. Il n'est pas retenu, d'autant que l'écart (0,003) est très inférieur à un écart-type. Retirer `sieges_souscrits` est neutre pour le
  churn : la garder ne dégrade rien et sert la régression CLV.

### 8.5 Solutions sur étagère et nature du résultat attendu

**Fallait-il développer un modèle ?** Trois options existaient :

| Critère | Score intégré à un outil du marché (CRM, plateforme Customer Success) | AutoML d'un fournisseur cloud | Modèle développé (retenu) |
|---|---|---|---|
| Contrôle de la fuite (`sante_compte_fin_periode`, §6.4) | Faible : variables et méthode souvent non visibles | Possible, si l'on prépare soi-même les données | Total |
| Explication de chaque score et contrôles (§9.6) | Limitée au format de l'éditeur | Partielle | Complète, contrôlée par la base de connaissance |
| Système de décision D9, D10, D14 | À reconstruire autour du score | À reconstruire | Intégré (§10) |
| Coût | Licence récurrente | Facturation du calcul | Temps de développement et de maintenance |
| Données et RGPD | Hébergées chez l'éditeur (contrat de sous-traitance, article 28) | Hébergées chez le fournisseur cloud | Maîtrisées |
| Délai de mise en œuvre | Court si l'outil est déjà en place | Moyen | Réalisé |

Le modèle développé est retenu : la fuite et le système de décision exigent un contrôle complet,
et une régression logistique suffit (§8.3). Cette comparaison est qualitative, sans essai des
produits. Si l'éditeur s'équipe d'une plateforme Customer Success, son score devra être comparé à
ce modèle **sur le même jeu de test** (PR-AUC, rappel au seuil D9) avant tout remplacement.

**Nature du résultat.** Le modèle produit un résultat **probabiliste** : une probabilité de
résiliation par compte (fiabilité vérifiée en §9.4), complétée par une estimation de la valeur
vie client en euros (§9.7). Le système de décision la transforme en résultat **déterministe** :
seuil D9, capacité D10, tri par perte attendue D14 et action D11. La même entrée donne toujours la
même liste priorisée.

### 8.6 Éco-conception : ce que coûte chaque famille de modèles

La performance attendue ne se limite pas à la précision : le temps d'inférence, la taille du
modèle et l'énergie consommée comptent aussi. On les mesure pour les trois familles comparées en
§8.3.

📘 **Explication — mesurer l'énergie avec CodeCarbon**

- **CodeCarbon** évalue l'énergie consommée par le processeur et la mémoire pendant un calcul,
  puis la convertit en émissions de CO₂ équivalent avec l'intensité carbone du réseau électrique
  du pays (ici la France ; mode hors ligne : aucune donnée n'est envoyée).
- Selon la machine, la puissance est mesurée par des capteurs ou **estimée** à partir du modèle de
  processeur et de sa charge : les valeurs absolues sont des ordres de grandeur, les **rapports
  entre modèles** sont l'information utile.
- Pour chaque famille : l'énergie de la sélection (la validation croisée à 5 plis de §8.3), le
  temps de prédiction des 1 000 comptes du test (le coût d'un scoring) et la taille du modèle
  sérialisé.
- Les quantités sont petites : on les exprime en milliwattheures (mWh) et en milligrammes de CO₂e.

In [ ]:
import logging
import pickle
import time

from sklearn.base import clone

from codecarbon import OfflineEmissionsTracker

logging.getLogger("codecarbon").setLevel(logging.ERROR)   # après l'import : codecarbon fixe son niveau à l'import


def mesurer_energie(calcul):
    """Exécute `calcul()` sous CodeCarbon (hors ligne, mix électrique français).
    Renvoie (résultat, durée en s, énergie en mWh, émissions en mg de CO2e)."""
    suivi = OfflineEmissionsTracker(country_iso_code="FRA", save_to_file=False,
                                    log_level="error", measure_power_secs=1)
    suivi.start()
    debut = time.perf_counter()
    resultat = calcul()
    duree = time.perf_counter() - debut
    suivi.stop()
    mesure = suivi.final_emissions_data
    return resultat, duree, mesure.energy_consumed * 1e6, mesure.emissions * 1e6


lignes_energie = []
for nom, pipe in candidats.items():
    _, duree_cv, mwh_cv, mg_cv = mesurer_energie(
        lambda: cross_validate(clone(pipe), X_train, y_train, cv=cv, scoring=SCORING))
    modele_mesure = clone(pipe).fit(X_train, y_train)
    _, duree_pred, _, _ = mesurer_energie(lambda: modele_mesure.predict_proba(X_test))
    lignes_energie.append({
        "modèle": nom, "PR-AUC (CV)": modeles_ml.loc[nom, "pr_auc"],
        "sélection : durée (s)": duree_cv, "sélection : énergie (mWh)": mwh_cv,
        "sélection : CO₂e (mg)": mg_cv, "scoring de 1 000 comptes (ms)": duree_pred * 1000,
        "taille du modèle (Ko)": len(pickle.dumps(modele_mesure)) / 1e3,
    })
energie_modeles = pd.DataFrame(lignes_energie).set_index("modèle")
display(energie_modeles.round(3))

reference = energie_modeles.loc[MODELE_RETENU]
for nom, ligne in energie_modeles.drop(index=MODELE_RETENU).iterrows():
    rapport = ligne["sélection : énergie (mWh)"] / reference["sélection : énergie (mWh)"]
    print(f"{nom} : {rapport:.0f} fois l'énergie du modèle retenu ({MODELE_RETENU.lower()}), "
          f"PR-AUC {ligne['PR-AUC (CV)']:.3f} contre {reference['PR-AUC (CV)']:.3f}")

**Lecture.**

- La régression logistique, retenue en §8.3 pour sa performance et son explicabilité, est aussi
  la plus **sobre** : les deux familles à base d'arbres consomment plus de dix fois plus
  d'énergie pour une PR-AUC qui n'est pas meilleure, et pèsent beaucoup plus lourd (la forêt
  aléatoire, près de 30 Mo).
- En valeur absolue, ces consommations sont infimes (quelques milliwattheures). À cette échelle,
  l'éco-conception tient surtout à des **choix de sobriété** : ne pas retenir un modèle plus
  lourd sans gain mesuré, scorer une fois par mois plutôt qu'en continu (§11.4), ne réentraîner
  que sur déclencheur (§13.2).

### 📓 Journal de bord — §8 Choix du modèle [C4]

- **Protocole** : validation croisée stratifiée 5 plis sur le seul train ; test fermé.
- **Baselines** : naïve (PR-AUC ≈ taux de base) et **métier D8** reformulée sans la variable de
  fuite (ancienneté, intégrations, récence de connexion).
- **Trois familles** comparées : linéaire, bagging, boosting. La régression logistique est la
  meilleure ; les modèles plus complexes n'apportent rien sur ces données.
- **Règle de sélection fixée à l'avance** et appliquée par le code (pas de choix a posteriori
  sur le jeu de test, **correction d'un défaut de la v1** qui choisissait le modèle sur le test).
- **Ablation** : le Gold v2 (20 variables) fait aussi bien que la v1 (30 variables) ; il est plus
  simple, conforme à l'énoncé (leurres exclus) et plus lisible.
- **Neutralisation des retards impossibles** : coût de 0,008 de PR-AUC en validation croisée,
  inférieur à un écart-type. L'indicateur « retards manquants » réintroduirait le défaut : écarté.
- **D8 respecté** en validation croisée (vérifié à nouveau sur le test en §9).
- **Solutions sur étagère** : score d'un outil du marché, AutoML et modèle développé comparés
  (contrôle de la fuite, explications, système de décision, coût, RGPD) ; modèle développé
  retenu. Résultat probabiliste, rendu déterministe par le système de décision (§8.5).
- **Éco-conception** : énergie, temps de scoring et taille mesurés par famille avec CodeCarbon ;
  le modèle retenu est aussi le plus sobre (§8.6).

## 9. Entraînement, validation et ajustement — journal de bord [C5]

### 9.1 Réglage de la régularisation

📘 **Explication — régler l'hyperparamètre C**

La régression logistique a un **hyperparamètre de régularisation** `C` : plus `C` est petit,
plus le modèle est contraint à garder des coefficients faibles (il « simplifie » pour éviter de
coller au bruit des données, le **surapprentissage**) ; plus `C` est grand, plus il est libre.

`GridSearchCV` essaie plusieurs valeurs de `C` et mesure chacune en validation croisée, toujours
sur le seul jeu d'entraînement. Il réentraîne ensuite automatiquement le meilleur modèle sur
**tout** le jeu d'entraînement (`refit=True`).

Le graphique montre le score moyen ± un écart-type pour chaque valeur : une courbe plate
signifie que le choix de `C` compte peu, ce qui est un signe de robustesse.

In [ ]:
from sklearn.model_selection import GridSearchCV

grille = GridSearchCV(pipeline_modele(LogisticRegression(max_iter=3000)),
                      param_grid={"modele__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3]},
                      scoring="average_precision", cv=cv, refit=True)
grille.fit(X_train, y_train)
resultats_grille = pd.DataFrame(grille.cv_results_)[["param_modele__C", "mean_test_score", "std_test_score"]]
C_RETENU = grille.best_params_["modele__C"]
modele_churn = grille.best_estimator_

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(resultats_grille["param_modele__C"].astype(float), resultats_grille["mean_test_score"],
            yerr=resultats_grille["std_test_score"], marker="o", capsize=3)
ax.set_xscale("log")
ax.set_xlabel("C (échelle logarithmique)")
ax.set_ylabel("PR-AUC moyenne (CV)")
ax.set_title("Réglage de la régularisation")
sauver_figure(fig, "08_reglage_C")
plt.show()
print(f"C retenu : {C_RETENU} | PR-AUC CV : {grille.best_score_:.4f}")

### 9.2 Choix du seuil de décision (D9) sans toucher au test

Un modèle de classification produit une **probabilité**. Pour décider quels comptes signaler, il
faut un **seuil** : au-dessus, le compte est signalé. Le seuil par défaut de 0,5 n'a aucune
justification métier. D9 impose : le seuil doit garantir un **rappel ≥ 80 %** (au moins 8 comptes
sur 10 qui vont partir doivent être signalés).

**Correction par rapport à la v1** : la v1 calculait ce seuil sur le jeu de test, ce qui garantit
80 % de rappel sur le test *par construction* et rend la mesure optimiste. Ici, le seuil est
calculé sur des **prédictions hors pli** du jeu d'entraînement ; le rappel mesuré ensuite sur le
test est une estimation honnête de ce que l'on obtiendra en production.

📘 **Explication — prédictions hors pli et calibration du seuil**

`cross_val_predict` produit, pour chaque compte du jeu d'entraînement, une probabilité calculée
par un modèle qui **n'a pas vu ce compte** pendant son entraînement (il a été entraîné sur les 4
autres plis). Ces prédictions « hors pli » (*out-of-fold*) se comportent comme des prédictions
sur des données nouvelles.

`calibrer_seuil_d9` (module `src/scoring.py`) cherche ensuite **le seuil le plus élevé** qui
respecte encore un rappel de 80 %. Pourquoi le plus élevé ? Parce qu'un seuil plus haut signale
moins de comptes, donc moins de faux positifs : c'est la meilleure précision possible sous la
contrainte de rappel.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_predict
from src.scoring import calibrer_seuil_d9

proba_oof = cross_val_predict(clone(modele_churn), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
SEUIL_D9 = calibrer_seuil_d9(y_train, proba_oof, rappel_cible=RAPPEL_CIBLE_D9)
signales_oof = proba_oof >= SEUIL_D9
print(f"Seuil D9 (calculé hors pli, sur le train) : {SEUIL_D9:.3f}")
print(f"  rappel hors pli    : {recall_score(y_train, signales_oof):.3f}")
print(f"  précision hors pli : {precision_score(y_train, signales_oof):.3f}")
print(f"  comptes signalés   : {signales_oof.mean():.1%}")

### 9.3 Évaluation finale sur le jeu de test

Le jeu de test n'a servi à aucun choix : ni variables, ni modèle, ni hyperparamètre, ni seuil.
**Une seconde lecture est assumée.** La version v2.0 de ce notebook avait déjà été évaluée sur ce
test. La révision v2.1 (neutralisation des retards de paiement impossibles, §6.5) a été décidée
sur un critère de **qualité des données**, établi sans regarder le test, puis évaluée sur ce même
test. Le risque habituel d'une seconde lecture (retenir la version qui a le meilleur score sur le
test) ne s'applique pas : la v2.1 aurait été retenue même avec un score plus faible, parce qu'un
modèle ne doit pas s'appuyer sur des valeurs impossibles. Les résultats de la v2.0 restent
consultables (runs MLflow v2, release GitHub v2.0).

📘 **Explication — l'évaluation finale**

C'est **la seule étape** où le jeu de test est utilisé (seconde lecture de la v2.1 : voir ci-dessus). Le modèle, son hyperparamètre `C` et son
seuil sont figés ; on mesure :

- la **ROC-AUC** : probabilité qu'un compte qui part reçoive un score plus élevé qu'un compte qui
  reste (0,5 = hasard, 1 = parfait) ;
- la **PR-AUC** : qualité du compromis précision/rappel, à comparer au taux de base (0,28) ;
- au **seuil D9** : le **rappel** (part des comptes qui partent effectivement signalés) et la
  **précision** (part des comptes signalés qui partent effectivement) ;
- le **critère D8** : écart de PR-AUC avec la baseline métier, sur le test cette fois.

In [ ]:
proba_test = modele_churn.predict_proba(X_test)[:, 1]
score_regle_test = score_regle_metier(X_test, X_train["nb_integrations"].median())

metriques_test = {
    "roc_auc": roc_auc_score(y_test, proba_test),
    "pr_auc": average_precision_score(y_test, proba_test),
    "pr_auc_baseline_naive": float(y_test.mean()),
    "pr_auc_baseline_metier_D8": average_precision_score(y_test, score_regle_test),
    "roc_auc_baseline_metier_D8": roc_auc_score(y_test, score_regle_test),
    "rappel_seuil_D9": recall_score(y_test, proba_test >= SEUIL_D9),
    "precision_seuil_D9": precision_score(y_test, proba_test >= SEUIL_D9),
    "part_signales_seuil_D9": float((proba_test >= SEUIL_D9).mean()),
    "rappel_seuil_0_5": recall_score(y_test, proba_test >= 0.5),
    "precision_seuil_0_5": precision_score(y_test, proba_test >= 0.5),
}
metriques_test["gain_pr_auc_vs_D8"] = metriques_test["pr_auc"] - metriques_test["pr_auc_baseline_metier_D8"]
metriques_test["critere_D8_respecte"] = bool(metriques_test["gain_pr_auc_vs_D8"] >= GAIN_MIN_D8)
display(pd.Series(metriques_test, name="jeu de test").to_frame().T.round(3).T)

📘 **Explication — courbes ROC et précision-rappel**

- **Courbe ROC** : pour chaque seuil possible, le taux de vrais positifs (rappel) en fonction du
  taux de faux positifs. Plus la courbe est proche du coin supérieur gauche, meilleur est le
  modèle ; la diagonale correspond au hasard.
- **Courbe précision-rappel** : pour chaque seuil, la précision en fonction du rappel. La ligne
  horizontale est le taux de base (ce que donnerait un tirage au hasard). Le point marque le
  **seuil D9** retenu : on y lit directement le compromis choisi.

La baseline métier D8 est tracée en pointillés pour visualiser l'apport du modèle.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for nom, score, style in [("Régression logistique", proba_test, "-"), ("Baseline métier D8", score_regle_test, "--")]:
    fpr, tpr, _ = roc_curve(y_test, score)
    ax1.plot(fpr, tpr, style, label=f"{nom} (AUC = {roc_auc_score(y_test, score):.3f})")
    prec, rapp, _ = precision_recall_curve(y_test, score)
    ax2.plot(rapp, prec, style, label=f"{nom} (PR-AUC = {average_precision_score(y_test, score):.3f})")
ax1.plot([0, 1], [0, 1], ":", color="gray", label="Hasard (AUC = 0,5)")
ax1.set_xlabel("Taux de faux positifs")
ax1.set_ylabel("Taux de vrais positifs (rappel)")
ax1.set_title("Courbe ROC — jeu de test")
ax1.legend(loc="lower right")
ax2.axhline(y_test.mean(), linestyle=":", color="gray", label=f"Taux de base ({y_test.mean():.2f})")
ax2.scatter([metriques_test["rappel_seuil_D9"]], [metriques_test["precision_seuil_D9"]], color=COULEURS["churn"],
            zorder=5, s=60, label=f"Seuil D9 = {SEUIL_D9:.3f}")
ax2.set_xlabel("Rappel")
ax2.set_ylabel("Précision")
ax2.set_title("Courbe précision-rappel — jeu de test")
ax2.legend(loc="lower left")
fig.tight_layout()
sauver_figure(fig, "09_courbes_roc_pr")
plt.show()

📘 **Explication — matrices de confusion**

Une matrice de confusion compte les quatre situations possibles :

| | prédit « reste » | prédit « résilie » |
|---|---|---|
| **reste réellement** | vrai négatif | **faux positif** (relance inutile) |
| **résilie réellement** | **faux négatif** (compte perdu sans alerte) | vrai positif |

On compare le seuil par défaut (0,5) au seuil D9 : on doit voir les faux négatifs diminuer
fortement, au prix de davantage de faux positifs. C'est exactement l'arbitrage voulu par D4.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, seuil, titre in [(axes[0], 0.5, "Seuil par défaut 0,5"), (axes[1], SEUIL_D9, f"Seuil D9 = {SEUIL_D9:.3f}")]:
    matrice = confusion_matrix(y_test, proba_test >= seuil)
    ConfusionMatrixDisplay(matrice, display_labels=["Reste", "Résilie"]).plot(ax=ax, cmap="Blues", colorbar=False)
    tn, fp, fn, tp = matrice.ravel()
    ax.set_title(f"{titre}\n{fn} comptes perdus sans alerte (FN) · {fp} relances inutiles (FP)", fontsize=11)
    ax.set_xlabel("Prédiction du modèle")
    ax.set_ylabel("Réalité")
    ax.grid(False)
fig.tight_layout()
sauver_figure(fig, "10_matrices_confusion")
plt.show()

**Lecture.** Au seuil D9, le rappel mesuré sur le test est proche de la cible de 80 % fixée sur
le train : le seuil se **généralise**. Le nombre de faux négatifs diminue fortement par rapport
au seuil de 0,5, au prix de faux positifs supplémentaires, ce qui est l'arbitrage voulu (D4).
Le critère D8 est respecté sur le test : le modèle apporte une valeur nette par rapport à une
règle métier simple.

### 9.4 Les probabilités sont-elles fiables ? Calibration

📘 **Explication — calibration des probabilités**

La priorisation (D14) multiplie la probabilité de churn par la CLV estimée pour obtenir une
**perte attendue en euros**. Ce calcul n'a de sens que si les probabilités sont **calibrées** :
parmi les comptes à qui le modèle donne 30 %, environ 30 % doivent effectivement partir.

- La **courbe de calibration** range les comptes du test en 10 groupes de probabilité prédite et
  compare, pour chaque groupe, la probabilité moyenne prédite au taux de churn observé. Un
  modèle parfaitement calibré suit la diagonale.
- Le **score de Brier** est l'erreur quadratique moyenne entre probabilité et issue réelle (0 =
  parfait). On le compare à celui d'un modèle qui prédirait toujours le taux de base.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

prob_observee, prob_predite = calibration_curve(y_test, proba_test, n_bins=10, strategy="quantile")
brier = brier_score_loss(y_test, proba_test)
brier_reference = brier_score_loss(y_test, np.full_like(proba_test, y_train.mean()))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], ":", color="gray", label="Calibration parfaite")
ax.plot(prob_predite, prob_observee, "o-", label="Régression logistique")
ax.set_xlabel("Probabilité prédite (moyenne du groupe)")
ax.set_ylabel("Taux de churn observé")
ax.set_title(f"Calibration — Brier = {brier:.3f} (référence {brier_reference:.3f})")
ax.legend()
sauver_figure(fig, "11_calibration")
plt.show()
print(f"Probabilité moyenne prédite : {proba_test.mean():.3f} | taux de churn réel : {y_test.mean():.3f}")

**Lecture.** La courbe suit la diagonale et le score de Brier est bien inférieur à la référence :
les probabilités sont fiables. Le calcul `perte attendue = p(churn) × CLV` est donc légitime.
C'est un avantage de la régression logistique, dont les probabilités sont naturellement bien
calibrées (les forêts aléatoires, par exemple, le sont souvent moins).

### 9.5 Seuil et coût métier : pourquoi D9 plutôt qu'une minimisation du coût

📘 **Explication — analyse de coût**

Pour chaque seuil possible, on calcule sur les prédictions hors pli du train :

`coût = nombre de faux négatifs × 6 879 € + nombre de faux positifs × coût d'une relance`

avec les deux bornes de l'hypothèse H01 (50 € et 150 €). On repère le seuil qui minimise ce
coût, et la part de comptes qu'il faudrait alors contacter.

Cette analyse montre pourquoi on ne retient pas le seuil de coût minimal : avec un faux négatif
46 à 137 fois plus cher qu'un faux positif, il faudrait contacter la majorité des comptes, bien
au-delà de la capacité de l'équipe (D10). D9 fixe donc un objectif de rappel atteignable, et
D10/D14 gèrent la capacité.

In [ ]:
seuils = np.linspace(0.01, 0.9, 90)
fig, ax = plt.subplots(figsize=(8, 4))
for cout_fp in COUTS_FP_EUR:
    couts = [(((proba_oof < s) & (y_train == 1)).sum() * COUT_FN_EUR
              + ((proba_oof >= s) & (y_train == 0)).sum() * cout_fp) / 1e3 for s in seuils]
    s_min = seuils[int(np.argmin(couts))]
    ax.plot(seuils, couts, label=f"coût FP = {cout_fp} € (minimum au seuil {s_min:.2f}, "
                                 f"{(proba_oof >= s_min).mean():.0%} des comptes contactés)")
ax.axvline(SEUIL_D9, color=COULEURS["churn"], linestyle="--", label=f"seuil D9 = {SEUIL_D9:.3f} ({signales_oof.mean():.0%} contactés)")
ax.set_xlabel("Seuil de décision")
ax.set_ylabel("Coût total estimé (k€, train hors pli)")
ax.set_title("Coût des erreurs selon le seuil (hypothèses F14 et H01)")
ax.legend(fontsize=8)
sauver_figure(fig, "12_cout_seuil")
plt.show()

### 9.6 Explicabilité : quels facteurs pèsent sur le risque ?

📘 **Explication — deux lectures complémentaires**

1. **Coefficients de la régression logistique.** Les variables numériques étant standardisées,
   un coefficient se lit ainsi : « quand la variable augmente d'un écart-type, le logarithme du
   rapport de chances de churn augmente de ce coefficient ». Positif = augmente le risque,
   négatif = le diminue. C'est l'explication que l'on peut donner à un CSM.
2. **Importance par permutation** (sur le test). Pour chaque variable, on mélange aléatoirement
   ses valeurs (ce qui casse son lien avec la cible) et on mesure la baisse de PR-AUC. Une
   forte baisse signifie que le modèle s'appuie vraiment sur cette variable. Cette méthode est
   indépendante du type de modèle. On répète le mélange 20 fois pour obtenir une moyenne et un
   écart-type.

In [ ]:
from sklearn.inspection import permutation_importance

noms = [n.split("__", 1)[1] for n in modele_churn.named_steps["preparation"].get_feature_names_out()]
coefs = pd.Series(modele_churn.named_steps["modele"].coef_[0], index=noms)
top_coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(12)[::-1]

perm = permutation_importance(modele_churn, X_test, y_test, scoring="average_precision",
                              n_repeats=20, random_state=RANDOM_STATE)
importance = (pd.DataFrame({"variable": FEATURES_V2, "baisse_pr_auc": perm.importances_mean,
                            "ecart_type": perm.importances_std})
              .sort_values("baisse_pr_auc", ascending=False).reset_index(drop=True))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.barh(top_coefs.index, top_coefs.values,
         color=[COULEURS["churn"] if v > 0 else COULEURS["non_churn"] for v in top_coefs.values])
ax1.set_title("12 plus forts coefficients (rouge = augmente le risque)")
ax1.set_xlabel("coefficient (variables standardisées)")
top_perm = importance.head(12)[::-1]
ax2.barh(top_perm["variable"], top_perm["baisse_pr_auc"], xerr=top_perm["ecart_type"], color=COULEURS["neutre"])
ax2.set_title("Importance par permutation (test, 20 répétitions)")
ax2.set_xlabel("baisse de PR-AUC quand la variable est mélangée")
fig.tight_layout()
sauver_figure(fig, "13_explicabilite")
plt.show()
display(importance.head(8).round(4))

📘 **Explication — les leurres n'apportaient rien : preuve sur le modèle v1**

Pour ne pas sur-interpréter les leurres, on vérifie leur rôle réel dans le **modèle v1**, qui
les contenait encore. On recharge ce modèle, on calcule l'importance par permutation de ses 30
variables sur le même jeu de test, et on regarde le rang des leurres : s'ils sont en bas du
classement avec une importance proche de zéro, les exclure était justifié.

In [ ]:
modele_v1 = joblib.load(MODEL_V1_DIR / "model.joblib")
features_v1_modele = json.loads((MODEL_V1_DIR / "model_manifest.json").read_text(encoding="utf-8"))["features"]
test_v1 = pd.read_parquet(GOLD_DIR / "clients_churn_gold.parquet").query("split == 'test'")
perm_v1 = permutation_importance(modele_v1, test_v1[features_v1_modele], test_v1["churn"],
                                 scoring="average_precision", n_repeats=10, random_state=RANDOM_STATE)
rang_v1 = (pd.DataFrame({"variable": features_v1_modele, "baisse_pr_auc": perm_v1.importances_mean})
           .sort_values("baisse_pr_auc", ascending=False).reset_index(drop=True))
rang_v1["rang"] = rang_v1.index + 1
display(rang_v1[rang_v1["variable"].isin(LEURRES)].round(4))
print(f"Rang moyen des leurres dans le modèle v1 : {rang_v1.loc[rang_v1['variable'].isin(LEURRES), 'rang'].mean():.1f} sur {len(rang_v1)}")

**Lecture.** Les facteurs de risque sont cohérents avec l'analyse exploratoire et avec
l'intuition métier : récence de connexion, ancienneté, nombre d'intégrations, tickets de
support et satisfaction. Les retards de paiement, 5ᵉ facteur du modèle v2.0, ne sont plus que
8ᵉ, avec une importance proche de zéro, une fois les valeurs impossibles neutralisées (§6.5) :
une bonne part de leur poids venait des comptes où ils sont impossibles. Dans le modèle v1, quatre leurres sur cinq se
classaient entre le 19ᵉ et le 29ᵉ rang sur 30, avec une importance nulle ou négative (le
mélange ne dégrade rien) ; `pays` arrivait au 11ᵉ rang, mais avec une baisse de PR-AUC
négligeable (≈ 0,002). Leur exclusion ne retire aucun signal utile, ce que confirme l'ablation
du §8.4. **Prudence
d'interprétation** : ces lectures décrivent des associations apprises par le modèle, pas des
causes. On ne peut pas conclure qu'ajouter une intégration à un compte réduirait son risque.

#### 9.6.1 Confronter le modèle à la base de connaissance

Les lectures précédentes montrent **ce que le modèle a appris** ; il reste à vérifier que c'est
**plausible**. Le projet tient pour cela une **base de connaissance** versionnée,
`knowledge/base_connaissance.yaml`, écrite **avant** de regarder les coefficients :

- pour chaque variable : libellé, unité et plage du dictionnaire de données (énoncé, §2.1), et
  **sens d'effet attendu** (hausse ou baisse du risque), justifié par le métier ou l'EDA du §6 ;
  « indéterminé » quand aucune hypothèse n'est défendable ;
- les **variables exclues** et leur raison : fuite, cibles, identifiant, texte libre, leurres,
  redondances ;
- les règles du système de décision (D9, D10, D14, D3).

Le modèle est confronté à cette base par des contrôles automatiques. Un écart ne prouve pas une
erreur ; il **oblige à l'examiner**. La même base sert à formuler les explications de chaque
compte en langage métier (§10.2.1).

📘 **Explication — les cinq contrôles**

| Contrôle | Question posée | Gravité |
|---|---|---|
| Exclusion | Une variable exclue (fuite, cible, identifiant, leurre…) est-elle dans le modèle ? | bloquant (redondance : avertissement) |
| Couverture | Chaque variable du modèle est-elle décrite dans la base ? | avertissement |
| Sens | Le coefficient appris va-t-il dans le sens attendu ? | avertissement : entre variables corrélées, un coefficient peut changer de signe sans erreur |
| Concentration | Une seule variable porte-t-elle plus de 50 % de l'explication ? | bloquant : aucune variable disponible avant la décision ne dépasse seule une AUC de 0,76 (§6) ; au-delà, fuite probable |
| Modalités | Les catégories apprises sont-elles celles du dictionnaire ? | avertissement ou information |

La **contribution** d'une variable au score d'un compte vaut coefficient × (valeur transformée −
moyenne d'entraînement) ; elle est définie et utilisée pour chaque compte au §10.2.1. La **part
d'explication** d'une variable est la moyenne de la valeur absolue de ses contributions,
rapportée au total : on la calcule ici sur le jeu d'entraînement, sans toucher au test. Les
seuils et les gravités sont fixés dans la base, a priori, et non ajustés aux résultats.

In [ ]:
from src.explain import (charger_base_connaissance, contributions, controler_modele, effets_lineaires,
                         parts_explication, reference_explication)

base_connaissance = charger_base_connaissance(ROOT / "knowledge" / "base_connaissance.yaml")
reference_expl = reference_explication(modele_churn, X_train)     # figée dans explication_reference.json (§10.1)
effets = effets_lineaires(modele_churn, reference_expl)

sens = pd.DataFrame([{"variable": v, "sens attendu": base_connaissance["variables"][v]["sens_attendu"],
                      "coefficient appris": effets[v]} for v in effets])
sens = sens[sens["sens attendu"] != "indetermine"].copy()
sens["conforme"] = np.where(np.sign(sens["coefficient appris"]) == sens["sens attendu"].map({"hausse": 1, "baisse": -1}),
                            "oui", "NON")
display(sens.sort_values("coefficient appris", key=np.abs, ascending=False).set_index("variable").round(3))

parts_v2 = parts_explication(contributions(modele_churn, X_train, reference_expl))
print("Part de l'explication (train) :", ", ".join(f"{v} {p:.0%}" for v, p in parts_v2.head(5).items()),
      f"... (maximum admis : {base_connaissance['controles']['part_max_une_variable']:.0%})")

controles_v2 = controler_modele(FEATURES_V2, base_connaissance, modele_churn, reference_expl, X_train)
print(f"\nContrôles du modèle v2 : {(controles_v2['gravite'] == 'bloquant').sum()} bloquant(s), "
      f"{(controles_v2['gravite'] == 'avertissement').sum()} avertissement(s), "
      f"{(controles_v2['gravite'] == 'information').sum()} information(s)")
display(controles_v2)

**Lecture.** Le modèle v2.1 ne déclenche **aucun contrôle bloquant** : aucune variable exclue, et
l'explication est répartie entre les variables (la plus lourde, `anciennete_mois`, en porte
17 %, loin du maximum de 50 %). Sur les 14 variables numériques pour lesquelles la base fixe un
sens attendu, **12 vont dans le sens attendu**, dont les onze plus influentes. Deux
avertissements : `taux_utilisation_fonctionnalites` et `taux_retard_paiement_par_mois`, les deux
ratios construits au §7.8, ont un coefficient faible et de signe contraire à l'attendu. On les
examine au §9.6.3.

#### 9.6.2 Ces contrôles auraient-ils détecté nos erreurs ?

📘 **Explication — tester le détecteur sur des erreurs connues**

Un contrôle qui ne sonne jamais ne prouve rien. On le soumet à deux erreurs **réelles** de ce
projet :

1. le **modèle v1**, qui contenait encore les cinq leurres et les cinq colonnes du catalogue (§7) ;
2. un modèle **volontairement fautif**, entraîné avec `sante_compte_fin_periode`, la variable de
   fuite identifiée au §6 (même pipeline et même régularisation que le modèle retenu). On mesure
   sa PR-AUC en validation croisée sur le train, puis on lui applique les contrôles. Ce modèle
   n'est ni sauvegardé ni utilisé ailleurs.

In [ ]:
features_v1 = json.loads((MODEL_V1_DIR / "model_manifest.json").read_text(encoding="utf-8"))["features"]
controles_v1 = controler_modele(features_v1, base_connaissance)
print(f"Modèle v1 ({len(features_v1)} variables) :", controles_v1["gravite"].value_counts().to_dict())
display(controles_v1[["variable", "gravite", "constat"]])

features_fuite = FEATURES_V2 + ["sante_compte_fin_periode"]
modele_fuite = pipeline_modele(clone(modele_churn.named_steps["modele"]),
                               numeriques=NUMERIQUES_V2 + ["sante_compte_fin_periode"])
cv_fuite = evaluer_cv("avec la variable de fuite", modele_fuite, X=train[features_fuite])
modele_fuite.fit(train[features_fuite], y_train)
reference_fuite = reference_explication(modele_fuite, train[features_fuite])
parts_fuite = parts_explication(contributions(modele_fuite, train[features_fuite], reference_fuite))
controles_fuite = controler_modele(features_fuite, base_connaissance, modele_fuite, reference_fuite, train[features_fuite])

print(f"\nModèle avec fuite : PR-AUC CV {cv_fuite['pr_auc']:.3f} (modèle retenu : {grille.best_score_:.3f})")
print("Part de l'explication :", ", ".join(f"{v} {p:.0%}" for v, p in parts_fuite.head(3).items()))
display(controles_fuite[controles_fuite["gravite"] == "bloquant"])

**Lecture — le contrôle détecte les deux erreurs.**

- **Modèle v1** : les cinq leurres sont bloquants, les cinq colonnes du catalogue donnent un
  avertissement. Si la base avait existé avant la v1, ce modèle n'aurait pas été mis en service.
- **Modèle avec fuite** : sa PR-AUC en validation croisée atteint **0,997**, contre 0,785 pour
  le modèle retenu. **Aucune métrique de performance ne l'aurait fait refuser** : il aurait même
  été sélectionné. Deux contrôles le bloquent, indépendamment l'un de l'autre : la variable
  figure dans les exclusions, et elle porte à elle seule **83 %** de l'explication. Le second
  contrôle ne dépend d'aucune liste : il aurait aussi signalé une fuite que personne n'avait
  encore identifiée.

C'est le rôle de l'explicabilité dans ce projet : un score trop beau ne se célèbre pas, il
s'explique, et c'est l'explication qui révèle l'échec.

#### 9.6.3 Deux avertissements de sens : diagnostic

📘 **Explication — un avertissement se vérifie**

Le contrôle de sens signale des variables dont le coefficient va à l'inverse de l'attendu. Deux
hypothèses : une vraie anomalie (erreur de construction, effet réel inattendu), ou un
**artefact de corrélation**. Les deux variables signalées sont des ratios construits au §7.8 à
partir de variables déjà présentes dans le modèle, et entre variables corrélées le modèle peut
répartir l'effet de façon arbitraire. Deux vérifications, **sur le seul jeu d'entraînement** :

1. **stabilité du signe** : on réentraîne le modèle sur 200 rééchantillonnages avec remise
   (bootstrap) du train. Un effet réel garde le même signe presque à chaque fois ; un effet
   instable change de signe au gré de l'échantillon ;
2. **ablation** : PR-AUC en validation croisée avec et sans les variables signalées.

In [ ]:
avertis = controles_v2.loc[controles_v2["controle"] == "sens", "variable"].tolist()
print("Variables signalées par le contrôle de sens :", avertis)

rng = np.random.default_rng(RANDOM_STATE)
signe_initial = np.sign(pd.Series(effets))
signes_bootstrap = []
for _ in range(200):
    tirage = rng.integers(0, len(X_train), len(X_train))
    m = clone(modele_churn).fit(X_train.iloc[tirage], y_train.iloc[tirage])
    signes_bootstrap.append(np.sign(pd.Series(effets_lineaires(m, reference_explication(m, X_train.iloc[tirage])))))
stabilite = (pd.DataFrame(signes_bootstrap) == signe_initial).mean()
tableau_stabilite = pd.DataFrame({"coefficient": pd.Series(effets), "même signe (bootstrap)": stabilite,
                                  "signalée": pd.Series({v: v in avertis for v in effets})})
display(tableau_stabilite.sort_values("même signe (bootstrap)").head(6).round(3))

sans_avertis = [c for c in NUMERIQUES_V2 if c not in avertis]
ablation_sens = pd.DataFrame([
    evaluer_cv("modèle retenu (20 variables)", clone(modele_churn)),
    evaluer_cv(f"sans les {len(avertis)} variables signalées",
               pipeline_modele(clone(modele_churn.named_steps["modele"]), numeriques=sans_avertis),
               X=X_train[CATEGORIELLES_V2 + sans_avertis]),
]).drop(columns="plis").set_index("modèle")
display(ablation_sens.round(4))

**Lecture — deux ratios sans apport, pour deux raisons différentes.**

- `taux_utilisation_fonctionnalites` ne garde son signe que dans **69 %** des rééchantillonnages,
  contre au moins 80 % pour toutes les variables non signalées. Son effet « inversé » est instable :
  c'est un **artefact de corrélation** avec les deux variables dont il est calculé (fonctionnalités
  utilisées et offertes).
- `taux_retard_paiement_par_mois` a changé de comportement avec la v2.1 : son signe est désormais
  **stable** (94 %), alors qu'il l'était dans 55 % des cas seulement en v2.0. Une fois les valeurs
  impossibles neutralisées, le ratio est une fonction exacte de deux variables déjà présentes
  (retards et ancienneté) ; à retards et ancienneté donnés, son coefficient ne peut pas se lire
  comme l'effet d'un retard de plus. Son sens « contraire » n'a donc pas d'interprétation métier.
- Dans les deux cas, le coefficient est proche de zéro (+0,037 et −0,074), et retirer les deux
  ratios ne change pas la PR-AUC en validation croisée (0,785 et 0,784).

Ce n'est pas une faute grave : le modèle reste performant et bien calibré. C'est en revanche
une complexité inutile, qui brouille l'explication donnée aux CSM. **Décision** : la révision
v2.1 se limite à corriger une donnée impossible ; retirer les ratios est un choix de
modélisation, qui doit repasser par toute la procédure de sélection du §8. On le recommande
pour la prochaine version (§14).

### 9.7 Cible secondaire : régression de la valeur vie client

📘 **Explication — modéliser la CLV**

La CLV varie de 300 € à 2 M€ : sa distribution est très étirée vers les grandes valeurs. On
modélise donc `log(1 + CLV)` : le logarithme ramène les écarts à une échelle comparable, et le
modèle n'est plus dominé par quelques très gros comptes. Les prédictions sont retransformées en
euros (`exp(x) − 1`) avant tout calcul métier.

Mêmes règles que pour le churn : mêmes 20 variables (ni `churn`, ni la variable de fuite),
validation croisée à 5 plis sur le train, baseline (médiane), puis trois modèles. Le
coefficient de détermination **R²** mesure la part de la variance expliquée (1 = parfait, 0 =
pas mieux que la moyenne).

La cellule mesure aussi la **taille sur disque** des modèles à base d'arbres : c'est une
contrainte d'exploitation (§11) : un modèle de 60 Mo alourdit l'image Docker et son chargement.

In [ ]:
import io

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

y_clv_train = np.log1p(train["valeur_vie_client_eur"])
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidats_clv = {
    "Baseline (médiane)": pipeline_modele(DummyRegressor(strategy="median")),
    "Ridge": pipeline_modele(Ridge(alpha=1.0)),
    "Forêt aléatoire": pipeline_modele(RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                                                             random_state=RANDOM_STATE, n_jobs=-1), standardiser=False),
    "Gradient boosting": pipeline_modele(HistGradientBoostingRegressor(learning_rate=0.05, max_iter=400,
                                                                       random_state=RANDOM_STATE), standardiser=False),
}
lignes = []
for nom, pipe in candidats_clv.items():
    res = cross_validate(pipe, X_train, y_clv_train, cv=cv_reg, scoring=["r2", "neg_mean_absolute_error"])
    taille = np.nan
    if nom in ("Forêt aléatoire", "Gradient boosting"):
        tampon = io.BytesIO()
        joblib.dump(clone(pipe).fit(X_train, y_clv_train), tampon)
        taille = len(tampon.getvalue()) / 1e6
    lignes.append({"modèle": nom, "r2_log": res["test_r2"].mean(), "r2_log_std": res["test_r2"].std(),
                   "mae_log": -res["test_neg_mean_absolute_error"].mean(), "taille_mo": taille})
comparaison_clv = pd.DataFrame(lignes).set_index("modèle")
display(comparaison_clv.round(3))

meilleur_clv = comparaison_clv["r2_log"].idxmax()
egalite_clv = comparaison_clv[comparaison_clv["r2_log"] >= comparaison_clv.loc[meilleur_clv, "r2_log"]
                              - comparaison_clv.loc[meilleur_clv, "r2_log_std"]]
MODELE_CLV_RETENU = egalite_clv["taille_mo"].fillna(0).idxmin() if len(egalite_clv) > 1 else meilleur_clv
print(f"Meilleur R² moyen : {meilleur_clv} ; à égalité : {list(egalite_clv.index)}")
print(f"Modèle CLV retenu : {MODELE_CLV_RETENU} (à performance égale, le plus léger à exploiter)")
modele_clv = clone(candidats_clv[MODELE_CLV_RETENU]).fit(X_train, y_clv_train)

**Lecture.** La forêt aléatoire et le gradient boosting sont à égalité (écart inférieur à un
écart-type), loin devant Ridge et la baseline. À performance égale, je retiens le **gradient
boosting**, environ 40 fois plus léger sur disque que la forêt : un critère d'exploitation
légitime (taille de l'image Docker, temps de chargement de l'API). La performance en euros sur
le test est présentée en §12.

### 9.8 Transfert de connaissances : non applicable

Le **transfert de connaissances** (*transfer learning*) consiste à partir d'un modèle déjà
entraîné sur une autre tâche et à l'adapter, au lieu d'apprendre de zéro. Il sert quand les
données sont rares et qu'un modèle pré-entraîné proche existe (images, textes). Ici, il ne
s'applique pas :

- les données sont tabulaires et propres à cet éditeur : aucun modèle pré-entraîné ne connaît ses
  variables ;
- 4 000 comptes suffisent à une régression logistique à 20 variables, stable d'un pli à l'autre
  (§8.3) ;
- la régression CLV ne réutilise pas le modèle de churn : les deux sont entraînés séparément, sur
  les mêmes variables.

Ce qui se transmet d'une version à l'autre, ce sont les **règles** (préparation dans `src/`, base
de connaissance, décisions D1–D15), pas les poids : au réentraînement (§13.2), le modèle est
réappris de zéro sur les données à jour, avec le même pipeline.

### 📓 Journal de bord — §9 Entraînement, validation, ajustement [C5]

- **Régularisation** réglée par recherche sur grille en validation croisée ; la courbe est
  plate autour de l'optimum : le modèle est robuste au choix de `C`.
- **Seuil D9 calculé hors pli sur le train**, puis vérifié sur le test : le rappel obtenu sur le
  test est proche de la cible de 80 %. **Correction d'un défaut de la v1**, qui calibrait le
  seuil sur le test.
- **Évaluation sur le test** (seconde lecture assumée pour la v2.1, §9.3) : ROC-AUC, PR-AUC, matrices de confusion, critère D8
  respecté.
- **Calibration vérifiée** (courbe et score de Brier) : les probabilités sont fiables, ce qui
  légitime la perte attendue en euros.
- **Analyse de coût** : minimiser le coût seul obligerait à contacter la majorité des comptes,
  incompatible avec la capacité CS : cela justifie D9 + D10.
- **Explicabilité** : coefficients et importance par permutation cohérents avec le métier ;
  leurres confirmés inutiles sur le modèle v1.
- **Base de connaissance** (`knowledge/base_connaissance.yaml`, écrite avant de regarder le
  modèle) : aucun contrôle bloquant, 12 sens attendus sur 14 respectés. Les contrôles bloquent le
  modèle v1 (5 leurres) et un modèle avec fuite que sa PR-AUC (0,997) aurait fait sélectionner
  (83 % de l'explication sur une seule variable). Les deux avertissements (ratios construits)
  sont sans apport en ablation (l'un instable, l'autre fonction exacte de deux variables du
  modèle) : à retirer en v3.
- **CLV** : gradient boosting sur `log(1 + CLV)`, retenu à performance égale pour sa légèreté.
- **Transfert de connaissances** : non applicable (données tabulaires propres à l'éditeur,
  volume suffisant) ; d'une version à l'autre se transmettent les règles, pas les poids (§9.8).

## 10. Implémentation et mise en exploitation — journal de bord [C6]

### 10.1 Sérialisation des modèles et de la règle de décision

📘 **Explication — sauvegarder ce qui sera exploité**

**Sérialiser** un modèle, c'est l'enregistrer sur disque pour le réutiliser sans le réentraîner.
On sauvegarde le **pipeline complet** (préparation + modèle) avec `joblib` : en production, on
lui donnera des données brutes au format des 20 features, et il appliquera lui-même
l'imputation, l'encodage et la standardisation apprises à l'entraînement.

On écrit aussi deux **manifestes** JSON, lus par l'API au démarrage :

- `model_manifest.json` : liste ordonnée des features attendues, version et hash du Gold qui a
  servi à l'entraînement, hyperparamètres, métriques ;
- `scoring_manifest.json` : la **règle de décision** (seuil D9, capacité D10, catalogue D11,
  formule D14). Le seuil est figé ici : il n'est jamais recalculé au moment du scoring, où le
  churn réel n'est par définition pas connu ;
- `explication_reference.json` : la **référence des explications** (moyennes du jeu
  d'entraînement après préparation, §9.6.1). L'API en a besoin pour expliquer un score, car
  elle ne dispose pas du jeu d'entraînement.

Les noms de fichiers sont ceux qu'attend l'API (`src/api.py`) : pour mettre ce modèle en
service, il suffit de pointer la variable `MODEL_DIR` de l'API vers `data/model_v2`.

In [ ]:
joblib.dump(modele_churn, MODEL_DIR / "model.joblib")
joblib.dump(modele_clv, MODEL_DIR / "model_clv.joblib")

manifeste_modele = {
    "couche": "model", "version": "v2.1", "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "commit_git": commit_git(), "gold_version": "v2.1", "gold_sha256": sha_gold_v2,
    "modele_retenu": MODELE_RETENU, "hyperparametres": {"C": C_RETENU}, "random_state": RANDOM_STATE,
    "features": FEATURES_V2, "cible": "churn",
    "metriques_cv_train": comparaison.drop(columns="plis").set_index("modèle").round(4).to_dict(orient="index"),
    "metriques_test": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in metriques_test.items()},
    "modele_clv": {"modele_retenu": MODELE_CLV_RETENU, "cible": "log1p(valeur_vie_client_eur)"},
}
write_manifest(MODEL_DIR / "model_manifest.json", manifeste_modele)
write_manifest(MODEL_DIR / "metrics.json", {**manifeste_modele["metriques_test"],
                                            "pr_auc_cv": round(float(grille.best_score_), 4),
                                            "seuil_D9": round(float(SEUIL_D9), 4)})   # suivi par dvc metrics

manifeste_scoring = {
    "couche": "model", "sous_etape": "regle_de_decision", "version": "v2.1",
    "regle_decision": {
        "seuil_D9_valeur": round(float(SEUIL_D9), 4),
        "seuil_D9_calcule_sur": "prédictions hors pli du jeu d'entraînement (jamais le test)",
        "seuil_D9_rappel_cible": RAPPEL_CIBLE_D9,
        "rappel_test": round(metriques_test["rappel_seuil_D9"], 4),
        "precision_test": round(metriques_test["precision_seuil_D9"], 4),
        "capacite_csm_D10": CAPACITE_CSM_D10,
        "formule_priorite_D14": "perte_attendue = score_churn x valeur_vie_estimee_eur",
        "catalogue_actions_D11": ACTIONS_PAR_PRIORITE,
    },
    "colonnes_export_crm": ["client_id", "score_churn", "valeur_vie_estimee_eur", "priorite", "action_recommandee"],
}
write_manifest(MODEL_DIR / "scoring_manifest.json", manifeste_scoring)
write_manifest(MODEL_DIR / "explication_reference.json", reference_expl)   # référence des explications (§9.6.1)
print("Fichiers écrits dans", MODEL_DIR.relative_to(ROOT), ":", sorted(p.name for p in MODEL_DIR.glob("*.*")))

📘 **Explication — vérifier que la sauvegarde est fidèle**

Un test simple mais indispensable : on **recharge** les modèles depuis le disque et on vérifie
qu'ils donnent exactement les mêmes prédictions que les modèles en mémoire. Si ce n'était pas le
cas (version de bibliothèque différente, étape manquante dans le pipeline…), la production ne
reproduirait pas les résultats de ce notebook.

In [ ]:
churn_recharge = joblib.load(MODEL_DIR / "model.joblib")
clv_recharge = joblib.load(MODEL_DIR / "model_clv.joblib")
assert np.allclose(churn_recharge.predict_proba(X_test)[:, 1], proba_test)
assert np.allclose(clv_recharge.predict(X_test), modele_clv.predict(X_test))
print("Modèles rechargés : prédictions identiques à celles du notebook.")

### 10.2 Exemple d'usage : le système de décision appliqué au cycle mensuel

📘 **Explication — de deux scores à une liste de travail**

C'est la **troisième brique** de la solution (§2.3 bis) : le système de décision.

On simule un cycle de scoring mensuel sur les 1 000 comptes du jeu de test (comptes jamais vus
par les modèles) :

1. `scorer_batch` calcule pour chaque compte la probabilité de churn, la CLV estimée (en euros)
   et la **perte attendue** = probabilité × CLV ;
2. `assigner_priorites` applique la règle D14 :
   - **Haute** : comptes signalés (probabilité ≥ seuil D9), classés par perte attendue
     décroissante, dans la limite de **150** (capacité D10) ;
   - **Moyenne** : les autres comptes signalés ;
   - **Basse** : les comptes non signalés ;
3. chaque priorité reçoit l'action correspondante du catalogue D11.

On affiche la répartition des priorités et le **Top 10**, c'est-à-dire ce que l'équipe CS verrait
en tête de liste le premier jour du mois. L'export vers le CRM ne contient que 5 colonnes
(minimisation, §4).

In [ ]:
from src.scoring import assigner_priorites, scorer_batch

cycle = scorer_batch(X_test, test["client_id"], modele_churn, modele_clv)
cycle = assigner_priorites(cycle, seuil_d9=SEUIL_D9, capacite_haute=CAPACITE_CSM_D10)
print("Répartition des priorités :", cycle["priorite"].value_counts().reindex(["Haute", "Moyenne", "Basse"]).to_dict())
display(cycle.sort_values("perte_attendue_eur", ascending=False)
        .head(10)[["client_id", "score_churn", "valeur_vie_estimee_eur", "perte_attendue_eur", "priorite", "action_recommandee"]])

export_crm = cycle[manifeste_scoring["colonnes_export_crm"]]
export_crm.to_parquet(MODEL_DIR / "scoring_exemple_cycle.parquet", index=False)
assert "commentaire_csm" not in export_crm.columns
assert all(action_est_conforme_d3(a) for a in export_crm["action_recommandee"].unique())
print(f"Export CRM : {export_crm.shape[0]} lignes x {export_crm.shape[1]} colonnes -> scoring_exemple_cycle.parquet")

**Lecture.** Les 9 premiers comptes par perte attendue sont en priorité Haute. Le 10ᵉ reste
en priorité **Basse** alors que sa perte attendue est élevée : sa probabilité de churn est sous
le seuil D9, mais sa CLV estimée approche le million d'euros (974 k€). C'est une conséquence directe de
D14 (le seuil D9 est la porte d'entrée, la valeur ne fait qu'ordonner les comptes signalés).
Ce choix est assumé, mais il a un coût, mesuré en §12.4.

#### 10.2.1 Expliquer chaque priorité

📘 **Explication — pourquoi ce compte, et pourquoi cette priorité ?**

Chaque compte de la liste reçoit deux explications.

1. **Son score.** Pour une régression logistique, le logit du score (logarithme du rapport de
   chances) est une somme : constante + Σ coefficient × valeur transformée. On le réécrit par
   rapport à un **compte de référence**, la moyenne du jeu d'entraînement :
   logit(compte) = logit(référence) + Σ contributions, avec
   contribution = coefficient × (valeur transformée − valeur de référence).
   Une contribution positive **augmente** le risque, une négative le **diminue**. La
   décomposition est **exacte** (vérifiée ci-dessous) ; pour un modèle linéaire, ces
   contributions sont exactement les valeurs SHAP, sans approximation ni bibliothèque
   supplémentaire. On retient les trois facteurs qui poussent le plus le risque vers le haut et
   les trois qui le poussent le plus vers le bas, formulés avec les libellés et les unités de la
   base de connaissance.
2. **Sa décision.** La trace de la règle appliquée : score comparé au seuil D9, rang de la perte
   attendue parmi les comptes signalés (D14), capacité D10.

S'y ajoutent des **avertissements** sur les valeurs d'entrée : valeur manquante (remplacée par la
médiane d'entraînement), valeur hors de la plage du dictionnaire, modalité inconnue. Une
explication fondée sur une valeur imputée est fragile : le CSM doit le savoir.

Exemples affichés : le compte en tête de liste, le premier compte en priorité Moyenne, et le
compte du Top 10 resté en priorité Basse (§10.2).

**Limites de lecture** : une contribution décrit ce que le modèle a appris, pas une cause (§9.6) ;
entre variables corrélées (licences, utilisateurs actifs, taux d'adoption), le partage des
contributions est instable (§9.6.3).

In [ ]:
from src.explain import expliquer_batch

contrib_cycle = contributions(modele_churn, X_test, reference_expl)
ecart_decomposition = float(np.abs(reference_expl["logit_reference"] + contrib_cycle.sum(axis=1)
                                   - modele_churn.decision_function(X_test)).max())
print(f"Décomposition exacte : écart maximal entre logit(référence) + Σ contributions et le logit du modèle "
      f"= {ecart_decomposition:.1e}")

explications_cycle = expliquer_batch(modele_churn, X_test, cycle, reference_expl, base_connaissance,
                                     seuil_d9=SEUIL_D9, capacite_haute=CAPACITE_CSM_D10)


def afficher_explication(position: int) -> None:
    ligne, e = cycle.iloc[position], explications_cycle[position]
    print(f"\n{ligne['client_id']} | score {ligne['score_churn']:.3f} | perte attendue "
          f"{ligne['perte_attendue_eur']:,.0f} € | priorité {ligne['priorite']}".replace(",", " "))
    print("  Décision :", e["decision"])
    for f in e["facteurs_hausse"]:
        print("  ▲", f["texte"])
    for f in e["facteurs_baisse"]:
        print("  ▼", f["texte"])
    for a in e["avertissements"]:
        print("  ⚠", a)


classement = cycle.sort_values("perte_attendue_eur", ascending=False)
top10_basse = classement.head(10).query("priorite == 'Basse'")
exemples = [classement.index[0], classement.query("priorite == 'Moyenne'").index[0],
            (top10_basse if len(top10_basse) else classement.query("priorite == 'Basse'")).index[0]]
for etiquette in exemples:
    afficher_explication(cycle.index.get_loc(etiquette))

avec_avertissement = np.mean([len(e["avertissements"]) > 0 for e in explications_cycle])
premier_facteur_haute = pd.Series([e["facteurs_hausse"][0]["variable"] for e, p in zip(explications_cycle, cycle["priorite"])
                                   if p == "Haute" and e["facteurs_hausse"]]).value_counts(normalize=True)
print(f"\nComptes du cycle avec au moins un avertissement sur leurs données : {avec_avertissement:.0%}")
print("Premier facteur de hausse des 150 priorités Hautes :",
      ", ".join(f"{v} {p:.0%}" for v, p in premier_facteur_haute.head(4).items()))

**Lecture.** La décomposition est exacte à la précision de la machine (écart de l'ordre de
10⁻¹⁵). Les trois exemples se lisent sans connaître le modèle :

- le **compte en tête de liste** cumule les signaux de désengagement : 45 jours sans connexion,
  aucune intégration, un taux d'adoption des licences de 0 %. Sa perte attendue très élevée
  vient de sa valeur vie estimée ;
- le **premier compte en priorité Moyenne** est signalé, mais sa perte attendue le place au
  151ᵉ rang, juste au-delà de la capacité de 150 comptes. C'est la règle D10, pas le modèle, qui
  le prive d'un appel. C'est aussi un compte d'un mois dont les retards de paiement étaient
  impossibles (§6.5) : neutralisés, ils sont remplacés par la médiane (0 retard), qui **diminue**
  son risque par rapport à la moyenne. L'imputation n'est pas neutre pour l'explication, et
  l'avertissement le dit au CSM ;
- le **compte du Top 10 resté en priorité Basse** (§10.2) : un revenu mensuel de 69 461 €,
  817 licences et un taux d'adoption de 3,7 % tirent son risque vers le haut, mais l'absence de
  ticket, les 40 fonctionnalités de son plan et un délai de réponse du support de 0,5 h pèsent
  davantage dans l'autre sens ; sa probabilité (0,20) reste sous le seuil.

Sur l'ensemble du cycle, 36 % des comptes ont au moins une valeur imputée (28 % en v2.0 : la
hausse vient des retards neutralisés). Parmi les 150 priorités Hautes, le premier facteur de
risque est l'ancienneté (37 %), la récence de connexion (24 %), les tickets (18 %) ou les
intégrations (9 %) : le CSM sait d'emblée par quel sujet ouvrir l'échange. En v2.0, les retards
de paiement étaient le premier facteur de 23 % des priorités Hautes ; ils ne figurent plus parmi
les quatre premiers.

### 10.3 Scorer un export brut : la même chaîne qu'à l'entraînement

📘 **Explication — éviter l'écart entre entraînement et production**

Un piège classique en production : le modèle a été entraîné sur des données nettoyées par le
notebook, mais on lui envoie en production des données préparées autrement. Les prédictions
deviennent alors fausses sans aucune erreur visible (*training-serving skew*).

Ici, on prend le fichier `churn_saas_echantillon.csv` (50 comptes, **au format brut** d'un export
CRM, avec ses défauts) et on lui applique **exactement les mêmes fonctions** qu'à l'entraînement :
`load_raw` → `clean_silver` → `construire_features_v2`. Les valeurs manquantes restantes sont
laissées au pipeline, qui les impute avec les médianes apprises sur le train.

On montre aussi que le **modèle v1** échouait sur une valeur manquante : il n'avait pas
d'imputation interne. C'est le défaut corrigé par la v2.

In [ ]:
brut_echantillon = load_raw(FICHIERS_SOURCES["echantillon"], ingested_at=datetime.now(timezone.utc).isoformat())
silver_echantillon, _ = clean_silver(brut_echantillon, catalogue_brut, impute_medians=False)
X_echantillon = construire_features_v2(silver_echantillon)
print(f"Export brut : {len(brut_echantillon)} comptes -> {X_echantillon.shape[1]} features, "
      f"{int(X_echantillon.isna().sum().sum())} valeurs manquantes laissées au pipeline")

scores_echantillon = scorer_batch(X_echantillon, silver_echantillon["client_id"], modele_churn, modele_clv)
display(scores_echantillon.head(5))

ligne_avec_manquant = test_v1[features_v1_modele].head(1).assign(csat=np.nan)
try:
    modele_v1.predict_proba(ligne_avec_manquant)
    print("Modèle v1 : valeur manquante acceptée")
except ValueError as erreur:
    print(f"Modèle v1 sur une valeur manquante -> ValueError : {str(erreur).splitlines()[0]}")

### 10.4 L'API de scoring, testée de bout en bout

📘 **Explication — tester l'API sans serveur**

L'API (`src/api.py`, FastAPI) expose `/health`, `/ready`, `/metrics` et `/score-batch`. Elle est
déployée en conteneur Docker (§11). Ici, on la teste **dans le notebook**, sans démarrer de
serveur, grâce au `TestClient` de FastAPI qui simule des requêtes HTTP.

Les tests :

1. `/health` et `/ready` répondent (le modèle v2 est chargé) ;
2. `/score-batch` sur les 50 comptes de l'export brut renvoie **exactement les mêmes scores**
   que l'appel direct aux modèles (même code, mêmes modèles, aucune divergence) ;
3. **robustesse** : une valeur manquante et un plan inconnu sont acceptés (code 200) ;
4. **sécurité** : une requête sans clé API est refusée (code 401).

In [ ]:
from fastapi.testclient import TestClient

os.environ["API_KEY"] = "cle-demo-notebook"
import src.api as api

api.load_artifacts(MODEL_DIR)
client_api = TestClient(api.app)
entete = {"X-API-Key": "cle-demo-notebook"}

print("GET /health ->", client_api.get("/health").json())
print("GET /ready  ->", client_api.get("/ready").json())

lignes_json = json.loads(X_echantillon.to_json(orient="records"))      # NaN -> null
charge = {"clients": [{"client_id": cid, "features": f}
                      for cid, f in zip(silver_echantillon["client_id"], lignes_json)]}
reponse = client_api.post("/score-batch", json=charge, headers=entete)
resultats_api = pd.DataFrame(reponse.json()["resultats"])
ecart = (resultats_api.set_index("client_id")["score_churn"]
         - scores_echantillon.set_index("client_id")["score_churn"]).abs().max()
print(f"POST /score-batch -> {reponse.status_code}, {len(resultats_api)} comptes, "
      f"écart maximal avec l'appel direct : {ecart}")

cas_limites = {
    "valeur manquante (csat = null)": dict(lignes_json[0], csat=None),
    "plan inconnu (Premium)": dict(lignes_json[0], plan="Premium"),
}
for nom, f in cas_limites.items():
    r = client_api.post("/score-batch", json={"clients": [{"client_id": "TEST", "features": f}]}, headers=entete)
    print(f"{nom:32s} -> {r.status_code}, score {r.json()['resultats'][0]['score_churn']}")
r = client_api.post("/score-batch", json=charge)
print(f"{'sans clé API':32s} -> {r.status_code} {r.json()['detail']}")
assert reponse.status_code == 200 and ecart <= 1e-3 and r.status_code == 401

📘 **Explication — l'explication servie par l'API**

Avec le paramètre `?explain=true`, `/score-batch` renvoie pour chaque compte les explications du
§10.2.1 : facteurs, trace de la décision, avertissements. Sans ce paramètre, la réponse est
inchangée (compatibilité avec le connecteur CRM). La capacité D10 s'applique au lot envoyé.

On vérifie :

1. que l'explication de l'API est **identique** à l'appel direct (mêmes contributions) ;
2. le **garde-fou** de `/ready` : si le manifeste du modèle déclarait une variable que la base de
   connaissance exclut (ici la variable de fuite), le service refuserait de se déclarer prêt. Le
   test de fumée de la CI interroge `/ready` : un tel modèle ne passerait pas la CI.

In [ ]:
import shutil
import tempfile

r = client_api.post("/score-batch?explain=true", json={"clients": charge["clients"][:1]}, headers=entete)
exemple_api = r.json()["resultats"][0]
e = exemple_api["explication"]
print(f"POST /score-batch?explain=true -> {r.status_code}, compte {exemple_api['client_id']} "
      f"(score {exemple_api['score_churn']}, priorité {exemple_api['priorite']})")
print("  Décision :", e["decision"])
for f in e["facteurs_hausse"] + e["facteurs_baisse"]:
    print(f"  {'▲' if f['contribution'] > 0 else '▼'} {f['texte']} (contribution {f['contribution']:+.2f})")
for a in e["avertissements"]:
    print("  ⚠", a)

direct = contributions(modele_churn, X_echantillon.iloc[:1], reference_expl).iloc[0]
ecart_api = max(abs(f["contribution"] - direct[f["variable"]]) for f in e["facteurs_hausse"] + e["facteurs_baisse"])
print(f"Écart maximal avec les contributions calculées directement : {ecart_api:.1e}")

with tempfile.TemporaryDirectory() as dossier:
    for fichier in ["model.joblib", "model_clv.joblib", "scoring_manifest.json", "explication_reference.json"]:
        shutil.copy(MODEL_DIR / fichier, dossier)
    (Path(dossier) / "model_manifest.json").write_text(
        json.dumps({"features": FEATURES_V2 + ["sante_compte_fin_periode"]}), encoding="utf-8")
    api.load_artifacts(Path(dossier))
    r_garde_fou = client_api.get("/ready")
api.load_artifacts(MODEL_DIR)
print(f"GET /ready avec un modèle déclarant la variable de fuite -> {r_garde_fou.status_code} : {r_garde_fou.json()['detail']}")
print(f"GET /ready avec le modèle v2 -> {client_api.get('/ready').status_code}")
assert r.status_code == 200 and "explication" not in reponse.json()["resultats"][0]
assert ecart_api < 1e-3 and r_garde_fou.status_code == 503 and client_api.get("/ready").status_code == 200

### 📓 Journal de bord — §10 Implémentation [C6]

- **Sérialisation** : pipelines complets (préparation + modèle) enregistrés avec `joblib` ;
  rechargement vérifié (prédictions identiques).
- **Règle de décision figée** dans `scoring_manifest.json` (seuil D9, capacité D10, catalogue
  D11), jamais recalculée au scoring.
- **Exemple d'usage** : cycle mensuel sur 1 000 comptes → 150 priorités Hautes ordonnées par
  perte attendue, export CRM réduit à 5 colonnes, conformité D3 vérifiée.
- **Chaque priorité est expliquée** (§10.2.1) : trois facteurs de hausse et trois de baisse en
  langage métier, trace de la règle D9/D10/D14, avertissements sur les valeurs imputées (36 % des
  comptes). Décomposition exacte vérifiée ; même explication servie par l'API
  (`?explain=true`), dont `/ready` refuse un modèle contenant une variable exclue.
- **Même chaîne à l'entraînement et en production** : un export brut est préparé par les mêmes
  fonctions (`clean_silver`, `construire_features_v2`) ; plus d'écart possible. L'API applique la
  même neutralisation des retards impossibles (v2.1).
- **Défaut de la v1 corrigé** : l'imputation fait partie du pipeline ; une valeur manquante ne
  fait plus échouer le scoring.
- **API testée de bout en bout** : scores identiques à l'appel direct, robustesse aux manquants
  et aux catégories inconnues, refus sans clé (401).
- **Mise en service** : pointer `MODEL_DIR` de l'API (et de l'image Docker) vers `data/model_v2`.

## 11. Architecture cible et contraintes [C7]

### 11.1 Architecture cible : scoring batch mensuel intégré au CRM

```
 CRM / facturation / support                         Équipe Customer Success
 (export mensuel des comptes)                          (vue CRM : priorité + action)
          │                                                     ▲
          ▼                                                     │ 5 colonnes
 ┌─────────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──┴───────────────┐
 │ 1. Ingestion     │──▶│ 2. Portique   │──▶│ 3. Silver +   │──▶│ 4. Scoring batch  │
 │ Bronze (brut,    │   │ RGPD (GO /    │   │ features v2   │   │ churn + CLV +     │
 │ horodaté, hash)  │   │ NO-GO)        │   │ (src/)        │   │ règle D9/D10/D14  │
 └─────────────────┘   └──────────────┘   └──────────────┘   └──┬───────────────┘
                                                                  │ métriques
          ┌───────────────────────────────────────────────────────┘
          ▼
 ┌─────────────────────────┐   alerte PSI > 0,25    ┌──────────────────────────────┐
 │ 5. Surveillance          │──────────────────────▶│ 6. Ré-entraînement            │
 │ Prometheus + Grafana :   │   rappel < 75 % sur    │ dvc repro → comparaison MLflow │
 │ dérive (PSI), API, volume│   cycle labellisé      │ → promotion si critères OK     │
 └─────────────────────────┘                         └──────────────────────────────┘
   Versioning transverse : Git (code) · DVC + DagsHub (données, modèles) · MLflow (expériences)
```

### 11.2 Correspondance avec ce qui est déjà implémenté

| Brique cible | Implémentation actuelle | Statut |
|---|---|---|
| Ingestion, RGPD, Silver, features | `src/bronze.py`, `notebooks/00`, `src/silver.py`, `src/features.py` | ✅ |
| Modèles churn et CLV (briques 1 et 2) | `data/model_v2/model.joblib`, `model_clv.joblib` | ✅ |
| Système de décision (brique 3 : seuil D9, capacité D10, priorité D14, actions D11) | `src/scoring.py`, `data/model_v2/scoring_manifest.json` | ✅ |
| Exposition | API FastAPI en conteneur Docker (clé API, limite de débit, taille de requête) | ✅ |
| Surveillance | Exporteur de dérive + Prometheus + Grafana (`compose.yaml`), alerte PSI | ✅ (sans routage d'alerte) |
| Versioning | Git + GitHub, DVC + DagsHub, MLflow sur DagsHub | ✅ |
| Intégration continue | GitHub Actions : tests, `dvc pull`, build Docker, test de santé | ✅ |
| Orchestration du batch mensuel | — (scoring lancé à la demande) | ⚠️ à ajouter (planificateur type cron / Airflow) |
| Intégration CRM réelle | — (export Parquet d'exemple) | ⚠️ connecteur à développer |

### 11.3 Contraintes

| Contrainte | Exigence | Réponse |
|---|---|---|
| **Volumétrie** | ~5 000 comptes par mois | Scoring en moins d'une seconde (mesuré ci-dessous) : aucune infrastructure distribuée nécessaire |
| **Latence** | Non critique (batch mensuel) | Batch ; l'API sert aussi un scoring à la demande depuis le CRM |
| **Poids des modèles** | Image Docker légère, démarrage rapide | Modèle CLV v2 ≈ 1,5 Mo contre 62 Mo en v1 (mesuré ci-dessous) |
| **Sécurité** | Accès restreint | Clé API, limite de débit, taille de requête bornée, conteneur non-root ; TLS à ajouter derrière un proxy |
| **RGPD** | Minimisation, pas de décision automatisée | Export limité à 5 colonnes, recommandations non exécutoires (D3), clé de pseudonymisation jamais publiée |
| **Explicabilité** | Les CSM doivent comprendre le score | Régression logistique : facteurs de risque lisibles (§9.6) |
| **Reproductibilité** | Rejouer un résultat passé | DVC (données), Git (code), MLflow (runs), graine fixe |

📘 **Explication — mesurer les contraintes plutôt que les affirmer**

La cellule mesure deux chiffres cités dans le tableau des contraintes :

- le **temps de scoring** des 5 000 comptes (churn + CLV + priorisation), en répétant la mesure
  pour obtenir une valeur stable ;
- la **taille sur disque** des modèles v1 et v2.

In [ ]:
import time

X_tout = gold_v2[FEATURES_V2]
durees = []
for _ in range(5):
    debut = time.perf_counter()
    assigner_priorites(scorer_batch(X_tout, gold_v2["client_id"], modele_churn, modele_clv), SEUIL_D9, CAPACITE_CSM_D10)
    durees.append(time.perf_counter() - debut)
print(f"Scoring de {len(X_tout)} comptes : {np.median(durees):.2f} s (médiane de 5 mesures)")

tailles = pd.DataFrame([{"version": v, "fichier": f, "taille_mo": round((d / f).stat().st_size / 1e6, 2)}
                        for v, d in [("v1", MODEL_V1_DIR), ("v2", MODEL_DIR)]
                        for f in ["model.joblib", "model_clv.joblib"]])
display(tailles)

### 11.4 Scénarios d'architecture et contraintes économiques

L'architecture de §11.1 n'est pas la seule possible. Trois scénarios répondent au besoin ; on
compare ce qui les distingue économiquement, à partir des mesures de ce notebook.

📘 **Explication — chiffrer ce qui peut l'être**

La cellule mesure l'énergie d'un cycle de scoring complet (5 000 comptes, avec la fonction
`mesurer_energie` de §8.6), compare le temps machine d'un batch mensuel à celui d'un service
toujours actif, et met ces chiffres en regard du coût humain de l'action CS (capacité D10 × coût
d'une relance H01). Les tarifs d'hébergement et de licence ne sont pas connus dans cet exercice :
ils ne sont pas inventés ; la cellule donne les quantités auxquelles les appliquer.

In [ ]:
def euros(montant):
    return f"{montant:,.0f}".replace(",", " ")


_, duree_cycle, mwh_cycle, mg_cycle = mesurer_energie(
    lambda: assigner_priorites(scorer_batch(X_tout, gold_v2["client_id"], modele_churn, modele_clv),
                               SEUIL_D9, CAPACITE_CSM_D10))
secondes_batch_an = 12 * float(np.median(durees))
heures_service_an = 365 * 24
cout_cs_mois = [CAPACITE_CSM_D10 * cout for cout in COUTS_FP_EUR]

print(f"Un cycle de scoring ({len(X_tout)} comptes) : {duree_cycle:.2f} s, {mwh_cycle:.2f} mWh, "
      f"{mg_cycle:.2f} mg de CO₂e")
print(f"Scénario A, batch mensuel : {secondes_batch_an:.1f} s de calcul par an")
print(f"Scénario B, service toujours actif : {euros(heures_service_an)} h de fonctionnement par an, "
      f"soit {euros(heures_service_an * 3600 / secondes_batch_an)} fois le temps machine du batch")
print(f"Coût humain de l'action CS, identique dans les trois scénarios (D10 x H01) : "
      f"{euros(cout_cs_mois[0])} à {euros(cout_cs_mois[1])} € par mois")

| Critère | A. Batch mensuel conteneurisé (retenu, §11.1) | B. Scoring en continu (service toujours actif, score à chaque événement d'usage) | C. Solution sur étagère (plateforme Customer Success, §8.5) |
|---|---|---|---|
| Infrastructure | Un conteneur lancé une fois par mois ; stockage objet | Service disponible en permanence, flux d'événements depuis le produit, base de données opérationnelle | Aucune à opérer ; connecteurs vers le CRM et les données d'usage |
| Coût de calcul | Négligeable (quelques secondes par an, mesuré) | Des dizaines de millions de fois le temps machine du batch, pour le même nombre de comptes | Inclus dans la licence |
| Coût récurrent dominant | Maintenance : suivi de la dérive, réentraînement sur déclencheur (§13.2) | Exploitation en continu (supervision, astreinte) en plus de la maintenance | Licence, souvent par utilisateur ou par compte suivi |
| Apport métier | Aligné sur le cycle mensuel et la capacité CS (D10) | Faible : les CSM agissent par cycle, un score plus frais ne crée pas de capacité | Dépend de l'outil : fuite et système de décision à vérifier (§8.5) |
| Risques | Score daté jusqu'au cycle suivant | Complexité, surface d'attaque, coût fixe sans gain | Dépendance à l'éditeur, données hébergées chez un tiers |

**Lecture.**

- Le coût de calcul ne départage pas les scénarios : il est infime dans le scénario A. Le coût
  dominant est **humain** (l'action des CSM), le même dans les trois scénarios. Une
  infrastructure plus complexe (B) ajoute un coût fixe sans augmenter la capacité d'action.
- Le scénario A est retenu. Le scénario B ne se justifierait que si les CSM devaient réagir dans
  la journée à un signal : un changement du besoin, à confirmer avec le commanditaire.
- **À porter à la connaissance des acteurs** : la DSI chiffre l'hébergement du scénario A (tarif
  du conteneur et du stockage), les achats chiffrent les licences du scénario C, la direction CS
  confirme que le rythme mensuel suffit.

## 12. Mesure de performance et impacts [C8]

### 12.1 Performance technique du modèle de churn

📘 **Explication — tableau de synthèse**

On rassemble en un tableau les métriques du modèle retenu : en validation croisée (train) et
sur le test. Des valeurs proches entre les deux indiquent que le modèle **généralise** (il ne
s'est pas contenté de mémoriser le jeu d'entraînement). On rappelle la baseline métier et le
critère d'acceptation D8.

In [ ]:
synthese_churn = pd.DataFrame({
    "validation croisée (train)": {
        "PR-AUC": grille.best_score_,
        "ROC-AUC": modeles_ml.loc[MODELE_RETENU, "roc_auc"],
        "rappel au seuil D9": recall_score(y_train, signales_oof),
        "précision au seuil D9": precision_score(y_train, signales_oof)},
    "test": {
        "PR-AUC": metriques_test["pr_auc"], "ROC-AUC": metriques_test["roc_auc"],
        "rappel au seuil D9": metriques_test["rappel_seuil_D9"],
        "précision au seuil D9": metriques_test["precision_seuil_D9"]},
    "baseline métier D8 (test)": {
        "PR-AUC": metriques_test["pr_auc_baseline_metier_D8"],
        "ROC-AUC": metriques_test["roc_auc_baseline_metier_D8"],
        "rappel au seuil D9": np.nan, "précision au seuil D9": np.nan},
})
display(synthese_churn.round(3))
print(f"Critère D8 sur le test : gain de PR-AUC = {metriques_test['gain_pr_auc_vs_D8']:+.3f} "
      f"(exigé : +{GAIN_MIN_D8}) -> {'respecté' if metriques_test['critere_D8_respecte'] else 'NON respecté'}")

### 12.2 Performance de la régression CLV (cible secondaire)

📘 **Explication — lire l'erreur en euros**

Les prédictions sont retransformées en euros, puis comparées aux valeurs réelles du test :

- **MAE** (erreur absolue moyenne) : l'erreur moyenne en euros ;
- **RMSE** : racine de l'erreur quadratique moyenne ; elle pénalise fortement les grosses
  erreurs, donc elle est dominée par les très gros comptes ;
- **R²** en euros et en échelle logarithmique ;
- **erreur relative médiane** : pour un compte « typique », de combien de pourcents se trompe-t-on ?
  C'est l'indicateur le plus parlant pour un CSM.

On détaille l'erreur relative par taille d'entreprise : un modèle peut être bon en moyenne et
moins fiable pour un segment.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

clv_reelle = test["valeur_vie_client_eur"]
clv_predite = np.expm1(modele_clv.predict(X_test))
clv_baseline = np.full_like(clv_predite, np.expm1(y_clv_train.median()))
erreur_relative = (clv_predite - clv_reelle).abs() / clv_reelle

metriques_clv = {
    "modele_retenu": MODELE_CLV_RETENU,
    "mae_eur": mean_absolute_error(clv_reelle, clv_predite),
    "rmse_eur": mean_squared_error(clv_reelle, clv_predite) ** 0.5,
    "r2_eur": r2_score(clv_reelle, clv_predite),
    "r2_log": r2_score(np.log1p(clv_reelle), np.log1p(clv_predite)),
    "erreur_relative_mediane": float(erreur_relative.median()),
    "baseline_mae_eur": mean_absolute_error(clv_reelle, clv_baseline),
    "baseline_r2_eur": r2_score(clv_reelle, clv_baseline),
}
display(pd.Series(metriques_clv).to_frame("test").T)
par_taille = (pd.DataFrame({"taille": test["taille_entreprise"], "erreur_relative": erreur_relative, "clv": clv_reelle})
              .groupby("taille").agg(comptes=("clv", "size"), clv_mediane=("clv", "median"),
                                     erreur_relative_mediane=("erreur_relative", "median")))
display(par_taille.round(2))

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(clv_reelle, clv_predite, s=8, alpha=0.4)
bornes = [clv_reelle.min(), clv_reelle.max()]
ax.plot(bornes, bornes, "--", color="gray")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("CLV réelle (€, échelle log)")
ax.set_ylabel("CLV prédite (€, échelle log)")
ax.set_title(f"CLV : réel contre prédit (test) — R² log = {metriques_clv['r2_log']:.2f}")
sauver_figure(fig, "14_clv_reel_vs_predit")
plt.show()
write_manifest(MODEL_DIR / "metrics_clv.json", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in metriques_clv.items()})

**Lecture.** Le modèle explique l'essentiel de la variance en échelle logarithmique, et son
erreur en euros est bien inférieure à celle de la baseline. En euros, le R² et le RMSE restent
pénalisés par quelques comptes à plusieurs centaines de milliers d'euros. L'erreur relative
médiane donne l'ordre de grandeur utile au métier : la CLV estimée sert à **ordonner** les
comptes par enjeu, pas à prévoir un chiffre d'affaires au euro près.

### 12.3 Impact métier : combien de comptes et d'euros la priorisation peut-elle préserver ?

📘 **Explication — estimer l'impact, sans prétendre l'avoir mesuré**

Le modèle n'a pas été déployé : on ne peut pas **mesurer** des comptes sauvés. On peut en
revanche **estimer** l'intérêt de la liste produite sur le cycle simulé (les 1 000 comptes du
test), en comparant quatre façons de choisir les 150 comptes à appeler (D10) :

1. **le modèle** (priorité Haute : signalés au seuil D9, triés par perte attendue) ;
2. **au hasard** (espérance mathématique) ;
3. **les 150 plus gros revenus mensuels** (règle naïve « on protège les gros comptes ») ;
4. **la baseline métier D8** (les 150 scores de règle les plus élevés).

Pour chaque méthode : combien de comptes contactés allaient réellement partir, et quelle CLV
réelle ces comptes représentent (l'enjeu économique couvert).

On traduit ensuite en comptes sauvés avec un **taux de succès des relances** inconnu (hypothèse
**H05**, testée à 10, 20 et 30 %), diminué du coût des 150 relances (H01). Ces montants sont des
ordres de grandeur ; la mesure réelle passera par le groupe témoin de D12 (§13).

In [ ]:
cycle_reel = cycle.merge(test[["client_id", "churn", "valeur_vie_client_eur", "revenu_mensuel_recurrent_eur"]], on="client_id")
cycle_reel["score_regle"] = score_regle_metier(X_test, X_train["nb_integrations"].median()).values
selections = {
    "Modèle (priorité Haute)": cycle_reel[cycle_reel["priorite"] == "Haute"],
    "150 plus gros MRR": cycle_reel.nlargest(CAPACITE_CSM_D10, "revenu_mensuel_recurrent_eur"),
    "Baseline métier D8": cycle_reel.nlargest(CAPACITE_CSM_D10, "score_regle"),
}
impact = []
for nom, sel in selections.items():
    impact.append({"méthode": nom, "comptes_contactés": len(sel), "churners_contactés": int(sel["churn"].sum()),
                   "clv_en_jeu_eur": float((sel["churn"] * sel["valeur_vie_client_eur"]).sum())})
impact.append({"méthode": "Au hasard (espérance)", "comptes_contactés": CAPACITE_CSM_D10,
               "churners_contactés": round(CAPACITE_CSM_D10 * cycle_reel["churn"].mean(), 1),
               "clv_en_jeu_eur": float(CAPACITE_CSM_D10 * (cycle_reel["churn"] * cycle_reel["valeur_vie_client_eur"]).mean())})
impact = pd.DataFrame(impact).set_index("méthode")
impact["précision"] = impact["churners_contactés"] / impact["comptes_contactés"]
display(impact.style.format({"clv_en_jeu_eur": "{:,.0f} €", "précision": "{:.0%}", "churners_contactés": "{:.0f}"}))

haute = selections["Modèle (priorité Haute)"]
sensibilite = pd.DataFrame([
    {"taux de succès H05": f"{tau:.0%}", "coût relance H01": f"{c} €",
     "comptes sauvés": haute["churn"].sum() * tau,
     "CLV préservée (€)": (haute["churn"] * haute["valeur_vie_client_eur"]).sum() * tau,
     "coût des 150 relances (€)": CAPACITE_CSM_D10 * c}
    for tau in (0.10, 0.20, 0.30) for c in COUTS_FP_EUR])
sensibilite["gain net estimé (€)"] = sensibilite["CLV préservée (€)"] - sensibilite["coût des 150 relances (€)"]
display(sensibilite.style.format({"comptes sauvés": "{:.1f}", "CLV préservée (€)": "{:,.0f}",
                                  "coût des 150 relances (€)": "{:,.0f}", "gain net estimé (€)": "{:,.0f}"}))

**Lecture.** Sur 150 appels, le modèle cible environ 2,4 fois plus de comptes qui allaient
réellement partir que le hasard, 3 fois plus que la règle des plus gros revenus, et davantage
que la baseline métier. Il couvre aussi la plus grande CLV en jeu : la règle des gros comptes
s'en approche en euros, mais en appelant surtout des clients qui ne comptaient pas partir, et
la baseline métier trouve des comptes à risque mais de faible valeur. Même avec l'hypothèse de succès la plus prudente (10 %) et le coût de
relance le plus élevé, le gain net estimé reste positif. **Limites** : ces chiffres portent sur
un cycle simulé de 1 000 comptes, avec un taux de succès supposé ; seul le protocole de groupe
témoin (D12–D13) permettra de mesurer l'impact réel.

### 12.4 Recette de la priorisation (D15)

📘 **Explication — le critère de recette D15**

D15 vérifie que la priorisation **ordonne** correctement les comptes par enjeu réel. On calcule,
pour chaque compte du test, une **perte réelle** = churn réel × CLV réelle (0 pour un compte resté
client), jamais transmise au modèle, puis trois contrôles :

- **(a)** exactitude arithmétique : perte attendue = score × CLV estimée, pour chaque compte ;
- **(b)** cohérence du classement : **corrélation de rang de Spearman** entre perte attendue et
  perte réelle. Spearman compare les *rangs* et non les valeurs : il répond à « les comptes
  classés en tête par le modèle sont-ils ceux qui représentent la plus grosse perte réelle ? ».
  Seuil D15 : ≥ 0,7 ;
- **(c)** contrôle individuel : aucun faux négatif (compte parti sans être signalé) ne doit se
  trouver parmi les plus grosses pertes réelles. D15 parle du « quartile supérieur du jeu de
  recette » ; sur un jeu où 72 % des pertes réelles valent 0, j'interprète ce quartile parmi les
  **pertes réelles non nulles**, sinon presque tous les comptes partis y figureraient.

In [ ]:
from scipy.stats import spearmanr

cycle_reel["perte_reelle_eur"] = cycle_reel["churn"] * cycle_reel["valeur_vie_client_eur"]
ecart_arithmetique = (cycle_reel["score_churn"] * cycle_reel["valeur_vie_estimee_eur"] - cycle_reel["perte_attendue_eur"]).abs().max()
rho = spearmanr(cycle_reel["perte_attendue_eur"], cycle_reel["perte_reelle_eur"]).statistic
q3_pertes = cycle_reel.loc[cycle_reel["perte_reelle_eur"] > 0, "perte_reelle_eur"].quantile(0.75)
fn_critiques = cycle_reel[(cycle_reel["churn"] == 1) & (~cycle_reel["signale_D9"]) & (cycle_reel["perte_reelle_eur"] >= q3_pertes)]

# Le score est arrondi à 3 décimales et les montants à l'euro : l'écart toléré par l'arrondi
# vaut au plus 0,0005 x CLV estimée + 1 €. Tout écart supérieur signalerait une erreur de calcul.
ecarts = (cycle_reel["score_churn"] * cycle_reel["valeur_vie_estimee_eur"] - cycle_reel["perte_attendue_eur"]).abs()
borne_arrondi = 0.0005 * cycle_reel["valeur_vie_estimee_eur"] + 1
print(f"(a) écart maximal : {ecart_arithmetique:.1f} € ; comptes dont l'écart dépasse la borne d'arrondi : "
      f"{int((ecarts > borne_arrondi).sum())} -> {'respecté' if (ecarts <= borne_arrondi).all() else 'NON respecté'}")
print(f"(b) Spearman(perte attendue, perte réelle) = {rho:.3f} -> {'respecté' if rho >= 0.7 else 'NON respecté'} (seuil 0,7)")
print(f"(c) faux négatifs parmi le quartile supérieur des pertes réelles (≥ {q3_pertes:,.0f} €) : {len(fn_critiques)}")
if len(fn_critiques):
    display(fn_critiques[["client_id", "score_churn", "valeur_vie_estimee_eur", "valeur_vie_client_eur", "perte_reelle_eur"]]
            .sort_values("perte_reelle_eur", ascending=False).head(10))

# Diagnostics complémentaires pour comprendre le résultat de (b)
partis = cycle_reel["churn"] == 1
rho_partis = spearmanr(cycle_reel.loc[partis, "perte_attendue_eur"], cycle_reel.loc[partis, "perte_reelle_eur"]).statistic
rho_clv = spearmanr(cycle_reel["valeur_vie_estimee_eur"], cycle_reel["valeur_vie_client_eur"]).statistic
perte_totale = cycle_reel["perte_reelle_eur"].sum()
capture_haute = cycle_reel.loc[cycle_reel["priorite"] == "Haute", "perte_reelle_eur"].sum() / perte_totale
capture_oracle = cycle_reel["perte_reelle_eur"].nlargest(CAPACITE_CSM_D10).sum() / perte_totale
print("\nDiagnostics complémentaires :")
print(f"  part des pertes réelles nulles (comptes restés clients) : {(cycle_reel['perte_reelle_eur'] == 0).mean():.0%}")
print(f"  Spearman parmi les seuls comptes partis                 : {rho_partis:.3f}")
print(f"  Spearman CLV estimée / CLV réelle (tous les comptes)    : {rho_clv:.3f}")
print(f"  part de la perte réelle totale captée par les 150 Hautes : {capture_haute:.0%} "
      f"(maximum atteignable avec 150 comptes, en connaissant l'avenir : {capture_oracle:.0%})")

**Lecture — la recette D15 n'est pas entièrement satisfaite, et c'est une information utile.**

- **(a) respecté** : la perte attendue est exactement score × CLV ; le seul écart vient de
  l'arrondi du score affiché.
- **(b) non respecté tel que formulé** (Spearman = 0,29 contre 0,7 exigé). Le diagnostic montre
  pourquoi : 72 % des comptes ont une perte réelle nulle (ils sont restés), alors que leur perte
  *attendue* est positive, puisqu'une probabilité n'est jamais nulle. Sur une cible aussi
  dominée par des zéros, la corrélation de rang globale est mécaniquement faible, même pour un
  bon classement. Les autres indicateurs le confirment : parmi les comptes réellement partis,
  le classement par perte attendue suit bien la perte réelle (Spearman ≈ 0,85), la CLV estimée
  ordonne correctement les comptes (≈ 0,94), et les 150 priorités Hautes captent environ 82 %
  de la perte réelle totale, pour un maximum théorique de 98 %.
- **(c) non respecté** : 14 comptes partis, à forte perte réelle, n'ont pas été signalés parce
  que leur probabilité était sous le seuil D9. C'est le coût de la règle D14 déjà observé en
  §10.2 : un compte de très grande valeur mais de risque modéré n'entre jamais dans la liste.

**Ce que j'en conclus, sans modifier les critères après coup** : le critère (b) a été fixé comme
un ordre de grandeur de travail, pour un petit jeu de recette ; appliqué à un portefeuille
complet, il mesure surtout la proportion de zéros. Je propose au métier de le remplacer par la
**part de la perte réelle captée par la priorité Haute** (82 % ici), qui répond directement à
la question « les 150 appels portent-ils sur les plus grosses pertes ? ». Pour (c), je propose
un **filet de sécurité** : faire passer en priorité Moyenne les comptes sous le seuil dont la
perte attendue dépasse celle du 150ᵉ compte Haut. Ces deux évolutions touchent aux décisions
D14 et D15 : elles relèvent d'un arbitrage métier, pas d'un ajustement technique unilatéral.

#### 12.4.1 Les faux négatifs critiques, expliqués

📘 **Explication — expliquer les erreurs, pas seulement les réussites**

Le critère D15(c) a relevé des comptes partis, à forte perte réelle, que le système n'a **pas
signalés** : leur probabilité était sous le seuil D9. Les explications locales permettent de
demander **pourquoi** le modèle les a jugés peu risqués. On compare, variable par variable, la
contribution moyenne chez ces faux négatifs critiques et chez les comptes partis **correctement
signalés** : les variables qui « rassurent » le modèle à tort apparaissent directement. C'est
l'usage le plus utile de l'explicabilité pour améliorer un système : localiser l'échec.

In [ ]:
contrib_par_client = contrib_cycle.set_axis(cycle["client_id"].to_numpy())
partis_signales = cycle_reel.loc[(cycle_reel["churn"] == 1) & cycle_reel["signale_D9"], "client_id"]
comparaison_fn = pd.DataFrame({
    f"faux négatifs critiques (n={len(fn_critiques)})": contrib_par_client.loc[fn_critiques["client_id"]].mean(),
    f"partis signalés (n={len(partis_signales)})": contrib_par_client.loc[partis_signales].mean(),
})
comparaison_fn["écart"] = comparaison_fn.iloc[:, 0] - comparaison_fn.iloc[:, 1]
comparaison_fn = comparaison_fn.sort_values("écart")
print("Contribution moyenne au logit (négatif = diminue le risque) :")
display(comparaison_fn.head(6).round(3))

variables_ecart = [v for v in comparaison_fn.index[:4] if reference_expl["types"][v] == "numerique"]
test_par_client = test.set_index("client_id")
display(pd.DataFrame({
    "médiane faux négatifs critiques": test_par_client.loc[fn_critiques["client_id"], variables_ecart].median(),
    "médiane partis signalés": test_par_client.loc[partis_signales, variables_ecart].median(),
}).round(1))

plus_grosse_perte = fn_critiques.sort_values("perte_reelle_eur", ascending=False)["client_id"].iloc[0]
afficher_explication(int(np.flatnonzero(cycle["client_id"].to_numpy() == plus_grosse_perte)[0]))

**Lecture — ces comptes partent sans prévenir.** Les 14 faux négatifs critiques ne ressemblent
pas aux comptes qui partent habituellement : dernière connexion médiane il y a 1,5 jour (contre 6
pour les comptes partis signalés), 1,5 intégration en médiane (contre aucune), 9,5 mois
d'ancienneté (contre 6). La récence de connexion suffit à les « rassurer » : 0,80 point de logit
de moins, en moyenne, que les comptes partis signalés.

Le compte à la plus forte perte réelle (403 k€) est une exception, et elle est instructive : un
compte d'**un mois**, sans intégration, dont le score (0,253) frôle le seuil (0,283). Ses retards
de paiement faisaient partie des valeurs impossibles neutralisées au §6.5. La v2.0 le signalait ;
la v2.1, qui ne s'appuie plus sur ces valeurs, le place juste sous le seuil. C'est le coût
visible de la révision, assumé : le nombre total de faux négatifs critiques baisse (16 en v2.0,
14 ici).

Pour les autres, rien ne distingue ces comptes d'un compte en bonne santé sur les variables
disponibles : **ce n'est pas une erreur de calcul, c'est une limite des données**. Leur départ
tient vraisemblablement à des facteurs absents du jeu (budget, concurrent, renégociation). Trois
conséquences :

1. aucun réglage du modèle ne les rattrapera sans nouvelles données : c'est la recommandation
   d'enrichissement des données (§14) ;
2. pour ces comptes, seule leur valeur peut justifier un contact : cela appuie la proposition
   de **filet de sécurité par la valeur** (§12.4) ;
3. la récence de connexion, qui explique l'essentiel de ces erreurs, est aussi la variable dont
   la date de calcul reste à confirmer auprès du propriétaire des données (§14) : raison de plus
   de le faire.

### 12.5 Audit d'équité par segment

📘 **Explication — le modèle sert-il tous les segments de la même façon ?**

Pour chaque modalité de la taille d'entreprise, du secteur et du plan, on mesure au seuil D9 :

- le **rappel** : part des comptes qui partent effectivement détectés dans ce segment. Un rappel
  plus faible signifie que les clients de ce segment reçoivent moins souvent l'aide proposée ;
- la **précision** et la **part de comptes signalés**.

Les effectifs par segment sont petits (parfois moins de 30 comptes partis) : on calcule un
**intervalle de confiance de Wilson** à 95 % pour le rappel. Si les intervalles se recouvrent,
l'écart observé peut être dû au hasard ; on ne conclut pas à un biais sur un écart non
significatif, mais on le documente et on le surveille.

In [ ]:
def intervalle_wilson(succes, n, z=1.96):
    """Intervalle de confiance à 95 % d'une proportion (méthode de Wilson, fiable sur petits effectifs)."""
    if n == 0:
        return (np.nan, np.nan)
    p = succes / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    demi = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return (centre - demi, centre + demi)


audit = []
signale_test = pd.Series(proba_test >= SEUIL_D9, index=test.index)
for segment in ["taille_entreprise", "secteur", "plan"]:
    for modalite, idx in test.groupby(segment).groups.items():
        y_seg, s_seg = test.loc[idx, "churn"], signale_test.loc[idx]
        vp = int((y_seg & s_seg).sum())
        bas, haut = intervalle_wilson(vp, int(y_seg.sum()))
        audit.append({"segment": segment, "modalité": modalite, "comptes": len(idx), "churners": int(y_seg.sum()),
                      "rappel": vp / max(int(y_seg.sum()), 1), "IC95_bas": bas, "IC95_haut": haut,
                      "précision": vp / max(int(s_seg.sum()), 1), "part_signalés": s_seg.mean()})
audit = pd.DataFrame(audit)
display(audit.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
etiquettes = audit["segment"].str[:6] + " · " + audit["modalité"].astype(str)
ax.errorbar(audit["rappel"], range(len(audit)), xerr=[audit["rappel"] - audit["IC95_bas"], audit["IC95_haut"] - audit["rappel"]],
            fmt="o", capsize=3)
ax.axvline(metriques_test["rappel_seuil_D9"], color=COULEURS["churn"], linestyle="--", label="rappel global (test)")
ax.set_yticks(range(len(audit)), etiquettes, fontsize=8)
ax.set_xlabel("Rappel au seuil D9 (intervalle de confiance à 95 %)")
ax.set_title("Audit d'équité : rappel par segment")
ax.legend()
sauver_figure(fig, "15_audit_equite")
plt.show()

**Lecture.** Un seul segment a un intervalle de confiance qui ne recouvre pas le rappel global
(82,5 %) : le secteur **Finance** (rappel 70 %, intervalle de 55 % à 81 %), et de peu. Avec
16 intervalles à 95 % calculés, on s'attend à environ un écart de ce type par le seul hasard
(5 % × 16 ≈ 0,8) : ce n'est pas une preuve de biais, mais c'est le premier **point de
surveillance**. Les autres rappels faibles (plan Business ≈ 75 %, PME ≈ 77 %) restent compatibles
avec le hasard compte tenu des effectifs (15 à 125 comptes partis par segment). Si l'écart de la
Finance persiste sur plusieurs cycles, il faudra en chercher la cause (signaux d'usage moins
lisibles pour ces clients ?) plutôt que d'ajuster un seuil par segment, ce qui introduirait un
traitement différencié explicite.

## 13. Amélioration continue : ré-entraînement, suivi, versioning [C9]

Un modèle se dégrade avec le temps : les usages évoluent, l'offre change, la clientèle se
transforme. Cette section décrit comment détecter une dégradation, quand et comment
ré-entraîner, et comment tracer chaque version.

### 13.1 Détecter une dérive des données (PSI)

📘 **Explication — le Population Stability Index (PSI)**

Le PSI compare la distribution d'une variable entre une **référence** (les données
d'entraînement) et une **fenêtre courante** (le cycle du mois) :

1. on découpe la référence en 10 intervalles de même effectif (déciles) ;
2. on compte la part des comptes du cycle courant dans chaque intervalle ;
3. PSI = Σ (part courante − part de référence) × ln(part courante / part de référence).

Lecture usuelle : **< 0,10** stable, **0,10 à 0,25** à surveiller, **> 0,25** dérive importante
(seuil de l'alerte Prometheus du projet, `drift_alert_rules.yml`).

On calcule le PSI dans deux situations :

- **test contre train** : les deux viennent du même découpage aléatoire, donc le PSI doit être
  proche de zéro. C'est un contrôle de bon fonctionnement, pas une démonstration de détection ;
- **un lot volontairement dérivé**, qui simule un trimestre où l'engagement baisse (connexions
  plus anciennes de 30 jours, une intégration de moins, CSAT en baisse d'un point, 40 % d'usage
  en moins). Le dispositif doit alors déclencher l'alerte sur les variables concernées.

La fonction `drift_table` (module `src/drift.py`) est celle qu'utilise l'exporteur Prometheus
en production.

In [ ]:
from src.drift import drift_table

reference = train[NUMERIQUES_V2]
lot_derive = test[NUMERIQUES_V2].copy()
lot_derive["derniere_connexion_jours"] = lot_derive["derniere_connexion_jours"] + 30
lot_derive["nb_integrations"] = (lot_derive["nb_integrations"] - 1).clip(lower=0)
lot_derive["csat"] = (lot_derive["csat"] - 1).clip(lower=1)
lot_derive["heures_usage_30j"] = lot_derive["heures_usage_30j"] * 0.6

psi_test = drift_table(reference, test[NUMERIQUES_V2], NUMERIQUES_V2).set_index("feature")["psi"]
psi_derive = drift_table(reference, lot_derive, NUMERIQUES_V2).set_index("feature")["psi"]
tableau_psi = pd.DataFrame({"PSI test vs train": psi_test, "PSI lot dérivé vs train": psi_derive}).sort_values("PSI lot dérivé vs train")

fig, ax = plt.subplots(figsize=(8, 6))
positions = np.arange(len(tableau_psi))
ax.barh(positions - 0.2, tableau_psi["PSI test vs train"], height=0.4, label="test vs train", color=COULEURS["non_churn"])
ax.barh(positions + 0.2, tableau_psi["PSI lot dérivé vs train"], height=0.4, label="lot dérivé vs train", color=COULEURS["churn"])
ax.axvline(PSI_SEUIL_ALERTE, color="black", linestyle="--", label=f"seuil d'alerte {PSI_SEUIL_ALERTE}")
ax.set_yticks(positions, tableau_psi.index, fontsize=8)
ax.set_xlabel("PSI")
ax.set_title("Dérive des variables numériques")
ax.legend(loc="lower right")
sauver_figure(fig, "16_derive_psi")
plt.show()
print(f"PSI maximal test vs train : {psi_test.max():.3f} -> aucune alerte attendue")
print("Variables en alerte sur le lot dérivé :", list(psi_derive[psi_derive > PSI_SEUIL_ALERTE].index))

proba_derive = modele_churn.predict_proba(test[FEATURES_V2].assign(**lot_derive))[:, 1]
print(f"Part de comptes signalés : {np.mean(proba_test >= SEUIL_D9):.0%} sur le test, "
      f"{np.mean(proba_derive >= SEUIL_D9):.0%} sur le lot dérivé (dérive de la sortie du modèle)")

**Lecture.** Sans dérive, tous les PSI restent sous 0,05 : aucune fausse alerte. Sur le lot
simulé, **deux des quatre variables modifiées** (récence de connexion et CSAT) dépassent le seuil
et déclencheraient l'alerte `ChurnSaasDriftPSIEleve`. Les deux autres (usage réduit de 40 %,
une intégration de moins) restent sous 0,25 : **un PSI variable par variable peut laisser passer
une dérive modérée**. D'où l'intérêt d'un second indicateur, la **dérive de la sortie du
modèle** : la part de comptes signalés passe d'environ 37 % à 75 %. Un tel afflux dépasserait
largement la capacité CS et justifierait une analyse avant toute action. Un PSI élevé dit que **les données ont changé**, pas
que **le modèle se trompe** : la décision de ré-entraîner s'appuie aussi sur la performance
mesurée dès que les issues réelles sont connues.

#### 13.1.1 Quelles variables font monter les scores ?

📘 **Explication — relier la dérive des données à la dérive des scores**

Le PSI mesure le changement de chaque variable, sans dire s'il **compte pour le modèle** ; la part
de comptes signalés dit que les scores ont bougé, sans dire **pourquoi**. Les contributions font
le lien : l'écart de logit moyen entre deux lots se décompose **exactement** en une somme
d'écarts de contributions moyennes, une par variable. On l'applique au lot dérivé du §13.1,
comparé au jeu de test d'origine.

In [ ]:
from src.explain import attribuer_derive

X_derive = test[FEATURES_V2].assign(**lot_derive)
attribution = attribuer_derive(contrib_cycle, contributions(modele_churn, X_derive, reference_expl))
attribution["PSI"] = psi_derive.reindex(attribution.index)
attribution["part de l'écart"] = attribution["ecart"] / attribution["ecart"].sum()
ecart_logit = modele_churn.decision_function(X_derive).mean() - modele_churn.decision_function(X_test).mean()
display(attribution.head(6).round(3))
print(f"Écart de logit moyen entre les deux lots : {ecart_logit:.3f} ; somme des écarts de contributions : "
      f"{attribution['ecart'].sum():.3f}")

**Lecture.** La hausse des scores du lot dérivé (+1,93 point de logit en moyenne) s'explique à
**70 %** par la récence de connexion, à 18 % par le CSAT et à **10,5 % par le nombre
d'intégrations**, alors que le PSI de cette dernière (0,14) restait sous le seuil d'alerte. À
l'inverse, la baisse de 40 % de l'usage a un PSI proche du seuil (0,22) mais ne compte que pour
1,6 % de la hausse : le modèle s'appuie peu sur cette variable. **Le PSI dit ce qui a changé ;
l'attribution dit ce qui compte pour la décision.** En production, quand l'indicateur
« explosion du volume signalé » (§13.2) se déclenche, c'est cette table qu'on regarde en premier.

### 13.2 Politique de ré-entraînement

| Déclencheur | Seuil | Action |
|---|---|---|
| Dérive des données | PSI > 0,25 sur une variable du top 5 d'importance | Analyse de la variable ; ré-entraînement si la performance baisse |
| Baisse de performance | Rappel < 75 % au seuil D9 sur un cycle dont les issues sont connues | Ré-entraînement prioritaire |
| Explosion du volume signalé | Signalés > 2 × la valeur habituelle | Vérification des données en amont (défaut d'export ?) avant tout |
| Calendrier | Tous les trimestres (aligné sur D13) | Ré-entraînement de routine sur les données fraîches |
| Explication non conforme | Contrôle bloquant de la base de connaissance (variable exclue, une variable > 50 % de l'explication) | Refus de mise en service : notebook en échec (§15.3), `/ready` en 503 donc CI en échec (§10.4) |

**Procédure** (entièrement outillée dans le dépôt) :

1. nouvel export → `dvc add` des données brutes → `dvc repro` rejoue tout le pipeline ;
2. le nouvel entraînement est tracé dans **MLflow** (métriques, paramètres, `gold_sha256`) ;
3. **critères de promotion** du nouveau modèle : critère D8 respecté, rappel ≥ 80 % au seuil D9
   recalculé hors pli, PR-AUC au moins égale au modèle en service sur le même jeu de test,
   calibration vérifiée, **aucun contrôle bloquant de la base de connaissance** (§9.6.1) ;
4. mise en service : `MODEL_DIR` de l'API pointe vers le nouveau dossier ; **retour arrière** en
   une commande (`git checkout <commit précédent>` puis `dvc checkout`).

### 13.3 Traçabilité de cet entraînement dans MLflow

📘 **Explication — enregistrer le run dans MLflow**

MLflow garde l'historique des entraînements : paramètres, métriques et fichiers produits
(**artefacts**) de chaque run. On enregistre ici deux runs (churn et CLV) avec le tag
**`gold_sha256`**, l'empreinte du Gold v2 : à partir d'un run, on retrouve la version exacte des
données qui l'a produit.

- Si la variable d'environnement `MLFLOW_TRACKING_URI` pointe vers DagsHub, les runs y sont
  publiés (lien en page de garde) ; sinon, ils sont enregistrés localement dans
  `data/model_v2/mlflow.db`.
- Les expériences portent le même nom que celles de la v1 : dans l'interface MLflow, on compare
  directement les runs v1 et v2.
- L'appel est protégé (`try/except`) : un serveur MLflow indisponible ne doit pas empêcher le
  notebook de s'exécuter jusqu'au bout.

In [ ]:
from src.tracking import flatten_metrics, log_training_run, resolve_tracking_uri

uri_mlflow = resolve_tracking_uri(MODEL_DIR)
etiquettes_communes = {"gold_version": "v2.1", "notebook": "notebook_certifiant_churn_saas", "commit_git": commit_git()}
try:
    run_churn = log_training_run(
        run_name=f"churn_v2_{MODELE_RETENU}", experiment_name="churn_saas_classification",
        params={"modele": MODELE_RETENU, "C": C_RETENU, "n_features": len(FEATURES_V2),
                "seuil_D9": round(float(SEUIL_D9), 4), "random_state": RANDOM_STATE},
        metrics=flatten_metrics({"test": metriques_test, "cv": {"pr_auc": grille.best_score_}}),
        tracking_uri=uri_mlflow, gold_sha256=sha_gold_v2, gold_dataset_path="data/gold/clients_churn_gold_v2.parquet",
        tags={**etiquettes_communes, "modele_retenu": MODELE_RETENU},
        artifacts=[MODEL_DIR / "model.joblib", MODEL_DIR / "model_manifest.json", MODEL_DIR / "scoring_manifest.json",
                   FIG_DIR / "09_courbes_roc_pr.png", FIG_DIR / "11_calibration.png", FIG_DIR / "13_explicabilite.png"],
        artifact_location=(MODEL_DIR / "mlartifacts").resolve().as_uri())
    run_clv = log_training_run(
        run_name=f"clv_v2_{MODELE_CLV_RETENU}", experiment_name="churn_saas_regression_clv",
        params={"modele": MODELE_CLV_RETENU, "cible": "log1p(valeur_vie_client_eur)", "n_features": len(FEATURES_V2)},
        metrics=flatten_metrics({k: v for k, v in metriques_clv.items() if k != "modele_retenu"}),
        tracking_uri=uri_mlflow, gold_sha256=sha_gold_v2, gold_dataset_path="data/gold/clients_churn_gold_v2.parquet",
        tags={**etiquettes_communes, "modele_retenu": MODELE_CLV_RETENU},
        artifacts=[MODEL_DIR / "model_clv.joblib", MODEL_DIR / "metrics_clv.json", FIG_DIR / "14_clv_reel_vs_predit.png"],
        artifact_location=(MODEL_DIR / "mlartifacts").resolve().as_uri())
    print("Serveur MLflow :", uri_mlflow if uri_mlflow.startswith("http") else "store local data/model_v2/mlflow.db")
    print("Run churn :", run_churn, "| Run CLV :", run_clv)
except Exception as erreur:
    print(f"MLflow indisponible ({type(erreur).__name__}) : runs non enregistrés, le reste du notebook n'est pas affecté.")

### 13.4 Suivi en production et mesure de l'impact réel

| Dispositif | Ce qu'il surveille | Outil |
|---|---|---|
| Dérive des entrées | PSI et test de Kolmogorov-Smirnov par variable, chaque cycle | Exporteur de dérive → Prometheus → alerte `ChurnSaasDriftPSIEleve` |
| Santé de l'API | Requêtes par code de statut, latence (95ᵉ centile), taille des lots | `/metrics` → Prometheus → tableau de bord Grafana |
| Performance réelle | Rappel et précision au seuil D9, dès que les issues du cycle sont connues | Rapport mensuel (à automatiser) |
| Cohérence des explications | Contrôles de la base de connaissance à chaque entraînement ; attribution de la dérive des scores aux variables (§13.1.1) | Notebook (§9.6.1, §15.3) ; garde-fou `/ready` de l'API |
| **Impact métier (D12–D13)** | Taux de résiliation des comptes traités contre un **groupe témoin** de 20 % des comptes priorisés, tirés au hasard et traités avec un mois de décalage (jamais privés d'action) | Consolidation trimestrielle |
| Qualité du code | Tests unitaires, récupération des données, build et santé de l'image | GitHub Actions à chaque push |

Le groupe témoin est la seule façon de distinguer un compte **sauvé par l'action** d'un compte
qui ne serait **de toute façon pas parti**. Le traitement différé (et non refusé) limite le
coût éthique de cette mesure.

### 13.5 Versioning : état actuel

| Objet | Outil | Où le consulter |
|---|---|---|
| Code, notebooks, manifestes, métriques | Git | GitHub (lien en page de garde) |
| Données brutes, Gold, modèles | DVC | DagsHub (lien en page de garde) ; `dvc pull` |
| Pipeline complet (dont ce notebook) | `dvc.yaml` / `dvc.lock` | `dvc repro` ne rejoue que ce qui a changé |
| Expériences | MLflow | DagsHub (lien en page de garde) |
| Environnement | `requirements*.txt`, `Dockerfile` | §15 |

## 14. Conclusion : synthèse et recommandations

📘 **Explication — tableau de bord final**

Cette cellule rassemble les chiffres clés produits dans le notebook, pour une lecture en un coup
d'œil. Tous proviennent des cellules précédentes : aucun chiffre n'est saisi à la main.

In [ ]:
bilan = pd.Series({
    "Comptes (après dédoublonnage)": f"{len(gold_v2)}",
    "Taux de churn": f"{taux_churn:.1%}",
    "Features du modèle (Gold v2)": f"{len(FEATURES_V2)} (contre 30 en v1)",
    "Modèle churn retenu": f"{MODELE_RETENU} (C = {C_RETENU})",
    "PR-AUC validation croisée": f"{grille.best_score_:.3f}",
    "PR-AUC test / baseline métier D8": f"{metriques_test['pr_auc']:.3f} / {metriques_test['pr_auc_baseline_metier_D8']:.3f}",
    "ROC-AUC test": f"{metriques_test['roc_auc']:.3f}",
    "Critère D8 (gain ≥ 0,15)": f"{metriques_test['gain_pr_auc_vs_D8']:+.3f} -> {'respecté' if metriques_test['critere_D8_respecte'] else 'non respecté'}",
    "Seuil D9 (calculé hors pli)": f"{SEUIL_D9:.3f}",
    "Rappel / précision au seuil D9 (test)": f"{metriques_test['rappel_seuil_D9']:.1%} / {metriques_test['precision_seuil_D9']:.1%}",
    "Calibration (Brier test / référence)": f"{brier:.3f} / {brier_reference:.3f}",
    "Modèle CLV retenu": f"{MODELE_CLV_RETENU}",
    "CLV : R² log / erreur relative médiane": f"{metriques_clv['r2_log']:.3f} / {metriques_clv['erreur_relative_mediane']:.0%}",
    "Churners parmi les 150 priorités Hautes": f"{int(haute['churn'].sum())} (hasard : {CAPACITE_CSM_D10 * y_test.mean():.0f})",
    "Recette D15 : Spearman": f"{rho:.3f}",
}, name="valeur")
display(bilan.to_frame())

### Synthèse

Le projet répond au besoin avec **trois briques** : un modèle de churn **simple, explicable,
calibré** et **nettement meilleur** qu'une règle métier ; un modèle de valeur client qui ordonne
correctement les comptes par enjeu ; et un **système de décision** qui transforme ces deux
prédictions en une liste mensuelle de 150 comptes à contacter, dans la limite de la capacité de
l'équipe. Chaque priorité est expliquée, et le modèle est confronté à une base de connaissance
qui aurait bloqué ses erreurs passées (leurres, fuite). La démarche est reproductible de bout en bout : données versionnées par
DVC, code sur GitHub, entraînements tracés dans MLflow, notebook rejoué par `dvc repro`,
intégration continue.

| Compétence | Ce que le notebook démontre |
|---|---|
| C1 | Cadrage du besoin (D1–D15), inventaire et gouvernance des données, alternatives |
| C2 | RGPD (base légale, finalité, minimisation, art. 22 testé), conséquences FP/FN, audit d'équité |
| C3 | Nettoyage démontré étape par étape, portrait et distributions, 13 règles de cohérence (valeurs impossibles neutralisées, valeurs extrêmes justifiées), piège de fuite et leurres identifiés par des tests, Gold v2 versionné |
| C4 | Deux baselines, trois familles de modèles, ablation, règle de sélection fixée à l'avance |
| C5 | Réglage en validation croisée, seuil D9 hors pli, évaluation unique sur le test, calibration, explicabilité confrontée à une base de connaissance |
| C6 | Pipelines sérialisés, règle de décision figée, scoring d'un export brut, API testée de bout en bout, chaque priorité expliquée |
| C7 | Architecture cible batch mensuel + CRM, contraintes mesurées, écarts identifiés |
| C8 | Métriques techniques et métier, impact estimé avec analyse de sensibilité, recette D15, faux négatifs expliqués |
| C9 | Détection de dérive (PSI et sortie du modèle), politique de ré-entraînement, MLflow, protocole de mesure d'impact, attribution de la dérive des scores |

### Recommandations

1. **Mettre le modèle v2.1 en service** : pointer `MODEL_DIR` de l'API et de l'image Docker vers
   `data/model_v2` ; la CI vérifiera le démarrage.
2. **Soumettre deux arbitrages au métier** : remplacer le critère D15(b) par la part de perte
   réelle captée par la priorité Haute ; ajouter un **filet de sécurité** pour les comptes de très
   grande valeur sous le seuil (§12.4).
3. **Confirmer auprès du propriétaire des données** que `derniere_connexion_jours` est calculée à
   la date du scoring, et lui soumettre les questions du §6.5 : que comptent les retards de paiement
   des comptes récents, ce que mesurent les connexions et heures d'usage, d'où vient le délai de
   support des comptes sans ticket, pourquoi la CLV est plafonnée. Réévaluer aussi la capacité D10 à l'échelle du portefeuille complet (le cycle
   simulé ne contient que 1 000 comptes).
4. **Industrialiser le cycle mensuel** : planificateur du batch, connecteur CRM, routage des
   alertes Prometheus (Alertmanager), rapport mensuel de rappel sur les issues connues.
5. **Mesurer l'impact réel** avec le groupe témoin à traitement différé (D12), consolidé chaque
   trimestre (D13).
6. **Enrichir les données** : plusieurs instantanés mensuels (validation temporelle, tendances
   d'usage sur trois mois), qui sont la principale piste d'amélioration du modèle : les faux
   négatifs critiques ne présentent aucun signal dans les données actuelles (§12.4.1).
7. **Faire relire la base de connaissance par l'équipe Customer Success** (sens attendus,
   libellés), puis, à la prochaine version, **retirer les deux ratios sans apport** (§9.6.3) en
   repassant par la procédure de sélection du §8. La même procédure testera le retrait d'une
   variable dans chaque paire redondante (§6.3) et une transformation logarithmique des
   variables de taille (§6.1).

## 15. Annexes

### 15.1 Paramètres, versions et dépendances

📘 **Explication — tout ce qu'il faut pour reproduire**

On affiche les paramètres du notebook, les versions exactes des bibliothèques utilisées (lues
dans l'environnement d'exécution) et les fichiers de dépendances du projet. Avec le commit Git
et `dvc pull`, ces informations suffisent pour reproduire les résultats à l'identique.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

parametres = {
    "RANDOM_STATE": RANDOM_STATE, "RAPPEL_CIBLE_D9": RAPPEL_CIBLE_D9, "CAPACITE_CSM_D10": CAPACITE_CSM_D10,
    "GAIN_MIN_D8": GAIN_MIN_D8, "COUT_FN_EUR (F14)": COUT_FN_EUR, "COUTS_FP_EUR (H01)": COUTS_FP_EUR,
    "PSI_SEUIL_ALERTE": PSI_SEUIL_ALERTE, "C retenu": C_RETENU, "seuil D9": round(float(SEUIL_D9), 4),
    "validation croisée": "StratifiedKFold(5, shuffle=True, random_state=42)",
}
display(pd.Series(parametres, name="valeur").to_frame())

paquets = ["pandas", "numpy", "scikit-learn", "scipy", "matplotlib", "joblib", "pyarrow",
           "fastapi", "mlflow", "dvc", "nbconvert", "codecarbon"]
versions = {}
for paquet in paquets:
    try:
        versions[paquet] = version(paquet)
    except PackageNotFoundError:
        versions[paquet] = "non installé"
display(pd.Series(versions, name="version").to_frame())
print("requirements.txt (image de l'API) :")
print((ROOT / "requirements.txt").read_text(encoding="utf-8"))

### 15.2 Fonctions utilitaires réutilisées (dossier `src/`)

📘 **Explication — inventaire du code réutilisable**

Le notebook ne duplique pas la logique : il appelle des fonctions du dossier `src/`, couvertes par
des tests unitaires (`tests/`). La cellule liste chaque fonction utilisée avec la première ligne
de sa documentation, lue directement dans le code.

In [ ]:
import inspect

import src.drift
import src.explain
import src.features
import src.gold
import src.scoring
import src.silver
import src.tracking

fonctions = []
for module in [src.silver, src.features, src.gold, src.scoring, src.explain, src.drift, src.tracking]:
    for nom, objet in inspect.getmembers(module, inspect.isfunction):
        if objet.__module__ == module.__name__ and not nom.startswith("_"):
            doc = (inspect.getdoc(objet) or "").split("\n")[0]
            fonctions.append({"module": module.__name__, "fonction": nom, "rôle": doc})
display(pd.DataFrame(fonctions))

### 15.3 Vérifications finales

📘 **Explication — garde-fous exécutables**

Dernière cellule de code : une série d'**assertions**. Chacune vérifie une propriété
indispensable du livrable ; si l'une échoue, l'exécution du notebook s'arrête avec une erreur, au
lieu de produire des résultats faux en silence.

In [ ]:
interdites = {"client_id", "sante_compte_fin_periode", "valeur_vie_client_eur", "churn", "commentaire_csm"}
verifications = {
    "aucune variable interdite dans les features": interdites.isdisjoint(FEATURES_V2),
    "aucun leurre dans les features": set(LEURRES).isdisjoint(FEATURES_V2),
    "train et test disjoints": set(train["client_id"]).isdisjoint(test["client_id"]),
    "seuil D9 calculé sans le test (hors pli)": manifeste_scoring["regle_decision"]["seuil_D9_calcule_sur"].startswith("prédictions hors pli"),
    "critère D8 respecté sur le test": metriques_test["critere_D8_respecte"],
    "actions conformes à D3 (art. 22)": all(action_est_conforme_d3(a) for a in ACTIONS_PAR_PRIORITE.values()),
    "modèle conforme à la base de connaissance (aucun contrôle bloquant)": (controles_v2["gravite"] != "bloquant").all(),
    "explication exacte (décomposition du score)": ecart_decomposition < 1e-9,
    "artefacts présents": all((MODEL_DIR / f).exists() for f in ["model.joblib", "model_clv.joblib", "model_manifest.json",
                                                                 "scoring_manifest.json", "metrics.json", "metrics_clv.json",
                                                                 "scoring_exemple_cycle.parquet",
                                                                 "explication_reference.json"]),
    "Gold v2 et manifeste présents": (GOLD_DIR / "clients_churn_gold_v2.parquet").exists() and (GOLD_DIR / "gold_v2_manifest.json").exists(),
}
display(pd.Series(verifications, name="vérifiée").to_frame())
assert all(verifications.values()), [k for k, v in verifications.items() if not v]
print(f"Toutes les vérifications passent. Figures produites : {len(list(FIG_DIR.glob('*.png')))} dans {FIG_DIR.relative_to(ROOT)}")

### 15.4 Reproduire ce notebook

```bash
git clone https://github.com/guillaumesaidani-coder/churn_saas.git && cd churn_saas
pip install -r requirements-dev.txt
dvc pull                                  # données brutes, Gold, modèles depuis DagsHub
dvc repro certification                   # exécute ce notebook ; version exécutée dans reports/notebooks/
```

### 15.5 Glossaire

| Terme | Définition |
|---|---|
| **Churn** | Résiliation d'un abonnement à l'échéance (D2) |
| **CLV** | Valeur vie client : revenu total attendu d'un compte sur sa durée de vie |
| **Fuite de données** | Variable contenant une information indisponible au moment de la prédiction ; elle gonfle artificiellement la performance |
| **Leurre** | Variable sans pouvoir prédictif, ajoutée pour tester la démarche |
| **Validation croisée** | Découpage répété du jeu d'entraînement pour estimer la performance et sa dispersion |
| **Hors pli** (*out-of-fold*) | Prédiction faite par un modèle qui n'a pas vu la ligne à l'entraînement |
| **ROC-AUC** | Probabilité qu'un compte qui part ait un score supérieur à un compte qui reste |
| **PR-AUC** | Aire sous la courbe précision-rappel ; plus informative que la ROC-AUC en classes déséquilibrées |
| **Rappel** | Part des comptes qui partent effectivement détectés |
| **Précision** | Part des comptes signalés qui partent effectivement |
| **Calibration / Brier** | Accord entre probabilités prédites et fréquences observées ; le score de Brier en est l'erreur quadratique |
| **Importance par permutation** | Baisse de performance quand on mélange une variable |
| **Contribution** (explication locale) | Part du score d'un compte due à une variable, mesurée par rapport à un compte de référence (moyenne du train) ; pour un modèle linéaire, égale à la valeur SHAP |
| **Base de connaissance** | Référentiel versionné de ce que l'on sait du domaine avant de modéliser (libellés, plages, sens attendus, exclusions) ; sert à formuler et à contrôler les explications |
| **PSI** | Population Stability Index : mesure de dérive de la distribution d'une variable |
| **Pipeline** | Enchaînement figé préparation + modèle, identique à l'entraînement et en production |
| **DVC** | Outil de versioning des données et des modèles, complémentaire de Git |
| **MLflow** | Outil de suivi des expériences (paramètres, métriques, artefacts) |